# AquaVision - Data Preparation & Augmentation (final, from scratch)
**Run top to bottom.** CPU runtime is enough. If Colab resets, run everything again (zips stay on Drive).

| AquaVision report item | Where it is done |
|---|---|
| Sec. 2 / 6 ph.1: DeepFish + Brackish (OZFish dropped) -> ONE single-class YOLO(-seg) dataset | `prepare_data.py detector` |
| Sec. 6.2: 70/20/10 split **per dataset** (clip-wise, no leakage) | `prepare_data.py` (`group_split`) |
| 6.1 (i) geometric: flip, small rotation, scale/zoom, box-safe crop, mosaic | online in Ultralytics - `train_aug_config.py` |
| 6.1 (ii) HSV photometric: blue-green hue bias, saturation down-scaling + speckle, vignette/illumination, stochastic CLAHE, haze, motion blur | `underwater_aug.py` (UWColorCast, UWTurbidity, UWVignetteIllumination, UWStochasticCLAHE, UWHaze, UWMotionBlur) |
| 6.1 (iii) occlusion/domain: random erasing, synthetic overlays, cross-dataset compositing | UWRandomErasing, UWSyntheticOverlay (`__ovl`), UWDomainTransfer + fish paste (`__comp`) in `augment_offline.py` |
| 6.1: mosaic / MixUp (and overlays) off in the final epochs | `close_mosaic` + `data_late.yaml` (no `__ovl`/`__comp`) |
| Phase 1: augmentation cached to Drive | last cell |
| Sec. 2 / 5.2 / 6.3: 23-class Fish4Knowledge, class-imbalanced -> class-balanced train set, macro-F1 | `f4k_prepare.py` (rare-class aware) |
| 6.1 last sentence: held-out clean-vs-turbid split | `turbid_split.py` (section 7b) |
| 6.1 last sentence: per-category ablation, box IoU + mask-boundary error | `ablation_prep.py` (7c) + `ablation_run.py` (GPU hand-off) |
| 6.1 tiers on Fish4Knowledge crops too (geometric, photometric, overlay, domain transfer; no compositing) | `f4k_prepare.py` |
| 6.1 clean-vs-turbid for the classifier + 5.2 macro-F1 (with CI, head/mid/tail) | `cls_turbid_test.py` (8a) + `eval_cls.py` (8a-eval, GPU hand-off) |

**v7 (training-config fix, from the segmenter baseline overfitting - best epoch 20/100):** `train_aug_config.py` now sets `patience=25` (phase 1), `batch=16`, and fixes the phase-2 warm-up restart (`optimizer="SGD"`, `lr0=0.001`, `warmup_epochs=0`). Nothing else in the notebook changed. **Not in this notebook:** `prepare_data.py --size_strata 2` (fish-size-balanced clip split), `test_metrics.json` and recall-by-fish-size; use the updated segmenter scripts for those. Note `prepare_data.py --bg_ratio` defaults to 1.0 here; the brief suggests 0.3-0.5 and treats it as a separate (data) step.

**v7 (classifier brief, 29 Oct baseline):** Fish4Knowledge now runs with the brief's settings (the old defaults capped every class at 381 and threw away ~15k real crops); `f4k_prepare.py` is reproducible (`--seed` seeds the synthetic crops), prints how many real crops the cap discards and which classes are split per image, previews weak mid classes (`--preview_ids`); new `cls_turbid_test.py` + `eval_cls.py`; `train_classifier` has the online flags from brief P3; checklist has classifier rows. **Split logic untouched -> test set identical to the baseline.**

**Changes in this version (v2):** OZFish removed everywhere; one single detector-build cell placed *after* the Brackish frame extraction; `PSEUDO_MASKS = True` is now really passed to `prepare_data.py` (so GrabCut masks + cross-dataset compositing work); broken/stale duplicate build cells and diagnostics removed; new proof cell shows originals vs augmented copies per dataset and per Fish4Knowledge class; val/test leakage assertion; count tables saved to Drive.

**Changes in v5 (SCALE fix - found from the 4a-verify screenshot):**
- The Brackish PIXEL labels are in a **smaller coordinate space than the full video** (boxes landed ~2x too far up/left, so most missed the fish). Section 4a now measures how far the label boxes reach, compares it with the video size, and saves every frame at the **label resolution** (`LABEL_SCALE = 'auto'`, override by hand if needed). Frames are therefore smaller (faster GrabCut, less disk).
- 4a-verify now shows **zoomed crops of large fish** at k-2 / k-1 / k, so you can clearly see whether the box is on the fish.
- The extraction report also stores the video size, label extent and scale.

**Changes in v4 (Brackish frame-alignment fix + report-number fixes):**
- **Brackish label k -> video frame k-1.** Label files are numbered 1..N per video but OpenCV frames are 0..N-1. Older versions read frame k (one frame late), so boxes sat on empty water and GrabCut masks were wrong. Section 4a now reads frames *sequentially* (exact, no seek drift), uses `LABEL_BASE = 1`, deletes frames from older runs, and writes `brackish_extract_report.json`.
- New cell 4a-verify: shows the same labelled frame at k-2 / **k-1 (new)** / k so you can SEE which one fits the fish.
- `prepare_data.py` now saves `raw_counts.json` (true raw images / empty frames BEFORE the 1:1 empty-frame cap), so the summary no longer reports post-cap numbers as "raw".
- Section 8c checklist has a new row: every Brackish label has its frame.
- The "restore from work.zip" cell is now OFF by default (it used to overwrite freshly built data with the old zip).
- Removed the unused `scikit-learn` install and the old diagnostic cells.

**HOW TO RUN v4:** because the Brackish frames change, run **everything again from section 1 down** (4b GrabCut takes ~20-40 min). All old numbers/outputs are cleared on purpose.

**Changes in v3 (aligned with the AquaVision PDF):**
- Section 2 unzip now *checks every zip* (return code, size, content) and **stops if DeepFish is missing** (v2 silently ended with only Brackish + Fish4Knowledge, so domain transfer, compositing, turbid split and ablation all ran on Brackish only).
- Section 4b asserts both detector datasets are really in the built set.
- Fish4Knowledge: new `--extra_aug` option so the large classes also receive augmented copies (all 23 classes follow the PDF's tiers; rare classes still get the most).
- New section 8c: automatic **PDF-alignment checklist** + storage map (what is stored where).

**v6 (scale fix, confirmed on your screenshots):** Brackish pixel labels are in **960x540** space while the videos are **1920x1080**, so every label must be multiplied by **2** to land on the fish (green boxes in the scale test). `LABEL_SCALE` is now fixed to **2** (the old `'auto'` guess picked a shrink factor that was too large because fish rarely reach the image edge). Frames are saved at 960x540 so `prepare_data.py` normalises correctly. DeepFish zip is found automatically. **Everything built before this fix has mis-placed Brackish boxes - rebuild from 4a.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Mount Drive + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas matplotlib   # (scikit-learn removed: never imported)
%mkdir -p /content/aquavision
%cd /content/aquavision

## 2. Unzip + identify datasets
Edit `ZIPS` if your zip names differ. Datasets used: **Brackish, DeepFish, Fish4Knowledge** (OZFish is not used).

In [ ]:
import os, re, shutil, subprocess, collections
import glob
DRIVE = '/content/drive/MyDrive/aquavision'
# raw dataset zips on Drive; outputs written by this notebook (*cached*, work*) are ignored
ZIPS = sorted(z for z in glob.glob(f'{DRIVE}/*.zip')
              if not any(k in os.path.basename(z).lower() for k in ('cached', 'work')))
# ZIPS = [...]   # <- or list the three zip paths by hand
print('raw zips found:', [(os.path.basename(z), round(os.path.getsize(z) / 1e6)) for z in ZIPS])
assert len(ZIPS) >= 3, f'expected 3 dataset zips (Brackish, DeepFish, Fish4Knowledge) in {DRIVE}, found {len(ZIPS)}'
DATA = '/content/data'
need = {'brackish', 'DeepFish', 'fish4knowledge'}

def profile(root):
    dirs, ext = set(), collections.Counter()
    for dp, dn, fn in os.walk(root):
        dirs |= set(dn); ext.update(os.path.splitext(f)[1].lower() for f in fn)
    return dirs, ext

def guess(root):
    dirs, ext = profile(root)
    if ext['.avi'] or ext['.mp4']: return 'brackish'
    if any(re.fullmatch(r'fish_?\d+', d.lower()) or re.match(r'^\d{2}_[A-Za-z]', d) for d in dirs): return 'fish4knowledge'
    if ext['.jpg'] + ext['.jpeg'] + ext['.png'] > 0: return 'DeepFish'     # images, no videos, no species folders
    return 'unknown'

if not (os.path.exists(DATA) and need <= set(os.listdir(DATA))):
    shutil.rmtree(DATA, ignore_errors=True); os.makedirs(DATA)
    for i, z in enumerate(ZIPS, 1):
        if not os.path.exists(z):
            print(f'!! MISSING on Drive: {z}  (check the name / wait for Drive sync)'); continue
        print(f'unzipping {os.path.basename(z)} ({os.path.getsize(z)/1e6:.0f} MB) ... takes minutes')
        r = subprocess.run(['unzip', '-q', '-o', z, '-d', f'{DATA}/z{i}'], capture_output=True, text=True)
        print('   return code', r.returncode, (r.stderr or '')[:300])   # 0 / 1 = OK, >=2 = broken zip
    for d in sorted(os.listdir(DATA)):
        g = guess(f'{DATA}/{d}'); print(d, '->', g)
        if g != 'unknown' and not os.path.exists(f'{DATA}/{g}'):
            shutil.move(f'{DATA}/{d}', f'{DATA}/{g}')

print('DATASETS:', sorted(os.listdir(DATA)))
missing = need - set(os.listdir(DATA))
assert not missing, f'STOP - dataset(s) not found: {missing}. Fix the ZIPS paths above and re-run this cell.'

# what does the DeepFish copy look like? (loader uses masks/ if present, else YOLO .txt boxes)
d, e = profile(f'{DATA}/DeepFish')
print('DeepFish extensions:', dict(e)); print('DeepFish folders (sample):', sorted(d)[:15])

## 3. Write the code modules
Eight files are created in `/content/aquavision`: `underwater_aug.py`, `prepare_data.py`, `f4k_prepare.py`, `augment_offline.py`, `train_aug_config.py`, plus the report-6.1 verification tools `turbid_split.py`, `ablation_prep.py`, `ablation_run.py`.

In [ ]:
%%writefile underwater_aug.py
"""
underwater_aug.py  --  AquaVision underwater augmentation toolkit
==================================================================
Implements the three-tier augmentation scheme from the report (Sec. 6.1):

  (i)   geometric  -> handled by Ultralytics itself (flip, rotation, scale, mosaic, mixup)
                      + a tiny label-free flip/rotate helper for classifier crops here
  (ii)  underwater photometric (all label-preserving, written for OpenCV **BGR** images):
          UWColorCast, UWTurbidity, UWVignetteIllumination, UWStochasticCLAHE,
          UWHaze, UWMotionBlur
  (iii) occlusion / domain: UWRandomErasing, UWSyntheticOverlay, UWDomainTransfer

IMPORTANT: Ultralytics and cv2.imread both give BGR arrays, so every transform
here assumes BGR uint8 (H, W, 3).

Every transform is a real `albumentations.ImageOnlyTransform` when Albumentations
is installed (so it can be passed to `model.train(augmentations=[...])`), and a
tiny drop-in shim otherwise (so the offline scripts run without it).
"""
from __future__ import annotations

import math
from pathlib import Path
from typing import Sequence

import cv2
import numpy as np

try:  # ------------------------------------------------------------------ backend
    import albumentations as A
    _Base = A.ImageOnlyTransform
    HAS_ALBU = True
except ImportError:  # minimal shim with the same calling convention
    A = None
    HAS_ALBU = False

    class _Base:  # type: ignore
        def __init__(self, p: float = 0.5, **_):
            self.p = p

        def __call__(self, force_apply: bool = False, **data):
            if force_apply or np.random.random() < self.p:
                data["image"] = self.apply(data["image"])
            return data


_SEEDED = None


def set_seed(seed=None):
    """Make augmentation reproducible (offline scripts). None -> back to fresh OS entropy."""
    global _SEEDED
    _SEEDED = None if seed is None else np.random.default_rng(seed)
    if seed is not None:
        np.random.seed(seed)


def _rng() -> np.random.Generator:
    # default: fresh OS-seeded generator => safe with DataLoader workers
    return _SEEDED if _SEEDED is not None else np.random.default_rng()


def _u8(x: np.ndarray) -> np.ndarray:
    return np.clip(x, 0, 255).astype(np.uint8)


def _lowfreq_noise(h: int, w: int, rng, scales=(4, 8, 16)) -> np.ndarray:
    """Smooth 0..1 noise field (sum of upsampled random grids) -> fog / blobs."""
    acc = np.zeros((h, w), np.float32)
    for i, s in enumerate(scales):
        g = rng.random((s, max(2, int(s * w / max(h, 1))))).astype(np.float32)
        acc += cv2.resize(g, (w, h), interpolation=cv2.INTER_CUBIC) / (i + 1)
    acc -= acc.min()
    return acc / (acc.max() + 1e-6)


# =============================================================== (ii) PHOTOMETRIC
class UWColorCast(_Base):
    """Blue-green cast: attenuate red (fast in water) + pull hue toward cyan.
    Directional, NOT symmetric hue jitter."""

    def __init__(self, red_atten=(0.45, 0.9), hue_pull=(0.05, 0.30), p=0.5):
        super().__init__(p=p)
        self.red_atten, self.hue_pull = red_atten, hue_pull

    def apply(self, img, **params):
        rng = _rng()
        out = img.astype(np.float32)
        out[..., 2] *= rng.uniform(*self.red_atten)          # R (BGR idx 2)
        out[..., 1] *= rng.uniform(1.0, 1.08)                # G
        out[..., 0] *= rng.uniform(1.0, 1.08)                # B
        hsv = cv2.cvtColor(_u8(out), cv2.COLOR_BGR2HSV).astype(np.float32)
        t = rng.uniform(*self.hue_pull)
        h = hsv[..., 0]                                      # OpenCV hue 0..179, cyan ~ 90
        diff = ((90.0 - h + 90.0) % 180.0) - 90.0            # shortest signed distance to cyan
        hsv[..., 0] = (h + t * diff) % 180.0
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    def get_transform_init_args_names(self):
        return ("red_atten", "hue_pull")


class UWTurbidity(_Base):
    """Saturation down-scaling (turbidity) + mild speckle noise (particulate scatter)."""

    def __init__(self, sat_scale=(0.4, 0.9), speckle_sigma=(0.02, 0.08), p=0.4):
        super().__init__(p=p)
        self.sat_scale, self.speckle_sigma = sat_scale, speckle_sigma

    def apply(self, img, **params):
        rng = _rng()
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= rng.uniform(*self.sat_scale)
        out = cv2.cvtColor(_u8(hsv), cv2.COLOR_HSV2BGR).astype(np.float32)
        sigma = rng.uniform(*self.speckle_sigma)
        out *= 1.0 + rng.normal(0.0, sigma, out.shape).astype(np.float32)   # multiplicative = speckle
        return _u8(out)

    def get_transform_init_args_names(self):
        return ("sat_scale", "speckle_sigma")


class UWVignetteIllumination(_Base):
    """Radial vignette + optional localized spotlight hot-spot (ROV/AUV lamps)."""

    def __init__(self, vignette=(0.2, 0.7), hotspot=(0.0, 0.5), gain=(0.7, 1.1), p=0.25):
        super().__init__(p=p)
        self.vignette, self.hotspot, self.gain = vignette, hotspot, gain

    def apply(self, img, **params):
        rng = _rng()
        h, w = img.shape[:2]
        gh, gw = 64, 64                                      # compute on a small grid, upsample
        yy, xx = np.mgrid[0:gh, 0:gw].astype(np.float32)
        yy, xx = yy / (gh - 1), xx / (gw - 1)
        cx, cy = 0.5 + rng.uniform(-0.15, 0.15), 0.5 + rng.uniform(-0.15, 0.15)
        r2 = ((xx - cx) ** 2 + (yy - cy) ** 2) / 0.5
        field = 1.0 - rng.uniform(*self.vignette) * np.clip(r2, 0, 1)
        if rng.random() < 0.6:                               # spotlight
            sx, sy = rng.uniform(0.2, 0.8), rng.uniform(0.2, 0.8)
            sig = rng.uniform(0.05, 0.2)
            field += rng.uniform(*self.hotspot) * np.exp(-((xx - sx) ** 2 + (yy - sy) ** 2) / (2 * sig ** 2))
        field *= rng.uniform(*self.gain)
        field = cv2.resize(field.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)
        return _u8(img.astype(np.float32) * field[..., None])

    def get_transform_init_args_names(self):
        return ("vignette", "hotspot", "gain")


class UWStochasticCLAHE(_Base):
    """CLAHE on the L channel, applied stochastically so the model sees both
    enhanced and raw-turbid frames."""

    def __init__(self, clip=(1.5, 4.0), tiles=(8, 8), p=0.4):
        super().__init__(p=p)
        self.clip, self.tiles = clip, tiles

    def apply(self, img, **params):
        rng = _rng()
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=float(rng.uniform(*self.clip)), tileGridSize=tuple(self.tiles))
        lab[..., 0] = clahe.apply(lab[..., 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    def get_transform_init_args_names(self):
        return ("clip", "tiles")


class UWHaze(_Base):
    """Synthetic haze: I' = I*t + A*(1-t), with a spatially-varying low-frequency
    transmission map t and a blue-green veiling light A."""

    def __init__(self, density=(0.15, 0.6), p=0.25):
        super().__init__(p=p)
        self.density = density

    def apply(self, img, **params):
        rng = _rng()
        h, w = img.shape[:2]
        n = _lowfreq_noise(h, w, rng)
        d = rng.uniform(*self.density)
        t = (1.0 - d * (0.4 + 0.6 * n))[..., None]
        A_ = np.array([rng.uniform(170, 220), rng.uniform(150, 200), rng.uniform(90, 150)], np.float32)  # B,G,R
        return _u8(img.astype(np.float32) * t + A_ * (1.0 - t))

    def get_transform_init_args_names(self):
        return ("density",)


class UWMotionBlur(_Base):
    """Directional blur (random angle, small kernel): fish swimming + vehicle drift."""

    def __init__(self, kernel=(3, 9), p=0.2):
        super().__init__(p=p)
        self.kernel = kernel

    def apply(self, img, **params):
        rng = _rng()
        k = int(rng.integers(self.kernel[0], self.kernel[1] + 1)) | 1       # force odd
        ker = np.zeros((k, k), np.float32)
        ang = rng.uniform(0, math.pi)
        c = k // 2
        dx, dy = math.cos(ang) * c, math.sin(ang) * c
        cv2.line(ker, (int(round(c - dx)), int(round(c - dy))), (int(round(c + dx)), int(round(c + dy))), 1.0, 1)
        ker /= ker.sum()
        return cv2.filter2D(img, -1, ker, borderType=cv2.BORDER_REFLECT)

    def get_transform_init_args_names(self):
        return ("kernel",)


# ============================================================ (iii) OCCLUSION / DOMAIN
class UWRandomErasing(_Base):
    """Cutout patches (kept small so a fish is never fully erased while still labelled)."""

    def __init__(self, n_holes=(1, 4), size=(0.05, 0.2), p=0.4):
        super().__init__(p=p)
        self.n_holes, self.size = n_holes, size

    def apply(self, img, **params):
        rng = _rng()
        out = img.copy()
        h, w = img.shape[:2]
        for _ in range(int(rng.integers(self.n_holes[0], self.n_holes[1] + 1))):
            ph, pw = int(h * rng.uniform(*self.size)), int(w * rng.uniform(*self.size))
            y, x = int(rng.integers(0, max(1, h - ph))), int(rng.integers(0, max(1, w - pw)))
            mode = rng.integers(0, 3)
            if mode == 0:
                out[y:y + ph, x:x + pw] = img.mean(axis=(0, 1))                       # mean fill
            elif mode == 1:
                out[y:y + ph, x:x + pw] = rng.integers(0, 256, (ph, pw, 3), dtype=np.uint8)  # noise
            else:
                out[y:y + ph, x:x + pw] = int(rng.integers(0, 256))                   # gray
        return out

    def get_transform_init_args_names(self):
        return ("n_holes", "size")


class UWSyntheticOverlay(_Base):
    """Rock / coral / plant-like blobs (dark earthy or green, soft edges, biased to the
    lower frame = substrate). Crude stand-in for real cut-out silhouettes.
    Flagged `rare` -> switched off in the final epochs (see set_late_phase)."""

    rare = True
    _PALETTE = [(40, 55, 70), (60, 80, 60), (35, 60, 35), (90, 90, 90), (30, 40, 55), (50, 100, 70)]  # BGR

    def __init__(self, n_blobs=(1, 3), size=(0.05, 0.2), p=0.1):
        super().__init__(p=p)
        self.n_blobs, self.size = n_blobs, size

    def apply(self, img, **params):
        rng = _rng()
        h, w = img.shape[:2]
        layer = np.zeros_like(img)
        mask = np.zeros((h, w), np.uint8)
        for _ in range(int(rng.integers(self.n_blobs[0], self.n_blobs[1] + 1))):
            r = rng.uniform(*self.size) * min(h, w)
            cx = rng.uniform(0, w)
            cy = h * rng.uniform(0.6, 1.05) if rng.random() < 0.6 else rng.uniform(0, h)
            m = int(rng.integers(7, 12))
            ang = np.sort(rng.uniform(0, 2 * math.pi, m))
            rad = r * rng.uniform(0.5, 1.2, m)
            pts = np.stack([cx + rad * np.cos(ang), cy + rad * np.sin(ang)], 1).astype(np.int32)
            col = np.array(self._PALETTE[int(rng.integers(len(self._PALETTE)))], np.float32)
            col = col * rng.uniform(0.7, 1.3)
            cv2.fillPoly(layer, [pts], tuple(float(c) for c in np.clip(col, 0, 255)))
            cv2.fillPoly(mask, [pts], 255)
        k = int(max(3, (min(h, w) * 0.01))) | 1
        alpha = cv2.GaussianBlur(mask, (k, k), 0).astype(np.float32)[..., None] / 255.0 * rng.uniform(0.75, 0.95)
        return _u8(img * (1 - alpha) + layer * alpha)

    def get_transform_init_args_names(self):
        return ("n_blobs", "size")


def compute_lab_stats(paths: Sequence[Path], n: int = 200, seed: int = 0):
    """(mean, std) per LAB channel for a random subset of reference images."""
    rs = np.random.RandomState(seed)
    paths = list(paths)
    rs.shuffle(paths)
    stats = []
    for p in paths[:n]:
        im = cv2.imread(str(p))
        if im is None:
            continue
        im = cv2.resize(im, (128, 128))
        lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB).astype(np.float32).reshape(-1, 3)
        stats.append((lab.mean(0), lab.std(0) + 1e-6))
    return stats


class UWDomainTransfer(_Base):
    """Cross-dataset domain mixing: Reinhard colour transfer in LAB toward a random
    image from *another* dataset (partial blend). Label-preserving, so it reduces
    overfitting to one site's water colour/clutter profile."""

    def __init__(self, ref_stats=None, alpha=(0.5, 1.0), p=0.3):
        super().__init__(p=p)
        self.ref_stats, self.alpha = ref_stats or [], alpha

    def apply(self, img, **params):
        if not self.ref_stats:
            return img
        rng = _rng()
        m_r, s_r = self.ref_stats[int(rng.integers(len(self.ref_stats)))]
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
        flat = lab.reshape(-1, 3)
        m_s, s_s = flat.mean(0), flat.std(0) + 1e-6
        mixed = (lab - m_s) / s_s * s_r + m_r
        a = rng.uniform(*self.alpha)
        out = _u8(a * mixed + (1 - a) * lab)
        return cv2.cvtColor(out, cv2.COLOR_LAB2BGR)

    def get_transform_init_args_names(self):
        return ("alpha",)


# ====================================================== crop helper (classifier only)
class FlipRotate:
    """Mild geometric aug for fish crops (no boxes to update)."""

    def __init__(self, max_deg=15, p_flip=0.5, p_rot=0.5):
        self.max_deg, self.p_flip, self.p_rot = max_deg, p_flip, p_rot

    def __call__(self, image=None, **kw):
        rng = _rng()
        img = image
        if rng.random() < self.p_flip:
            img = cv2.flip(img, 1)
        if rng.random() < self.p_rot:
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-self.max_deg, self.max_deg), 1.0)
            img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
        return {"image": img}


# =============================================================== PIPELINE FACTORY
def build_underwater_transforms(ref_stats=None, strength: float = 1.0, overlay: bool = True):
    """Probabilities follow Sec. 5 of the augmentation strategy:
         always-on  (~0.4-0.5): colour cast, turbidity
         frequent   (0.3-0.5) : CLAHE, random erasing
         occasional (0.15-0.3): haze, motion blur, vignette/illumination
         rare       (0.05-0.1): synthetic overlays   (disabled in final epochs)
       `strength` scales every probability (use >1 offline, <1 to dial down)."""
    s = lambda p: float(min(1.0, p * strength))
    tf = [
        UWColorCast(p=s(0.5)),
        UWTurbidity(p=s(0.4)),
        UWStochasticCLAHE(p=s(0.4)),
        UWRandomErasing(p=s(0.4)),
        UWHaze(p=s(0.25)),
        UWMotionBlur(p=s(0.2)),
        UWVignetteIllumination(p=s(0.25)),
    ]
    if overlay:
        tf.append(UWSyntheticOverlay(p=s(0.1)))
    if ref_stats:
        tf.append(UWDomainTransfer(ref_stats=ref_stats, p=s(0.3)))
    return tf


def set_late_phase(transforms, late: bool = True):
    """Switch off the 'rare' transforms (synthetic occlusion overlays) for the final
    fine-tuning epochs so mask/box geometry statistics stay realistic."""
    for t in transforms:
        if getattr(t, "rare", False):
            if not hasattr(t, "_p0"):
                t._p0 = t.p
            t.p = 0.0 if late else t._p0


class Sequential:
    """Dependency-free Compose for the offline scripts."""

    def __init__(self, transforms):
        self.transforms = list(transforms)

    def __call__(self, image):
        for t in self.transforms:
            image = t(image=image)["image"]
        return image


# ----------------------------------------------------------------- visual preview
if __name__ == "__main__":
    import argparse

    ap = argparse.ArgumentParser(description="Preview each underwater transform on one image")
    ap.add_argument("image")
    ap.add_argument("--out", default="aug_preview.jpg")
    a = ap.parse_args()
    im = cv2.imread(a.image)
    assert im is not None, "cannot read image"
    im = cv2.resize(im, (320, 320))
    tiles = [("original", im)]
    for t in build_underwater_transforms():
        tiles.append((type(t).__name__, t(force_apply=True, image=im.copy())["image"]))
    cols = 3
    rows = math.ceil(len(tiles) / cols)
    canvas = np.zeros((rows * 320, cols * 320, 3), np.uint8)
    for i, (name, t) in enumerate(tiles):
        r, c = divmod(i, cols)
        t = t.copy()
        cv2.putText(t, name, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)
        canvas[r * 320:(r + 1) * 320, c * 320:(c + 1) * 320] = t
    cv2.imwrite(a.out, canvas)
    print("saved", a.out)

In [ ]:
%%writefile prepare_data.py
#!/usr/bin/env python
"""
prepare_data.py -- AquaVision detector data prep (report Sec. 2 / 6, phase 1)
=============================================================================
Brackish + DeepFish (+ OZFish if present) -> ONE single-class ("fish") YOLO-seg dataset,
70/20/10 split PER DATASET, split by video/sequence group (no frame leakage).

  python prepare_data.py detector --brackish B --deepfish D [--ozfish O] --out /content/work/det
  python prepare_data.py viz --det /content/work/det --n 12
  python prepare_data.py classifier --f4k F --out /content/work/cls      (forwards to f4k_prepare.py)

Handles what the real data looks like:
  * Brackish label ids are 1-based (1 fish, 2 small_fish, 3 crab, 4 shrimp, 5 jellyfish,
    6 starfish) and boxes are in PIXELS -> read class names from the COCO json, keep fish +
    small_fish, normalise by the real image size.
  * DeepFish copy with YOLO box txt files (no pixel masks) -> rectangle polygons, or GrabCut
    pseudo-masks with --pseudo_masks. If a masks/ folder exists, real masks are used.
  * Output folder is wiped first, so no stale (wrongly labelled) augmented copies survive.
"""
from __future__ import annotations

import argparse
import csv
import json
import random
import re
import subprocess
import sys
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
import shutil

import cv2
import numpy as np

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
FISH_NAMES = {"fish", "small_fish", "smallfish"}
SPLITS = (("train", 0.70), ("val", 0.20), ("test", 0.10))
SKIP_TXT = {"classes.txt", "labels.txt", "readme.txt", "obj.names", "train.txt", "valid.txt",
            "val.txt", "test.txt"}


@dataclass
class Instance:
    box: tuple | None = None          # (cx, cy, w, h)  normalised OR pixels
    poly: np.ndarray | None = None    # (N, 2)           normalised OR pixels


@dataclass
class Sample:
    dataset: str
    img: Path
    instances: list = field(default_factory=list)
    group: str = ""


# ------------------------------------------------------------------ helpers
def list_images(root: Path):
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT)


def norm_name(s: str) -> str:
    return re.sub(r"[\s\-]+", "_", s.strip().lower())


def group_key(stem: str) -> str:
    """'vid12_frame_000345' -> 'vid12' (strip the trailing frame counter)."""
    g = re.sub(r"[_\-\s]?(frame|img|image|f)?[_\-\s]?\d+$", "", stem, flags=re.I)
    return g or stem


def group_split(items, key, ratios=SPLITS, seed=42, quiet=False):
    """Group-aware split that fills the split with the biggest remaining deficit
    (old version could leave test=0 when a few large groups overshoot the 90% mark)."""
    groups = defaultdict(list)
    for it in items:
        groups[key(it)].append(it)
    keys = sorted(groups)
    if len(keys) < 10:
        if not quiet:
            print(f"  [warn] only {len(keys)} groups found; falling back to per-image split")
        groups = {i: [it] for i, it in enumerate(items)}
        keys = list(groups)
    rng = random.Random(seed)
    rng.shuffle(keys)
    keys.sort(key=lambda k: -len(groups[k]))          # big groups first, random tie-break
    total = sum(len(v) for v in groups.values())
    tgt = {n: r * total for n, r in ratios}
    cur = {n: 0 for n, _ in ratios}
    out = {n: [] for n, _ in ratios}
    for k in keys:
        n = max(tgt, key=lambda x: tgt[x] - cur[x])
        out[n].extend(groups[k])
        cur[n] += len(groups[k])
    return out


def poly_from_box(x1, y1, x2, y2):
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], np.float32)


def grabcut_poly(img, box, iters=3):
    """Pseudo-mask for a normalised (cx,cy,w,h) box via GrabCut -> normalised polygon or None."""
    H, W = img.shape[:2]
    cx, cy, w, h = box
    x1, y1 = int((cx - w / 2) * W), int((cy - h / 2) * H)
    x2, y2 = int((cx + w / 2) * W), int((cy + h / 2) * H)
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
    if x2 - x1 < 12 or y2 - y1 < 12:
        return None
    mx, my = int((x2 - x1) * 0.1), int((y2 - y1) * 0.1)
    X1, Y1, X2, Y2 = max(0, x1 - mx), max(0, y1 - my), min(W, x2 + mx), min(H, y2 + my)
    crop = img[Y1:Y2, X1:X2]
    mask = np.zeros(crop.shape[:2], np.uint8)
    rect = (x1 - X1, y1 - Y1, x2 - x1, y2 - y1)
    try:
        cv2.grabCut(crop, mask, rect, np.zeros((1, 65)), np.zeros((1, 65)), iters, cv2.GC_INIT_WITH_RECT)
    except cv2.error:
        return None
    fg = ((mask == 1) | (mask == 3)).astype(np.uint8)
    inbox = np.zeros_like(fg)
    inbox[rect[1]:rect[1] + rect[3], rect[0]:rect[0] + rect[2]] = 1
    fg *= inbox
    frac = fg.sum() / max(1, rect[2] * rect[3])
    if frac < 0.2 or frac > 0.95:
        return None
    cs, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cs:
        return None
    c = max(cs, key=cv2.contourArea)
    c = cv2.approxPolyDP(c, 0.003 * cv2.arcLength(c, True), True).reshape(-1, 2)
    if len(c) < 3:
        return None
    c = c.astype(np.float32) + np.array([X1, Y1], np.float32)
    return c / np.array([W, H], np.float32)


def parse_yolo_txt(path: Path, keep=None):
    """Lines 'cls cx cy w h' (box) or 'cls x1 y1 x2 y2 ...' (polygon). keep=None -> all classes."""
    inst = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            v = list(map(float, parts[1:]))
        except ValueError:
            continue
        if keep is not None and c not in keep:
            continue
        if len(v) == 4:
            inst.append(Instance(box=tuple(v)))
        elif len(v) >= 6 and len(v) % 2 == 0:
            inst.append(Instance(poly=np.array(v, np.float32).reshape(-1, 2)))
    return inst


# ------------------------------------------------------------------ Brackish
def brackish_fish_ids(root: Path):
    """Class ids that mean 'fish' (fish + small_fish) from the COCO json (1-based), else fallbacks."""
    for jf in sorted(Path(root).rglob("*.json")):
        try:
            d = json.loads(jf.read_text())
        except Exception:
            continue
        if isinstance(d, dict) and d.get("categories"):
            cats = {int(c["id"]): norm_name(c["name"]) for c in d["categories"]}
            ids = {i for i, n in cats.items() if n in FISH_NAMES}
            if ids:
                print(f"[brackish] class ids from {jf.name}: {cats} -> fish ids {sorted(ids)}")
                return ids
    for f in Path(root).rglob("classes.txt"):
        names = [l.strip() for l in f.read_text().splitlines() if l.strip()]
        ids = {i for i, n in enumerate(names) if norm_name(n) in FISH_NAMES}
        if ids:
            print(f"[brackish] class ids from classes.txt: {names} -> fish ids {sorted(ids)}")
            return ids
    print("[brackish] no class-name file found -> assuming Brackish default 1-based ids {1,2}")
    return {1, 2}


def load_brackish(root: Path):
    root = Path(root)
    fish_ids = brackish_fish_ids(root)
    imgs = [p for p in list_images(root) if "brackishmot" not in str(p).lower()]
    all_txt = [t for t in root.rglob("*.txt") if t.name.lower() not in SKIP_TXT
               and "brackishmot" not in str(t).lower()]
    yolo_txt = [t for t in all_txt if "annotations_yolo" in str(t).lower()]
    txts = {t.stem: t for t in (yolo_txt or all_txt)}
    samples, nolabel = [], 0
    for p in imgs:
        t = txts.get(p.stem)
        if t is None:
            nolabel += 1
            continue
        samples.append(Sample("brackish", p, parse_yolo_txt(t, fish_ids), group_key(p.stem)))
    bg = sum(1 for s in samples if not s.instances)
    print(f"[brackish] {len(samples)} images, {sum(len(s.instances) for s in samples)} fish instances, "
          f"{bg} background frames, {nolabel} images without label file (skipped)")
    return samples


# ------------------------------------------------------------------ OZFish (optional)
def load_ozfish(root: Path):
    root = Path(root)
    imgs = list_images(root)
    txts = {t.stem: t for t in root.rglob("*.txt") if t.name.lower() not in SKIP_TXT}
    samples = []
    for p in imgs:
        t = txts.get(p.stem)
        if t is not None:
            samples.append(Sample("ozfish", p, parse_yolo_txt(t), group_key(p.stem)))
    print(f"[ozfish] {len(samples)} images, {sum(len(s.instances) for s in samples)} boxes")
    return samples


# ------------------------------------------------------------------ DeepFish
def mask_to_polys(mask: np.ndarray, min_area_frac=1e-4, eps=0.002):
    H, W = mask.shape[:2]
    m = (mask > 0).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cs:
        if cv2.contourArea(c) < min_area_frac * H * W:
            continue
        c = cv2.approxPolyDP(c, eps * cv2.arcLength(c, True), True).reshape(-1, 2)
        if len(c) >= 3:
            out.append(c.astype(np.float32) / np.array([W, H], np.float32))
    return out


def load_deepfish(root: Path):
    root = Path(root)
    all_imgs = list_images(root)
    is_mask = lambda p: any(x.lower() in ("masks", "mask") for x in p.parts)
    masks = {re.sub(r"_?mask$", "", p.stem, flags=re.I): p for p in all_imgs if is_mask(p)}
    samples = []
    if masks:                                                  # real pixel masks
        for p in all_imgs:
            if is_mask(p) or p.stem not in masks:
                continue
            m = cv2.imread(str(masks[p.stem]), cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            samples.append(Sample("deepfish", p, [Instance(poly=q) for q in mask_to_polys(m)], group_key(p.stem)))
        print(f"[deepfish] mode=masks: {len(samples)} images, {sum(len(s.instances) for s in samples)} polygons")
        return samples
    txts = {t.stem: t for t in root.rglob("*.txt") if t.name.lower() not in SKIP_TXT}
    for p in all_imgs:
        t = txts.get(p.stem)
        if t is None:
            continue
        samples.append(Sample("deepfish", p, parse_yolo_txt(t), group_key(p.stem)))
    bg = sum(1 for s in samples if not s.instances)
    print(f"[deepfish] mode=yolo-box: {len(samples)} images, {sum(len(s.instances) for s in samples)} instances "
          f"({bg} background frames)")
    print("  [note] no pixel masks in this copy -> rectangle polygons (use --pseudo_masks for GrabCut masks)")
    return samples


# ------------------------------------------------------------------ writer
def to_norm_poly(ins, W, H, img, pseudo):
    """-> (normalised polygon (N,2), kind) or None."""
    if ins.poly is not None:
        p = np.array(ins.poly, np.float32)
        if p.max() > 1.5:
            p = p / np.array([W, H], np.float32)
        return np.clip(p, 0, 1), "mask"
    cx, cy, w, h = ins.box
    if max(ins.box) > 1.5:                                     # pixel units
        cx, w, cy, h = cx / W, w / W, cy / H, h / H
    x1, y1, x2, y2 = (max(0.0, cx - w / 2), max(0.0, cy - h / 2), min(1.0, cx + w / 2), min(1.0, cy + h / 2))
    if x2 - x1 < 1e-3 or y2 - y1 < 1e-3:
        return None
    if pseudo:
        poly = grabcut_poly(img, ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
        if poly is not None:
            return np.clip(poly, 0, 1), "pseudo"
    return poly_from_box(x1, y1, x2, y2), "box"


def fmt_poly(p):
    return "0 " + " ".join(f"{v:.6f}" for v in np.clip(p, 0, 1).reshape(-1))


def write_detector(samples, out: Path, max_side=1280, pseudo=False, seed=42, bg_ratio=1.0):
    out = Path(out)
    if out.exists():
        shutil.rmtree(out)                                     # never keep stale copies
    for s in ("train", "val", "test"):
        (out / "images" / s).mkdir(parents=True, exist_ok=True)
        (out / "labels" / s).mkdir(parents=True, exist_ok=True)
    by_ds = defaultdict(list)
    for s in samples:
        by_ds[s.dataset].append(s)
    rows, stats = [], defaultdict(int)
    raw_stats = {}                                             # true counts BEFORE the empty-frame cap
    for ds, items in by_ds.items():
        pos = [s for s in items if s.instances]
        neg = [s for s in items if not s.instances]
        raw_stats[ds] = {"raw_images": len(items), "raw_labelled": len(pos), "raw_empty": len(neg),
                         "raw_instances": sum(len(s.instances) for s in items)}
        if bg_ratio is not None and len(neg) > bg_ratio * len(pos):
            neg = random.Random(seed).sample(neg, int(bg_ratio * len(pos)))
            print(f"[{ds}] background frames capped to {len(neg)} (bg_ratio={bg_ratio})")
        items = pos + neg
        sp = group_split(items, key=lambda s: s.group, seed=seed)
        print(f"[split:{ds}] " + ", ".join(f"{k}={len(v)}" for k, v in sp.items()))
        for split, lst in sp.items():
            for s in lst:
                img = cv2.imread(str(s.img))
                if img is None:
                    stats["unreadable"] += 1
                    continue
                H, W = img.shape[:2]
                lines, kinds = [], set()
                for ins in s.instances:
                    r = to_norm_poly(ins, W, H, img, pseudo)
                    if r is None or len(r[0]) < 3:
                        continue
                    lines.append(fmt_poly(r[0]))
                    kinds.add(r[1])
                if max_side and max(H, W) > max_side:
                    f = max_side / max(H, W)
                    img = cv2.resize(img, (int(W * f), int(H * f)), interpolation=cv2.INTER_AREA)
                name = f"{ds}__{s.img.stem}"
                cv2.imwrite(str(out / "images" / split / f"{name}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
                (out / "labels" / split / f"{name}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
                rows.append([ds, split, f"{name}.jpg", len(lines), "+".join(sorted(kinds)) or "background"])
                stats[f"{split}_images"] += 1
                stats[f"{split}_instances"] += len(lines)
    with open(out / "manifest.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["dataset", "split", "file", "n_instances", "label_kind"])
        w.writerows(rows)
    (out / "raw_counts.json").write_text(json.dumps(raw_stats, indent=1))
    (out / "data.yaml").write_text(
        f"path: {out.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 1\nnames:\n  0: fish\n")
    print("[detector] done:", dict(stats))
    print(f"[detector] data.yaml -> {out / 'data.yaml'}")


# ------------------------------------------------------------------ viz
def viz(det: Path, n=12, seed=0):
    det = Path(det)
    imgs = [p for p in sorted((det / "images" / "train").glob("*.jpg")) if "__aug" not in p.stem]
    with_lab = [p for p in imgs if (det / "labels" / "train" / f"{p.stem}.txt").read_text().strip()]
    pick = random.Random(seed).sample(with_lab, min(n, len(with_lab)))
    Path("viz").mkdir(exist_ok=True)
    for i, p in enumerate(pick):
        im = cv2.imread(str(p))
        H, W = im.shape[:2]
        for line in (det / "labels" / "train" / f"{p.stem}.txt").read_text().splitlines():
            v = list(map(float, line.split()[1:]))
            pts = (np.array(v, np.float32).reshape(-1, 2) * [W, H]).astype(np.int32)
            cv2.polylines(im, [pts], True, (0, 255, 0), 2)
        cv2.imwrite(f"viz/{i:02d}_{p.stem[:40]}.jpg", im)
    print(f"saved {len(pick)} overlays to viz/")


# ------------------------------------------------------------------ CLI
def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    d = sub.add_parser("detector")
    d.add_argument("--brackish")
    d.add_argument("--ozfish")
    d.add_argument("--deepfish")
    d.add_argument("--out", default="work/det")
    d.add_argument("--max_side", type=int, default=1280)
    d.add_argument("--pseudo_masks", action="store_true")
    d.add_argument("--bg_ratio", type=float, default=1.0, help="max background frames per dataset = ratio x labelled frames")
    d.add_argument("--seed", type=int, default=42)
    c = sub.add_parser("classifier")
    c.add_argument("--f4k", required=True)
    c.add_argument("--out", default="work/cls")
    v = sub.add_parser("viz")
    v.add_argument("--det", required=True)
    v.add_argument("--n", type=int, default=12)
    a = ap.parse_args()
    if a.cmd == "detector":
        samples = []
        if a.brackish:
            samples += load_brackish(a.brackish)
        if a.ozfish:
            samples += load_ozfish(a.ozfish)
        if a.deepfish:
            samples += load_deepfish(a.deepfish)
        assert samples, "no data loaded - check the dataset paths"
        write_detector(samples, Path(a.out), a.max_side, a.pseudo_masks, a.seed, a.bg_ratio)
    elif a.cmd == "viz":
        viz(Path(a.det), a.n)
    else:
        here = Path(__file__).resolve().parent / "f4k_prepare.py"
        sys.exit(subprocess.run([sys.executable, str(here), "--f4k", a.f4k, "--out", a.out]).returncode)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile f4k_prepare.py
#!/usr/bin/env python
"""
f4k_prepare.py -- AquaVision Stage-B data prep (Fish4Knowledge, 23 species)
===========================================================================
Report refs: Sec. 2 (23-class classifier), Sec. 5.2 (macro-F1 because classes are
imbalanced), Sec. 6.1 (augmentation), Sec. 6.2 (70/20/10 split), Sec. 6.3 (class-balanced).

What this does differently from the old write_classifier():
  1. Finds the class folders by itself (works with 'fish4knowedge' typo / nested folders,
     '01_Dascyllus_reticulatus' or 'fish_01' naming) and keeps the species names.
  2. Group-aware 70/20/10 split PER CLASS (same video/clip never in 2 splits) with a
     guaranteed minimum of val/test images for rare classes, so macro-F1 is computable.
  3. Balances ONLY the train split:
        - dominant classes are capped (ceil_mult x target / max_per_class)
        - rare classes are oversampled with classifier-safe underwater augmentation,
          but at most `max_aug_ratio` synthetic copies per real image (no 1 -> 500 blow-up)
        - classes that stay below target are flagged in the report instead of hidden
  4. Never augments val/test. Verifies there is no leakage.
  6. Same three tiers as the detector data (report Sec. 6.1): geometric + underwater photometric +
     occlusion/domain (small synthetic overlay, random erasing, cross-dataset LAB colour transfer toward
     Brackish/DeepFish frames via --ref_det). Cross-dataset fish COMPOSITING is not applied to crops
     (F4K crops have no clean fish cut-out mask) - state this in the report.
  5. Writes report artefacts: class_report.csv, class_balance.png, rare_class_preview.jpg,
     split_counts.json, class_weights.json, data_card.json.

Usage (Colab):
    %cd /content/aquavision
    !python f4k_prepare.py --f4k /content/data/fish4knowledge --out /content/work/cls
    !python f4k_prepare.py --f4k /content/data/fish4knowledge --dry_run     # counts only
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import random
import re
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp"}
CLS_RE = re.compile(r"^(?:fish_?)?(\d{1,2})(?:[_\-](.*))?$", re.I)   # 01_Name | fish_01 | fish01
SPLITS = ("train", "val", "test")


def _sid(c):
    """species id for labels: 'fish_17' / '17_name' -> '17' (c[:2] would give 'fi' for fish_NN folders)"""
    return f"{int(re.search(r'\d+', c).group()):02d}"


def _rng():
    """Same seeded generator as the underwater_aug transforms -> `--seed` really makes the synthetic crops reproducible."""
    import underwater_aug as uw
    return uw._rng()


# ------------------------------------------------------------------ discovery
def find_root(root: Path) -> Path:
    """Descend (max 3 levels) to the folder that directly contains the class folders."""
    root = Path(root)
    for dp, dn, _ in os.walk(root):
        if len(Path(dp).relative_to(root).parts) > 3:
            dn[:] = []
            continue
        if sum(bool(CLS_RE.match(d)) for d in dn) >= 10:
            return Path(dp)
    return root


def norm(s: str) -> str:
    return re.sub(r"[\s\-]+", "_", s.strip().lower())


def scan(root: Path):
    root = find_root(root)
    files, names = defaultdict(list), {}
    for dp, _, fns in os.walk(root):
        for fn in fns:
            p = Path(dp) / fn
            if p.suffix.lower() not in IMG_EXT:
                continue
            rel = p.relative_to(root)
            if any("mask" in x.lower() for x in rel.parts[:-1]):
                continue
            for part in reversed(rel.parent.parts):
                m = CLS_RE.match(part)
                if m:
                    n = int(m.group(1))
                    files[n].append(p)
                    if m.group(2):
                        names.setdefault(n, norm(m.group(2)))
                    break
    classes = {}
    for n in sorted(files):
        label = f"{n:02d}_{names[n]}" if n in names else f"fish_{n:02d}"
        classes[label] = sorted(files[n])
    print(f"[f4k] root={root}")
    print(f"[f4k] {len(classes)} classes, {sum(len(v) for v in classes.values())} crops")
    if len(classes) != 23:
        print(f"  [warn] expected 23 species folders, found {len(classes)} -> check the path / unzip")
    return classes


def group_of(p: Path) -> str:
    """fish_<clip-id>_<n>.png -> clip-id (same clip must stay in one split)."""
    m = re.match(r"fish_(\d+)", p.stem, re.I)
    return m.group(1) if m else p.stem


# ------------------------------------------------------------------ split
def split_class(files, seed=42, ratios=(0.7, 0.2, 0.1), min_eval=2):
    """Per-class split. Returns ({split: [paths]}, mode). Guarantees val/test >= min_eval
    (when the class is big enough) and keeps clip groups intact whenever possible."""
    n = len(files)
    out = {s: [] for s in SPLITS}
    if n < 3:                                           # cannot evaluate; keep for training
        out["train"] = list(files)
        return out, "train-only"
    n_val = max(min_eval, round(ratios[1] * n))
    n_test = max(min_eval, round(ratios[2] * n))
    while n - n_val - n_test < max(1, n // 2) and (n_val > 1 or n_test > 1):
        if n_val >= n_test:
            n_val -= 1
        else:
            n_test -= 1
    tgt = {"train": n - n_val - n_test, "val": n_val, "test": n_test}

    groups = defaultdict(list)
    for f in files:
        groups[group_of(f)].append(f)
    rng = random.Random(seed)
    keys = sorted(groups)
    rng.shuffle(keys)
    keys.sort(key=lambda k: -len(groups[k]))            # big groups first (stable -> random ties)

    mode = "group"
    if len(keys) >= 10:
        cur = {s: 0 for s in SPLITS}
        for k in keys:
            s = max(SPLITS, key=lambda x: tgt[x] - cur[x])      # largest absolute deficit
            out[s].extend(groups[k])
            cur[s] += len(groups[k])
        if not out["val"] or not out["test"]:                   # one giant clip ate everything
            mode = "image"
    else:
        mode = "image"
    if mode == "image":                                         # few clips -> per-image split
        out = {s: [] for s in SPLITS}
        items = sorted(files)
        rng.shuffle(items)
        out["test"] = items[:tgt["test"]]
        out["val"] = items[tgt["test"]:tgt["test"] + tgt["val"]]
        out["train"] = items[tgt["test"] + tgt["val"]:]
    return out, mode


# ------------------------------------------------------------------ augmentation
class ScaleCropJitter:
    """Random crop of 75-100 % area, resized back (tiny crops -> zoom/scale tier of Sec. 6.1)."""
    def __init__(self, p=0.7, area=(0.75, 1.0)):
        self.p, self.area = p, area

    def __call__(self, image=None, **kw):
        rng = _rng()
        if rng.random() >= self.p:
            return {"image": image}
        h, w = image.shape[:2]
        f = np.sqrt(rng.uniform(*self.area))
        ch, cw = max(4, int(h * f)), max(4, int(w * f))
        y, x = int(rng.integers(0, h - ch + 1)), int(rng.integers(0, w - cw + 1))
        return {"image": cv2.resize(image[y:y + ch, x:x + cw], (w, h), interpolation=cv2.INTER_LINEAR)}


class BrightnessContrast:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, image=None, **kw):
        rng = _rng()
        if rng.random() >= self.p:
            return {"image": image}
        a, b = rng.uniform(0.8, 1.2), rng.uniform(-20, 20)
        return {"image": np.clip(image.astype(np.float32) * a + b, 0, 255).astype(np.uint8)}

def build_cls_aug(strength=1.0, ref_stats=None):
    """Classifier-SAFE underwater pipeline. Differences vs the detector pipeline:
    colour cast is removed/near-zero (body colour is a species cue), domain transfer is weak and rare,
    erasing is small, and the synthetic rock/coral overlay is tiny (would hide the whole fish on a 40 px crop)."""
    from underwater_aug import (FlipRotate, Sequential, UWColorCast, UWHaze, UWMotionBlur,
                                UWRandomErasing, UWStochasticCLAHE, UWTurbidity,
                                UWVignetteIllumination, UWDomainTransfer, UWSyntheticOverlay)
    s = lambda p: float(min(1.0, p * strength))
    return Sequential([
        FlipRotate(max_deg=15),
        ScaleCropJitter(),
        BrightnessContrast(),
        # Almost no cast: body colour is the species cue. Even the earlier "mild" setting (0.8-1.0 / 0-0.04)
        # moved yellow fish by ~20 deg of hue in a synthetic test. Detector pipeline keeps its strong cast.
        UWColorCast(red_atten=(0.92, 1.0), hue_pull=(0.0, 0.0), p=s(0.4)),
        UWTurbidity(sat_scale=(0.6, 0.95), p=s(0.4)),
        UWStochasticCLAHE(p=s(0.4)),
        UWHaze(density=(0.1, 0.2), p=s(0.12)),
        UWMotionBlur(kernel=(3, 5), p=s(0.1)),
        UWVignetteIllumination(p=s(0.12)),
        UWRandomErasing(n_holes=(1, 2), size=(0.05, 0.15), p=s(0.3)),
        # tier (iii): synthetic occlusion (small blob, rare) + cross-dataset colour transfer (weak, rarer)
        UWSyntheticOverlay(n_blobs=(1, 1), size=(0.03, 0.08), p=s(0.08)),
    ] + ([UWDomainTransfer(ref_stats=ref_stats, alpha=(0.1, 0.3), p=s(0.15))] if ref_stats else []))


def load_ref_stats(det_dir, n=300, seed=0):
    """LAB colour statistics of detector frames (Brackish/DeepFish originals) for cross-dataset domain transfer."""
    if not det_dir:
        return None
    imgs = [p for p in sorted((Path(det_dir) / "images" / "train").glob("*.jpg"))
            if not any(t in p.stem for t in ("__aug", "__ovl", "__comp"))]
    if not imgs:
        print("[f4k] no detector images found -> domain transfer skipped")
        return None
    from underwater_aug import compute_lab_stats
    st = compute_lab_stats(imgs, n=n, seed=seed)
    print(f"[f4k] domain-transfer reference stats from {len(st)} detector frames")
    return st


def make_aug(aug, src):
    for _ in range(5):                                  # make sure the copy really differs
        out = aug(src.copy())
        # really different AND not destroyed (all-black / all-white) -> species cues survive
        if out.shape == src.shape and not np.array_equal(out, src) and 8 < float(out.mean()) < 247:
            return out
    return out


# ------------------------------------------------------------------ main build
def build(a):
    classes = scan(Path(a.f4k))
    assert classes, "No class folders found - check --f4k path"
    rng = random.Random(a.seed)

    # ---- 1. split
    splits, modes = {}, {}
    for cls, files in classes.items():
        splits[cls], modes[cls] = split_class(files, a.seed, min_eval=a.min_eval)
    raw = {c: len(v) for c, v in classes.items()}
    tr0 = {c: len(splits[c]["train"]) for c in classes}

    # ---- 2. targets
    target = int(np.median(list(tr0.values()))) if a.target == "median" else int(a.target)
    target = max(1, min(target, a.max_per_class))
    ceiling = min(a.max_per_class, int(a.ceil_mult * target))
    print(f"[balance] target={target} per class | ceiling={ceiling} | max_aug_ratio={a.max_aug_ratio}")
    print("          (no cap at all: --ceil_mult 1000 --max_per_class 100000 ; --max_per_class alone does NOT remove the "
          "ceil_mult x target cap)")

    plan = {}
    for c in classes:
        n = tr0[c]
        keep = min(n, ceiling)
        synth = 0
        if keep < target and keep > 0:
            synth = min(target - keep, int(a.max_aug_ratio * keep))
        synth = max(synth, int(a.extra_aug * keep))   # every class also gets some augmented copies (PDF Sec 6.1)
        plan[c] = (keep, synth)

    dropped = sum(max(0, tr0[c] - plan[c][0]) for c in classes)
    print(f"[balance] real TRAIN crops thrown away by the cap: {dropped} of {sum(tr0.values())} "
          f"({100 * dropped / max(1, sum(tr0.values())):.0f} %)  <- keep this small for the head classes")
    img_mode = [c for c in classes if modes[c] == "image"]
    if img_mode:
        print(f"[note] {len(img_mode)} classes are split PER IMAGE (too few clips): {[_sid(c) for c in img_mode]} "
              f"-> clips may be shared between train and test; also report macro-F1 over the clip-split classes only")

    if a.dry_run:
        print_table(classes, raw, splits, plan, modes, target)
        return

    # ---- 3. write
    out = Path(a.out)
    if out.exists():
        shutil.rmtree(out)
    import underwater_aug as uw
    uw.set_seed(a.seed)                                   # reproducible synthetic crops (same seed -> same files)
    aug = build_cls_aug(a.strength, load_ref_stats(a.ref_det))
    synth_log = {}
    for c in classes:
        for s in SPLITS:
            (out / s / c).mkdir(parents=True, exist_ok=True)
        for s in ("val", "test"):
            for p in splits[c][s]:
                shutil.copy2(p, out / s / c / p.name)
        tr = list(splits[c]["train"])
        keep, n_syn = plan[c]
        if len(tr) > keep:
            tr = rng.sample(tr, keep)                   # random undersample of dominant classes
        for p in tr:
            shutil.copy2(p, out / "train" / c / p.name)
        # synthetic copies: round-robin over real images so every source is used equally often
        order, made = [], []
        for k in range(n_syn):
            if not order:
                order = tr[:]
                rng.shuffle(order)
            src_p = order.pop()
            src = cv2.imread(str(src_p))
            if src is None:
                continue
            name = f"aug{k:05d}__{src_p.stem}.png"
            cv2.imwrite(str(out / "train" / c / name), make_aug(aug, src))
            made.append(name)
        synth_log[c] = made
        print(f"  [{c:34s}] raw={raw[c]:6d} train {tr0[c]:5d} -> {len(tr):5d} real + {len(made):5d} synth")

    # ---- 4. leakage check
    leaks = 0
    for c in classes:
        g = {s: {group_of(p) for p in splits[c][s]} for s in SPLITS}
        f = {s: {p.name for p in splits[c][s]} for s in SPLITS}
        for x, y in (("train", "val"), ("train", "test"), ("val", "test")):
            leaks += len(f[x] & f[y])
            if modes[c] == "group":
                leaks += len(g[x] & g[y])
    print(f"[check] leakage between splits: {'NONE' if leaks == 0 else str(leaks) + ' !!'}")

    # ---- 5. artefacts
    rows = print_table(classes, raw, splits, plan, modes, target, synth_log)
    with open(out / "class_report.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["class", "raw", "train_real", "train_synth", "train_final", "val", "test",
                    "synth_per_real", "split_mode", "flags"])
        w.writerows(rows)
    (out / "split_counts.json").write_text(json.dumps(
        {c: {s: len(splits[c][s]) for s in SPLITS} for c in classes}, indent=1))
    # effective-number class weights (for a custom loss / sampler; computed on REAL train counts)
    beta = 0.999
    wts = {c: (1 - beta) / (1 - beta ** max(1, plan[c][0])) for c in classes}
    z = sum(wts.values())
    (out / "class_weights.json").write_text(json.dumps({c: round(len(wts) * v / z, 4) for c, v in wts.items()}, indent=1))
    (out / "data_card.json").write_text(json.dumps(vars(a) | {"target": target, "ceiling": ceiling,
                                                              "n_classes": len(classes)}, indent=1))
    plot_balance(out, classes, tr0, plan)
    rare_preview(out, classes, raw, synth_log, extra_ids=[int(x) for x in a.preview_ids.split(",") if x.strip()])
    print(f"[classifier] done -> {out}")


def print_table(classes, raw, splits, plan, modes, target, synth_log=None):
    print(f"\n{'class':34s} {'raw':>6s} {'trReal':>6s} {'trSyn':>6s} {'trAll':>6s} {'val':>5s} {'test':>5s}  flags")
    rows = []
    for c in classes:
        keep, syn = plan[c]
        syn = len(synth_log[c]) if synth_log else syn
        fl = []
        if raw[c] < 30:
            fl.append("VERY-LOW(<30 raw)")
        elif raw[c] < 150:
            fl.append("LOW")
        if keep > 0 and syn / keep >= 5:
            fl.append(f"heavy-oversample x{syn / keep:.0f}")
        if keep + syn < 0.9 * target:
            fl.append("below-target")
        if len(splits[c]["test"]) < 5:
            fl.append("test<5:F1-noisy")
        if modes[c] != "group":
            fl.append(f"split={modes[c]}")
        if len(splits[c]["train"]) > keep:
            fl.append("capped")
        r = [c, raw[c], keep, syn, keep + syn, len(splits[c]["val"]), len(splits[c]["test"]),
             round(syn / keep, 1) if keep else 0, modes[c], " ".join(fl)]
        rows.append(r)
        print(f"{c:34s} {raw[c]:6d} {keep:6d} {syn:6d} {keep + syn:6d} {r[5]:5d} {r[6]:5d}  {' '.join(fl)}")
    print()
    return rows


def plot_balance(out, classes, tr0, plan):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    names = list(classes)
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - 0.2, [tr0[c] for c in names], 0.4, label="train before (real)")
    ax.bar(x + 0.2, [plan[c][0] + plan[c][1] for c in names], 0.4, label="train after (real+aug)")
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels([_sid(c) for c in names])
    ax.set_xlabel("species id")
    ax.set_ylabel("train crops (log)")
    ax.set_title("Fish4Knowledge: class balance before / after")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "class_balance.png", dpi=150)
    plt.close(fig)


def rare_preview(out, classes, raw, synth_log, n_cls=4, n_aug=5, sz=128, extra_ids=()):
    """Preview grid: the rarest classes + any extra species ids (weak mid-size classes) -> LOOK at it:
    the fish must still be recognisable (strong haze/erasing can delete the species cues)."""
    rarest = sorted(classes, key=lambda c: raw[c])[:n_cls]
    for c in classes:
        cid = int(re.search(r"\d+", c).group())
        if cid in extra_ids and c not in rarest:
            rarest.append(c)
    rows = []
    for c in rarest:
        real = sorted((out / "train" / c).glob("*.png")) + sorted((out / "train" / c).glob("*.jpg"))
        real = [p for p in real if not p.name.startswith("aug")]
        if not real:
            continue
        tiles = [cv2.imread(str(real[0]))] + [cv2.imread(str(out / "train" / c / n)) for n in synth_log[c][:n_aug]]
        row = []
        for t in tiles:
            if t is None:
                t = np.zeros((sz, sz, 3), np.uint8)
            h, w = t.shape[:2]
            f = sz / max(h, w)
            t = cv2.resize(t, (max(1, int(w * f)), max(1, int(h * f))))
            canvas = np.zeros((sz, sz, 3), np.uint8)
            canvas[:t.shape[0], :t.shape[1]] = t
            row.append(canvas)
        row += [np.zeros((sz, sz, 3), np.uint8)] * (n_aug + 1 - len(row))
        img = np.hstack(row)
        cv2.putText(img, c, (4, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)
        rows.append(img)
    if rows:
        cv2.imwrite(str(out / "rare_class_preview.jpg"), np.vstack(rows))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--f4k", required=True, help="folder containing the 23 species folders (nested ok)")
    ap.add_argument("--out", default="/content/work/cls")
    ap.add_argument("--target", default="median", help="'median' or an int: train crops per class after balancing")
    ap.add_argument("--max_per_class", type=int, default=1500, help="hard cap on train crops per class")
    ap.add_argument("--ceil_mult", type=float, default=3.0, help="dominant classes cut to ceil_mult x target "
                    "(ceiling = min(max_per_class, ceil_mult x target)); default 3 x median is VERY aggressive")
    ap.add_argument("--max_aug_ratio", type=float, default=12, help="max synthetic copies per real train image")
    ap.add_argument("--min_eval", type=int, default=2, help="min val and test images per class")
    ap.add_argument("--strength", type=float, default=1.0, help="scales augmentation probabilities")
    ap.add_argument("--ref_det", default="/content/work/det", help="detector folder whose frames give the domain-transfer colour stats ('' = off)")
    ap.add_argument("--extra_aug", type=float, default=0.0, help="extra synthetic copies per REAL train image for ALL classes (0.2 = +20%%)")
    ap.add_argument("--preview_ids", default="", help="extra species ids for rare_class_preview.jpg, e.g. 8,13,14")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--dry_run", action="store_true", help="only print the plan, write nothing")
    build(ap.parse_args())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile augment_offline.py
#!/usr/bin/env python
"""
augment_offline.py -- cache underwater-augmented copies of the TRAIN split (report Sec. 6.1)
===========================================================================================
Three kinds of cached copies (val/test are NEVER touched):

  __aug{k}   tier (ii) photometric + tier (iii) random erasing + cross-dataset LAB colour
             transfer ("domain mixing").  Labels are copied unchanged.
  __ovl0     same as above PLUS a synthetic rock/coral overlay (occlusion).  Labels unchanged.
  __comp0    cross-dataset COMPOSITING: a fish cut out (polygon mask) of an image from ANOTHER
             dataset is pasted into this image.  New polygon is appended to the labels.
             Needs real/pseudo masks (polygons with >4 vertices) -> run prepare_data with
             --pseudo_masks, otherwise this step is skipped with a message.

Tier (i) geometric aug + mosaic/mixup stay ONLINE in Ultralytics (see train_aug_config.py).

Report: "overlays ... disabled in the final epochs" -> two image lists are written:
    lists/train_full.txt  (everything)          -> data_full.yaml   (phase 1)
    lists/train_late.txt  (no __ovl / __comp)   -> data_late.yaml   (last ~15 epochs)

    python augment_offline.py --det /content/work/det --copies 2 --domain_mix \
           --overlay_frac 0.1 --composite_frac 0.1 --clean
"""
import argparse
import random
import re
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

import underwater_aug as uw
from underwater_aug import Sequential, UWSyntheticOverlay, build_underwater_transforms, compute_lab_stats

TAGS = ("__aug", "__ovl", "__comp")


def is_copy(p: Path) -> bool:
    return any(t in p.stem for t in TAGS)


def clean(det: Path):
    n = 0
    for p in list((det / "images" / "train").glob("*.jpg")):
        if is_copy(p):
            p.unlink()
            (det / "labels" / "train" / f"{p.stem}.txt").unlink(missing_ok=True)
            n += 1
    shutil.rmtree(det / "lists", ignore_errors=True)
    print(f"cleaned {n} old cached copies")


def changed(pipe, im, tries=4):
    out = pipe(im.copy())
    for _ in range(tries):
        if not np.array_equal(out, im):
            break
        out = pipe(im.copy())
    return out


# ------------------------------------------------------------------ compositing
def read_polys(lab: Path):
    polys = []
    for line in lab.read_text().splitlines():
        v = line.split()
        if len(v) >= 7:
            polys.append(np.array(list(map(float, v[1:])), np.float32).reshape(-1, 2))
    return polys


def build_donors(det: Path, imgs, per_ds=1500, seed=0):
    """Donor fish = polygons with >4 vertices (real or GrabCut masks, not plain rectangles)."""
    donors = defaultdict(list)
    for p in imgs:
        ds = p.stem.split("__")[0]
        lab = det / "labels" / "train" / f"{p.stem}.txt"
        for poly in read_polys(lab):
            if len(poly) <= 4:
                continue
            (x0, y0), (x1, y1) = poly.min(0), poly.max(0)
            if 0.002 <= (x1 - x0) * (y1 - y0) <= 0.4:
                donors[ds].append((p, poly))
    rs = random.Random(seed)
    for ds in donors:
        rs.shuffle(donors[ds])
        donors[ds] = donors[ds][:per_ds]
    return donors


def paste_fish(target, donor_img, poly, rng):
    """Cut `poly` out of donor_img, scale/flip/colour-match, blend into target.
    Returns (image, new normalised polygon) or None."""
    H, W = target.shape[:2]
    h, w = donor_img.shape[:2]
    pts = poly * np.array([w, h], np.float32)
    x0, y0 = np.floor(pts.min(0)).astype(int) - 2
    x1, y1 = np.ceil(pts.max(0)).astype(int) + 2
    x0, y0, x1, y1 = max(0, x0), max(0, y0), min(w, x1), min(h, y1)
    cw, ch = x1 - x0, y1 - y0
    if cw < 8 or ch < 8:
        return None
    crop = donor_img[y0:y1, x0:x1].copy()
    pc = pts - np.array([x0, y0], np.float32)
    mask = np.zeros((ch, cw), np.uint8)
    cv2.fillPoly(mask, [pc.astype(np.int32)], 255)
    if rng.random() < 0.5:                                      # horizontal flip
        crop, mask = crop[:, ::-1].copy(), mask[:, ::-1].copy()
        pc[:, 0] = cw - pc[:, 0]
    s = rng.uniform(0.6, 1.4) * W / w                           # relative to image width
    s = min(s, 0.6 * W / cw, 0.6 * H / ch)
    nw, nh = max(8, int(cw * s)), max(8, int(ch * s))
    if nw >= W or nh >= H:
        return None
    crop = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_LINEAR)
    # colour-match the cut-out to the target scene (Reinhard in LAB, partial)
    lt = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype(np.float32)
    lc = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
    inside = mask > 127
    if inside.sum() > 10:
        mc, sc = lc[inside].mean(0), lc[inside].std(0) + 1e-6
        mt, st = lt.reshape(-1, 3).mean(0), lt.reshape(-1, 3).std(0) + 1e-6
        a = rng.uniform(0.4, 0.8)
        lc = a * ((lc - mc) / sc * st + mt) + (1 - a) * lc
        crop = cv2.cvtColor(np.clip(lc, 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR)
    ox, oy = int(rng.integers(0, W - nw + 1)), int(rng.integers(0, H - nh + 1))
    alpha = cv2.GaussianBlur(mask, (5, 5), 0).astype(np.float32)[..., None] / 255.0
    out = target.copy()
    reg = out[oy:oy + nh, ox:ox + nw].astype(np.float32)
    out[oy:oy + nh, ox:ox + nw] = np.clip(crop * alpha + reg * (1 - alpha), 0, 255).astype(np.uint8)
    newp = pc * np.array([nw / cw, nh / ch], np.float32) + np.array([ox, oy], np.float32)
    return out, np.clip(newp / np.array([W, H], np.float32), 0, 1)


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det", required=True)
    ap.add_argument("--copies", type=int, default=2, help="__aug copies per train image")
    ap.add_argument("--strength", type=float, default=1.5, help="scales all probabilities")
    ap.add_argument("--domain_mix", action="store_true", help="cross-dataset LAB colour transfer")
    ap.add_argument("--overlay_frac", type=float, default=0.1, help="fraction of train images that get an __ovl copy")
    ap.add_argument("--composite_frac", type=float, default=0.0, help="fraction that get a cross-dataset __comp copy")
    ap.add_argument("--only_labelled", action="store_true", help="skip background frames as sources")
    ap.add_argument("--clean", action="store_true", help="delete existing cached copies first")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    det = Path(a.det)
    if a.clean:
        clean(det)
    imgs = [p for p in sorted((det / "images" / "train").glob("*.jpg")) if not is_copy(p)]
    if a.only_labelled:
        imgs = [p for p in imgs if (det / "labels" / "train" / f"{p.stem}.txt").read_text().strip()]
    random.seed(a.seed)
    np.random.seed(a.seed)
    uw.set_seed(a.seed)
    rng = np.random.default_rng(a.seed)

    # ---- domain statistics (needs >= 2 datasets to be meaningful)
    by_ds = defaultdict(list)
    for p in imgs:
        by_ds[p.stem.split("__")[0]].append(p)
    stats_by_ds = {}
    if a.domain_mix:
        if len(by_ds) < 2:
            print("[warn] --domain_mix needs >= 2 datasets in train; colour transfer skipped")
        else:
            stats_by_ds = {k: compute_lab_stats(v) for k, v in by_ds.items()}
            print("domain pools:", {k: len(v) for k, v in stats_by_ds.items()})

    def pool_for(src_ds):
        return [s for k, v in stats_by_ds.items() if k != src_ds for s in v] or None

    n = {"aug": 0, "ovl": 0, "comp": 0}
    ovl_set = set(random.sample(range(len(imgs)), int(a.overlay_frac * len(imgs)))) if a.overlay_frac > 0 else set()
    comp_set = set(random.sample(range(len(imgs)), int(a.composite_frac * len(imgs)))) if a.composite_frac > 0 else set()

    donors = {}
    if comp_set:
        donors = build_donors(det, imgs, seed=a.seed)
        print("composite donor pools (fish with real/pseudo masks):", {k: len(v) for k, v in donors.items()})
        if len([k for k, v in donors.items() if v]) < 2:
            print("[warn] fewer than 2 datasets have mask polygons -> compositing skipped "
                  "(re-run prepare_data.py with --pseudo_masks)")
            comp_set = set()

    for i, p in enumerate(imgs):
        ds = p.stem.split("__")[0]
        im = cv2.imread(str(p))
        if im is None:
            continue
        lab_txt = (det / "labels" / "train" / f"{p.stem}.txt").read_text()
        ref = pool_for(ds) if stats_by_ds else None
        pipe = Sequential(build_underwater_transforms(ref_stats=ref, strength=a.strength, overlay=False))

        def save(tag, k, img, text):
            name = f"{p.stem}__{tag}{k}"
            cv2.imwrite(str(det / "images" / "train" / f"{name}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            (det / "labels" / "train" / f"{name}.txt").write_text(text)
            n[tag] += 1

        for k in range(a.copies):
            save("aug", k, changed(pipe, im), lab_txt)
        if i in ovl_set:
            opipe = Sequential([UWSyntheticOverlay(p=1.0)] + build_underwater_transforms(
                ref_stats=ref, strength=a.strength, overlay=False))
            save("ovl", 0, changed(opipe, im), lab_txt)
        if i in comp_set:
            other = [d for d in donors if d != ds and donors[d]]
            if other:
                dp, poly = random.choice(donors[random.choice(other)])
                dimg = cv2.imread(str(dp))
                r = paste_fish(im, dimg, poly, rng) if dimg is not None else None
                if r is not None:
                    out, newp = r
                    line = "0 " + " ".join(f"{v:.6f}" for v in newp.reshape(-1))
                    save("comp", 0, out, lab_txt.rstrip("\n") + ("\n" if lab_txt.strip() else "") + line + "\n")

    # ---- full / late lists (+ yaml) so overlays & composites can be switched off for the last epochs
    allp = sorted((det / "images" / "train").glob("*.jpg"))
    late = [p for p in allp if "__ovl" not in p.stem and "__comp" not in p.stem]
    (det / "lists").mkdir(exist_ok=True)
    (det / "lists" / "train_full.txt").write_text("\n".join(str(p.resolve()) for p in allp) + "\n")
    (det / "lists" / "train_late.txt").write_text("\n".join(str(p.resolve()) for p in late) + "\n")
    y = (det / "data.yaml").read_text()
    for nm in ("full", "late"):
        (det / f"data_{nm}.yaml").write_text(re.sub(r"(?m)^train:.*$", f"train: {(det / 'lists' / f'train_{nm}.txt').resolve()}", y))
    print(f"wrote {n['aug']} __aug + {n['ovl']} __ovl + {n['comp']} __comp copies "
          f"(originals: {len(imgs)}) -> train total {len(allp)}, late-phase list {len(late)}")
    print("yaml: data_full.yaml (phase 1) / data_late.yaml (final epochs, no overlays/composites)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_aug_config.py
"""
train_aug_config.py -- hand-off for the training teammates (report Sec. 6.1 / 6.3)
==================================================================================
NOT executed during data prep (needs `pip install ultralytics` + GPU). Kept here so the online
half of the augmentation scheme is documented next to the offline half.

Tier (i) GEOMETRIC is done online by Ultralytics:
    flip, small rotation, scale/zoom, bounding-box-safe crop (translate+scale, boxes/masks are
    transformed and clipped together), mosaic, mixup.
Tier (ii)/(iii) photometric + occlusion + domain are already cached offline (augment_offline.py).

"Mosaic and MixUp are disabled in the final epochs"  -> close_mosaic (Ultralytics switches
    mosaic / mixup / copy_paste off for the last N epochs).
"Synthetic overlays disabled in the final epochs"    -> phase 2 trains on data_late.yaml, whose
    image list excludes the __ovl (overlay) and __comp (composite) copies.
"""
from pathlib import Path

GEOMETRIC = dict(
    fliplr=0.5, flipud=0.0,
    degrees=10.0,            # small rotation
    translate=0.1, scale=0.5,  # zoom / crop (box-safe)
    shear=0.0, perspective=0.0,
    mosaic=1.0, mixup=0.1, copy_paste=0.0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,   # Ultralytics' own mild HSV jitter on top of the cached UW copies
)


def train_detector(det_dir="/content/work/det", weights="yolo11s-seg.pt", epochs=100,
                   late_epochs=15, imgsz=640, batch=16, patience=25,
                   project="/content/drive/MyDrive/aquavision/runs", name="det"):
    """Two-phase detector training (v7 fixes, from the segmenter baseline: best epoch was 20/100).

    * patience=25 in phase 1 -> stops once val fitness stops improving (Ultralytics default of 100
      would never trigger inside an 85-epoch phase).
    * phase 2 is a NEW train() call, so LR warm-up would restart and wreck the converged model.
      Fix: warmup_epochs=0, low lr0, and an EXPLICIT optimizer (with optimizer='auto' Ultralytics
      ignores lr0 and picks its own LR).
    * batch=16 / imgsz=640 / yolo11s-seg.pt to match the baseline run (fair comparison).
    Alternative (single run, no overlay switch-off): epochs=100, close_mosaic=15, patience=25 on data_full.yaml.
    """
    from ultralytics import YOLO
    det = Path(det_dir)
    m = YOLO(weights)
    # phase 1: everything (offline copies incl. overlays/composites), mosaic+mixup ON
    m.train(data=str(det / "data_full.yaml"), epochs=epochs - late_epochs, imgsz=imgsz, batch=batch,
            patience=patience, close_mosaic=0, project=project, name=f"{name}_p1", **GEOMETRIC)
    # phase 2: final epochs, mosaic/mixup OFF and no overlays/composites in the image list
    m2 = YOLO(str(m.trainer.best))
    late = dict(GEOMETRIC, mosaic=0.0, mixup=0.0, copy_paste=0.0)
    m2.train(data=str(det / "data_late.yaml"), epochs=late_epochs, imgsz=imgsz, batch=batch,
             patience=late_epochs,                      # no early stop inside the short phase
             optimizer="SGD", lr0=0.001, lrf=0.1, warmup_epochs=0,
             project=project, name=f"{name}_p2", **late)
    return m2


# Online (label-safe) augmentation for the classifier - brief P3: the baseline memorised the train set
# (train loss 0.000 at epoch 55, val loss lowest at epoch 28). These change every epoch.
# Compare with the baseline's args.yaml and change ONLY the data + these flags. Argument names follow
# Ultralytics' classification trainer - check `yolo cfg` if your version complains.
CLS_ONLINE = dict(fliplr=0.5, flipud=0.0,          # fish look the same mirrored; never flip vertically
                  scale=0.5,                       # mild scale / crop
                  hsv_h=0.01, hsv_s=0.3, hsv_v=0.2,   # a little HSV jitter (hue is a species cue -> keep it small)
                  degrees=0.0)


def train_classifier(cls_dir="/content/work/cls", weights="yolo11s-cls.pt", epochs=60, imgsz=224, batch=64,
                     project="/content/drive/MyDrive/aquavision/runs", name="cls_aug", device=0):
    # train/ is already class-balanced offline (f4k_prepare.py); val/ and test/ are untouched.
    # Same model + schedule as the baseline (yolo11s-cls, 224, batch 64, 60 epochs) -> fair comparison.
    from ultralytics import YOLO
    m = YOLO(weights)
    m.train(data=cls_dir, epochs=epochs, imgsz=imgsz, batch=batch, device=device,
            project=project, name=name, **CLS_ONLINE)
    return m


In [ ]:
%%writefile turbid_split.py
#!/usr/bin/env python
"""
turbid_split.py -- held-out clean-vs-turbid split (report Sec. 6.1, last sentence)
==================================================================================
"Value is verified by per-category ablation and a held-out clean-vs-turbid split."
Brackish/DeepFish carry no visibility label in this copy, so turbidity is MEASURED per image:
    dark-channel mean (haze veil)  UP,   RMS contrast DOWN,   mean saturation DOWN,   sharpness DOWN
Each cue is z-scored against that dataset's TRAIN originals, averaged -> turbidity score.
Threshold = quantile `--q` (default median) of the dataset's train scores, so each dataset
is split into its own clearer / murkier half (no dataset-identity shortcut). `--global_thr`
standardises over all datasets together instead.

Only ORIGINAL images are scored (augmented __aug/__ovl/__comp copies are ignored).
Writes into <det>/lists:
    test_clean.txt  test_turbid.txt  val_clean.txt  val_turbid.txt  train_clean.txt
    data_test_clean.yaml / data_test_turbid.yaml   (val+test point to that subset -> model.val())
    data_trainclean.yaml  (train on CLEAN only, val=val_clean, test=test_turbid -> held-out robustness)
and <det>/turbidity.csv (dataset, split, file, score, label).
"""
import argparse
import csv
import re
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

TAGS = ("__aug", "__ovl", "__comp", "__abl")


def cues(path):
    im = cv2.imread(str(path))
    if im is None:
        return None
    f = 320.0 / max(im.shape[:2])
    if f < 1:
        im = cv2.resize(im, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)
    dark = cv2.erode(im.min(axis=2), np.ones((9, 9), np.uint8)).mean()
    return np.array([dark, gray.std(), hsv[..., 1].mean(), cv2.Laplacian(gray, cv2.CV_64F).var()], np.float64)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det", required=True)
    ap.add_argument("--q", type=float, default=0.5, help="train-score quantile separating clean/turbid")
    ap.add_argument("--global_thr", action="store_true", help="one threshold over all datasets")
    a = ap.parse_args()
    det = Path(a.det)
    SIGN = np.array([+1, -1, -1, -1], np.float64)       # +dark, -contrast, -saturation, -sharpness

    items = defaultdict(list)                            # dataset -> [(split, path, cue)]
    for split in ("train", "val", "test"):
        for p in sorted((det / "images" / split).glob("*.jpg")):
            if any(t in p.stem for t in TAGS):
                continue
            c = cues(p)
            if c is not None:
                items[p.stem.split("__")[0]].append((split, p, c))
    assert items, "no images found - run prepare_data.py first"

    groups = {"ALL": [x for v in items.values() for x in v]} if a.global_thr else items
    rows, thr_info = [], {}
    for g, lst in groups.items():
        tr = np.stack([c for s, _, c in lst if s == "train"])
        mu, sd = tr.mean(0), tr.std(0) + 1e-6
        score = lambda c: float((((c - mu) / sd) * SIGN).mean())
        thr = float(np.quantile([score(c) for s, _, c in lst if s == "train"], a.q))
        thr_info[g] = round(thr, 3)
        for s, p, c in lst:
            sc = score(c)
            ds = p.stem.split("__")[0]
            rows.append([ds, s, p.name, round(sc, 4), "turbid" if sc > thr else "clean"])
    print("thresholds:", thr_info)

    with open(det / "turbidity.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["dataset", "split", "file", "score", "label"])
        w.writerows(rows)
    lst_dir = det / "lists"
    lst_dir.mkdir(exist_ok=True)
    out = defaultdict(list)
    for ds, s, fn, sc, lab in rows:
        out[(s, lab)].append(str((det / "images" / s / fn).resolve()))
    for (s, lab), v in out.items():
        (lst_dir / f"{s}_{lab}.txt").write_text("\n".join(v) + "\n")
    y = (det / "data.yaml").read_text()

    def with_(**kw):
        t = y
        for k, v in kw.items():
            t = re.sub(rf"(?m)^{k}:.*$", f"{k}: {(lst_dir / v).resolve()}", t)
        return t

    (det / "data_test_clean.yaml").write_text(with_(val="test_clean.txt", test="test_clean.txt"))
    (det / "data_test_turbid.yaml").write_text(with_(val="test_turbid.txt", test="test_turbid.txt"))
    (det / "data_trainclean.yaml").write_text(with_(train="train_clean.txt", val="val_clean.txt", test="test_turbid.txt"))

    import collections
    cnt = collections.Counter((r[0], r[1], r[4]) for r in rows)
    print(f"{'dataset':10s} {'split':6s} {'clean':>7s} {'turbid':>7s}")
    for ds in sorted({r[0] for r in rows}):
        for s in ("train", "val", "test"):
            print(f"{ds:10s} {s:6s} {cnt[(ds, s, 'clean')]:7d} {cnt[(ds, s, 'turbid')]:7d}")
    print("wrote lists/*, data_test_clean.yaml, data_test_turbid.yaml, data_trainclean.yaml, turbidity.csv")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ablation_prep.py
#!/usr/bin/env python
"""
ablation_prep.py -- per-category augmentation ablation sets (report Sec. 6.1, last sentence)
==========================================================================================
For ONE fixed random subset of labelled TRAIN originals (so every arm sees the same sources),
make one cached copy per image with exactly ONE augmentation category switched on (p=1):

    colorcast  turbidity  clahe  erasing  haze  motionblur  vignette   (tier ii / iii-erasing)
    overlay                                                              (tier iii synthetic overlay)
    domain                                                               (tier iii cross-dataset colour transfer)
    composite                                                            (tier iii cross-dataset fish paste)

Arms (each = a train image list + a yaml; val/test are the normal untouched ones):
    abl_baseline    originals only
    abl_control_dup originals + the SAME subset duplicated (control: gain is not just 'more iterations')
    abl_<category>  originals + that category's copies
    abl_all         originals + all categories (the combined effect)

Copies live in <out>/<category>/{images,labels}/train/<stem>__abl.jpg (labels copied; composite appends a polygon).
Train each arm with the same short schedule (ablation_run.py) and compare mAP / IoU on
data_test_clean.yaml vs data_test_turbid.yaml (from turbid_split.py).

    python ablation_prep.py --det /content/work/det --out /content/work/ablation --frac 0.15 --max_per_cat 1500
"""
import argparse
import csv
import random
import re
import zlib
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

import underwater_aug as uw
from underwater_aug import (Sequential, UWColorCast, UWDomainTransfer, UWHaze, UWMotionBlur,
                            UWRandomErasing, UWStochasticCLAHE, UWSyntheticOverlay, UWTurbidity,
                            UWVignetteIllumination, compute_lab_stats)
import augment_offline as ao

SINGLE = {
    "colorcast": lambda: UWColorCast(p=1.0),
    "turbidity": lambda: UWTurbidity(p=1.0),
    "clahe": lambda: UWStochasticCLAHE(p=1.0),
    "erasing": lambda: UWRandomErasing(p=1.0),
    "haze": lambda: UWHaze(p=1.0),
    "motionblur": lambda: UWMotionBlur(p=1.0),
    "vignette": lambda: UWVignetteIllumination(p=1.0),
    "overlay": lambda: UWSyntheticOverlay(p=1.0),
}


def read(p):
    im = cv2.imread(str(p))
    return im


def shrink(im, max_side):
    if max_side and max(im.shape[:2]) > max_side:
        f = max_side / max(im.shape[:2])
        im = cv2.resize(im, (int(im.shape[1] * f), int(im.shape[0] * f)), interpolation=cv2.INTER_AREA)
    return im


def seed_for(a, cat, stem):
    s = zlib.crc32(f"{a.seed}:{cat}:{stem}".encode()) & 0x7FFFFFFF
    random.seed(s)
    np.random.seed(s)
    uw.set_seed(s)
    return np.random.default_rng(s)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det", required=True)
    ap.add_argument("--out", default="/content/work/ablation")
    ap.add_argument("--frac", type=float, default=0.15, help="fraction of labelled train originals used as sources")
    ap.add_argument("--max_per_cat", type=int, default=1500)
    ap.add_argument("--max_side", type=int, default=960)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    det, out = Path(a.det), Path(a.out)
    originals = [p for p in sorted((det / "images" / "train").glob("*.jpg")) if not ao.is_copy(p)]
    labelled = [p for p in originals if (det / "labels" / "train" / f"{p.stem}.txt").read_text().strip()]
    n = max(1, min(int(a.frac * len(labelled)), a.max_per_cat))
    subset = random.Random(a.seed).sample(labelled, min(n, len(labelled)))
    print(f"{len(originals)} originals, {len(labelled)} labelled -> {len(subset)} source images per category")
    by_ds = defaultdict(list)
    for p in originals:
        by_ds[p.stem.split("__")[0]].append(p)
    print("sources per dataset:", {k: sum(1 for p in subset if p.stem.startswith(k + "__")) for k in by_ds})

    stats = {k: compute_lab_stats(v) for k, v in by_ds.items()} if len(by_ds) >= 2 else {}
    donors = ao.build_donors(det, originals, seed=a.seed) if len(by_ds) >= 2 else {}
    can_comp = len([k for k, v in donors.items() if v]) >= 2
    cats = dict(SINGLE)
    if stats:
        cats["domain"] = None
    else:
        print("[skip] domain: needs >= 2 datasets")
    if can_comp:
        cats["composite"] = None
    else:
        print("[skip] composite: fewer than 2 datasets with mask polygons (build with --pseudo_masks)")

    orig_paths = [str(p.resolve()) for p in originals]
    arms, y = {}, (det / "data.yaml").read_text()
    made = {}
    for cat in cats:
        idir, ldir = out / cat / "images" / "train", out / cat / "labels" / "train"
        idir.mkdir(parents=True, exist_ok=True)
        ldir.mkdir(parents=True, exist_ok=True)
        files = []
        for p in subset:
            ds = p.stem.split("__")[0]
            im = read(p)
            if im is None:
                continue
            im = shrink(im, a.max_side)
            rng = seed_for(a, cat, p.stem)
            lab = (det / "labels" / "train" / f"{p.stem}.txt").read_text()
            new_lab = lab
            if cat in SINGLE:
                t = SINGLE[cat]()
                res = t(force_apply=True, image=im.copy())["image"]
            elif cat == "domain":
                ref = [s for k, v in stats.items() if k != ds for s in v]
                res = UWDomainTransfer(ref_stats=ref, alpha=(0.7, 1.0), p=1.0)(force_apply=True, image=im.copy())["image"]
            else:  # composite
                other = [d for d in donors if d != ds and donors[d]]
                if not other:
                    continue
                dp, poly = random.choice(donors[random.choice(other)])
                dimg = read(dp)
                r = ao.paste_fish(im, dimg, poly, rng) if dimg is not None else None
                if r is None:
                    continue
                res, newp = r
                new_lab = lab.rstrip("\n") + "\n" + "0 " + " ".join(f"{v:.6f}" for v in newp.reshape(-1)) + "\n"
            name = f"{p.stem}__abl"
            cv2.imwrite(str(idir / f"{name}.jpg"), res, [cv2.IMWRITE_JPEG_QUALITY, 95])
            (ldir / f"{name}.txt").write_text(new_lab)
            files.append(str((idir / f"{name}.jpg").resolve()))
        made[cat] = files
        arms[f"abl_{cat}"] = orig_paths + files
        print(f"[{cat:10s}] {len(files)} copies")

    arms["abl_baseline"] = orig_paths
    arms["abl_control_dup"] = orig_paths + [str(p.resolve()) for p in subset]
    arms["abl_all"] = orig_paths + [f for v in made.values() for f in v]

    (out / "lists").mkdir(parents=True, exist_ok=True)
    rows = []
    for arm, lst in arms.items():
        lp = out / "lists" / f"{arm}.txt"
        lp.write_text("\n".join(lst) + "\n")
        yp = out / f"data_{arm}.yaml"
        yp.write_text(re.sub(r"(?m)^train:.*$", f"train: {lp.resolve()}", y))
        rows.append([arm, len(lst), str(yp)])
    with open(out / "ablation_index.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["arm", "train_images", "yaml"])
        w.writerows(rows)
    print(f"\n{len(arms)} arms -> {out}/ablation_index.csv")
    for r in rows:
        print(f"  {r[0]:18s} {r[1]:7d} imgs")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ablation_run.py
#!/usr/bin/env python
"""
ablation_run.py -- GPU hand-off: train each ablation arm briefly and evaluate on clean vs turbid
===============================================================================================
NOT run during data prep (needs `pip install ultralytics` + GPU). Report Sec. 6.1:
"Value is verified by per-category ablation and a held-out clean-vs-turbid split, tracking
 bounding-box IoU and mask-boundary error as length proxies."

For every arm in ablation_index.csv:
    1. train YOLO11s-seg (same short schedule for all arms)
    2. validate on data_test_clean.yaml and data_test_turbid.yaml -> box/mask mAP50, mAP50-95
    3. mean matched box IoU and mean mask-boundary error (px distance / image diagonal, matched fish only)
Results -> ablation_results.csv (one row per arm x {clean, turbid}).

    python ablation_run.py --det /content/work/det --abl /content/work/ablation --epochs 20
    python ablation_run.py ... --arms abl_baseline abl_colorcast abl_all      # subset
"""
import argparse
import csv
from pathlib import Path

import cv2
import numpy as np


def read_gt(lab, W, H):
    polys = []
    if lab.exists():
        for line in lab.read_text().splitlines():
            v = line.split()
            if len(v) >= 7:
                polys.append((np.array(list(map(float, v[1:])), np.float32).reshape(-1, 2) * [W, H]))
    return polys


def box_of(p):
    return np.array([p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max()], np.float32)


def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0


def boundary_err(p_poly, g_poly, W, H):
    """symmetric mean contour distance between two polygons, / image diagonal."""
    def edge(poly):
        m = np.zeros((H, W), np.uint8)
        cv2.fillPoly(m, [np.round(poly).astype(np.int32)], 1)
        return cv2.morphologyEx(m, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0
    e1, e2 = edge(p_poly), edge(g_poly)
    if not e1.any() or not e2.any():
        return None
    d2 = cv2.distanceTransform((~e2).astype(np.uint8), cv2.DIST_L2, 3)
    d1 = cv2.distanceTransform((~e1).astype(np.uint8), cv2.DIST_L2, 3)
    return float((d2[e1].mean() + d1[e2].mean()) / 2 / np.hypot(W, H))


def length_proxies(model, list_file, det, imgsz=640, conf=0.25, max_imgs=800):
    paths = [l.strip() for l in Path(list_file).read_text().splitlines() if l.strip()][:max_imgs]
    ious, berrs = [], []
    for ps in paths:
        p = Path(ps)
        im = cv2.imread(ps)
        if im is None:
            continue
        H, W = im.shape[:2]
        gts = read_gt(Path(str(p).replace("/images/", "/labels/")).with_suffix(".txt"), W, H)
        if not gts:
            continue
        r = model.predict(im, imgsz=imgsz, conf=conf, verbose=False)[0]
        preds = list(r.masks.xy) if r.masks is not None else []
        for g in gts:
            gb = box_of(g)
            best, bj = 0.0, None
            for j, pp in enumerate(preds):
                if len(pp) < 3:
                    continue
                s = iou(gb, box_of(np.asarray(pp)))
                if s > best:
                    best, bj = s, j
            ious.append(best)                              # missed fish count as IoU 0
            if bj is not None and best >= 0.3:
                e = boundary_err(np.asarray(preds[bj]), g, W, H)
                if e is not None:
                    berrs.append(e)
    return (float(np.mean(ious)) if ious else float("nan"),
            float(np.mean(berrs)) if berrs else float("nan"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det", required=True)
    ap.add_argument("--abl", required=True)
    ap.add_argument("--weights", default="yolo11s-seg.pt")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--arms", nargs="*")
    ap.add_argument("--project", default="/content/drive/MyDrive/aquavision/runs_ablation")
    a = ap.parse_args()
    from ultralytics import YOLO
    det, abl = Path(a.det), Path(a.abl)
    idx = list(csv.DictReader(open(abl / "ablation_index.csv")))
    arms = [r for r in idx if not a.arms or r["arm"] in a.arms]
    rows = []
    for r in arms:
        m = YOLO(a.weights)
        m.train(data=r["yaml"], epochs=a.epochs, imgsz=a.imgsz, project=a.project, name=r["arm"],
                mosaic=1.0, close_mosaic=3, fliplr=0.5, exist_ok=True, verbose=False)
        best = YOLO(str(m.trainer.best))
        for cond in ("clean", "turbid"):
            met = best.val(data=str(det / f"data_test_{cond}.yaml"), split="val", imgsz=a.imgsz, verbose=False)
            miou, berr = length_proxies(best, det / "lists" / f"test_{cond}.txt", det, a.imgsz)
            rows.append([r["arm"], cond, r["train_images"], round(met.box.map50, 4), round(met.box.map, 4),
                         round(met.seg.map50, 4), round(met.seg.map, 4), round(miou, 4), round(berr, 5)])
            print(rows[-1])
    out = abl / "ablation_results.csv"
    with open(out, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["arm", "test_subset", "train_images", "box_mAP50", "box_mAP50-95", "mask_mAP50",
                    "mask_mAP50-95", "mean_box_IoU", "mean_mask_boundary_err"])
        w.writerows(rows)
    print("saved", out)


if __name__ == "__main__":
    main()


## 4. Detector data (Stage A): Brackish + DeepFish
Single-class `fish`, YOLO-seg polygons, 70/20/10 per dataset, split by video/sequence.

**Order matters:** 4a prepares Brackish (frames must exist as images) -> 4b builds the detector set. 4b wipes `/content/work/det`, so always run it *before* the augmentation cell (section 7).

### 4a. Brackish: flatten folders, copy YOLO labels, extract video frames

In [ ]:
import os, shutil

# Let's see what is inside /content/data/brackish
brackish_path = '/content/data/brackish'
if os.path.exists(brackish_path):
    items = os.listdir(brackish_path)
    print("Items in brackish dir:", items)

    # If there is a nested 'brackish' or other folder containing the actual data, let's pull it up
    for item in items:
        item_path = os.path.join(brackish_path, item)
        if os.path.isdir(item_path) and ('brackish' in item.lower() or 'dataset' in item.lower() or 'archive' in item.lower() or item.startswith('z')):
            print(f"Found nested folder: {item}, merging its contents up to {brackish_path}")
            for sub_item in os.listdir(item_path):
                shutil.move(os.path.join(item_path, sub_item), os.path.join(brackish_path, sub_item))
            os.rmdir(item_path)
            print("Restructured directory successfully!")
            print("New items in brackish dir:", os.listdir(brackish_path))
            break


In [ ]:
import os, shutil
from pathlib import Path

# Source directories
yolo_anno_dir = Path('/content/data/brackish/annotations/annotations_YOLO')
brackish_dir = Path('/content/data/brackish')

# Let's count and copy files up to /content/data/brackish
if yolo_anno_dir.exists():
    files = list(yolo_anno_dir.glob('*.txt'))
    print(f"Found {len(files)} .txt label files in annotations_YOLO!")
    for f in files:
        shutil.copy2(f, brackish_dir / f.name)
    print("Successfully copied label files to the main brackish directory.")
else:
    print("Could not find annotations_YOLO directory.")



In [ ]:
import os, json, cv2
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter

# ---------------------------------------------------------------------------------------------
# BRACKISH FRAME EXTRACTION  (v5 - two fixes)
# FIX 1  label k -> frame k-1.  Label files are numbered 1..N (1-based), OpenCV frames are 0..N-1.
# FIX 2  COORDINATE SCALE.  The Brackish pixel labels are NOT in the pixel space of the full video
#        (boxes landed ~2x too far up/left).  We measure the label coordinate range, compare it with the
#        video size, and save every frame at the LABEL resolution, so pixel labels fit the image exactly
#        and prepare_data.py (which divides by the real image size) stays correct.
# ---------------------------------------------------------------------------------------------
LABEL_BASE = 1                  # label k belongs to frame index k-1 (set 0 only if the verify cell says so)
LABEL_SCALE = 2                 # CONFIRMED by the scale test: Brackish labels are in 960x540 pixels, videos are 1920x1080 -> 2
                                # ('auto' = video width / 960)
FORCE_REEXTRACT = True          # delete frames from older runs first (they were mis-scaled / one frame late)

B = Path('/content/data/brackish')
video_dir = B / 'videos'
SKIP = {'train.txt', 'valid.txt', 'test.txt', 'classes.txt'}

if FORCE_REEXTRACT:
    old = list(B.glob('*.jpg'))
    for f in old:
        f.unlink()
    print(f'deleted {len(old)} frames from an earlier extraction')

labels = [l for l in B.glob('*.txt') if l.name.lower() not in SKIP]
video_frames = defaultdict(dict)            # video -> {0-based frame index: label stem}
for l in labels:
    video, _, num = l.stem.rpartition('-')
    if video and num.isdigit() and int(num) - LABEL_BASE >= 0:
        video_frames[video][int(num) - LABEL_BASE] = l.stem

video_path_map = {}
for root, dirs, files in os.walk(video_dir):
    for f in files:
        if f.lower().endswith(('.mp4', '.avi')):
            video_path_map[Path(f).stem] = Path(root) / f
print(f'{len(labels)} label files, {len(video_frames)} videos with labels, {len(video_path_map)} video files')

# ---- 1. video resolution
def vsize(path):
    cap = cv2.VideoCapture(str(path))
    wh = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
    return wh
sizes = {v: vsize(video_path_map[v]) for v in video_frames if v in video_path_map}
(VW, VH), _ = Counter(sizes.values()).most_common(1)[0]
print('video resolution(s):', dict(Counter(sizes.values())))

# ---- 2. how far do the PIXEL label coordinates reach?
mx = my = 0.0
for l in labels:
    for line in l.read_text().splitlines():
        p = line.split()
        if len(p) >= 5:
            cx, cy, w, h = map(float, p[1:5])
            if max(cx, cy, w, h) > 1.5:                      # pixel units (normalised labels are <= 1)
                mx, my = max(mx, cx + w / 2), max(my, cy + h / 2)
print(f'label boxes reach x<={mx:.0f}, y<={my:.0f}   (video is {VW}x{VH})')

if LABEL_SCALE == 'auto':
    scale = VW / 960.0                                       # Brackish annotation images are 960x540
else:
    scale = float(LABEL_SCALE)
LW, LH = round(VW / scale), round(VH / scale)
print(f'==> label scale = {scale:g}: frames will be saved at {LW}x{LH} (label coordinate space)')
if mx and (mx > 1.03 * LW or my > 1.03 * LH):
    print(f'[WARNING] some labels reach beyond the saved frame ({mx:.0f}x{my:.0f} > {LW}x{LH}) - scale too small?')
if mx and (mx < 0.6 * LW or my < 0.6 * LH):
    print('[WARNING] labels only cover a small part of that space - scale too large? check the verify cell below.')

# ---- 3. extract (sequential read: exact frame indices, no seek drift)
extracted = 0
for video, targets in video_frames.items():
    if video not in video_path_map:
        continue
    vw, vh = sizes[video]
    out_size = (round(vw / scale), round(vh / scale))
    cap = cv2.VideoCapture(str(video_path_map[video]))
    last, i = max(targets), 0
    while i <= last:
        if not cap.grab():                                    # end of video
            break
        if i in targets:
            ok, frame = cap.retrieve()
            if ok:
                if scale != 1:
                    frame = cv2.resize(frame, out_size, interpolation=cv2.INTER_AREA)
                cv2.imwrite(str(B / f'{targets[i]}.jpg'), frame)
                extracted += 1
        i += 1
    cap.release()

n_labels = sum(len(t) for t in video_frames.values())
report = {'label_base': LABEL_BASE, 'label_scale': scale, 'video_size': [VW, VH], 'saved_size': [LW, LH],
          'label_extent': [round(mx), round(my)], 'videos': len(video_frames), 'labels': n_labels,
          'extracted': extracted, 'labels_without_frame': n_labels - extracted}
Path('/content/data/brackish_extract_report.json').write_text(json.dumps(report, indent=1))
print('Extraction report:', report)
assert extracted > 0, 'No Brackish frames extracted - check /content/data/brackish/videos and the label names'
if report['labels_without_frame']:
    print(f"[WARNING] {report['labels_without_frame']} labels have no frame - check LABEL_BASE / the video files.")
else:
    print('OK: every Brackish label has its frame.')


### 4a-verify. Do the boxes sit ON the fish? (zoomed crops)
Cell 1 (scale test): the **green** box (scale 2) must sit on the fish. Cell 2 (**new**, "saved frames"): the boxes drawn on the **actual frames this notebook saved** (what `prepare_data.py` will use) must sit on the fish.
- Boxes on fish -> go on to 4b.
- Still shifted -> do **not** run 4b; try `LABEL_BASE = 0` in 4a and re-run.

In [ ]:
import cv2, json, random, numpy as np
from pathlib import Path
from google.colab.patches import cv2_imshow

B = Path('/content/data/brackish'); Y = B/'annotations'/'annotations_YOLO'
S = json.load(open('/content/data/brackish_extract_report.json'))['label_scale']
vids = {p.stem: p for p in (B/'videos').rglob('*') if p.suffix.lower() in ('.mp4', '.avi')}

cands = []                                                   # large fish boxes (class 1 = fish, 2 = small_fish)
for t in Y.glob('*.txt'):
    vs, _, k = t.stem.rpartition('-')
    if vs not in vids or not k.isdigit():
        continue
    for line in t.read_text().splitlines():
        p = line.split()
        if len(p) >= 5 and int(float(p[0])) in (1, 2):
            v = list(map(float, p[1:5]))
            if (v[2] >= 50 and v[3] >= 25) if max(v) > 1.5 else (v[2] >= 0.05 and v[3] >= 0.03):
                cands.append((vs, int(k), v)); break
random.seed(11)
for vs, k, v in random.sample(cands, min(4, len(cands))):
    cap = cv2.VideoCapture(str(vids[vs])); row = []
    for name, idx in (('k-2', k-2), ('k-1 (NEW)', k-1), ('k (old)', k)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, idx)); ok, im = cap.read()
        if not ok: im = np.zeros((540, 960, 3), np.uint8)
        if S != 1: im = cv2.resize(im, (round(im.shape[1]/S), round(im.shape[0]/S)), interpolation=cv2.INTER_AREA)
        H, W = im.shape[:2]
        cx, cy, bw, bh = v if max(v) > 1.5 else (v[0]*W, v[1]*H, v[2]*W, v[3]*H)
        cv2.rectangle(im, (int(cx-bw/2), int(cy-bh/2)), (int(cx+bw/2), int(cy+bh/2)), (0, 255, 0), 2)
        cw = int(min(W, max(bw*3, 200))); ch = int(min(H, cw*0.75))
        x0 = int(np.clip(cx - cw/2, 0, W - cw)); y0 = int(np.clip(cy - ch/2, 0, H - ch))
        crop = cv2.resize(im[y0:y0+ch, x0:x0+cw], (420, 315))
        cv2.putText(crop, name, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        row.append(crop)
    cap.release()
    print(f'{vs}-{k:04d}'); cv2_imshow(np.hstack(row))


In [ ]:
import cv2, random
from pathlib import Path
from google.colab.patches import cv2_imshow

B = Path('/content/data/brackish'); Y = B/'annotations'/'annotations_YOLO'
vids = {p.stem: p for p in (B/'videos').rglob('*') if p.suffix.lower() in ('.mp4', '.avi')}
COL = {1: (0,0,255), 1.5: (0,165,255), 2: (0,255,0), 3: (255,255,0), 4: (255,0,255)}   # red, orange, green, cyan, magenta
files = [t for t in Y.glob('*.txt') if t.stem.rpartition('-')[0] in vids and t.stem.rpartition('-')[2].isdigit()]
random.seed(5); random.shuffle(files); shown = 0
for t in files:
    rows = []
    for l in t.read_text().splitlines():
        p = l.split()
        if len(p) >= 5 and int(float(p[0])) in (1, 2):
            v = list(map(float, p[1:5]))
            if v[2] >= 40: rows.append(v)
    if not rows: continue
    vs, _, k = t.stem.rpartition('-')
    cap = cv2.VideoCapture(str(vids[vs])); cap.set(cv2.CAP_PROP_POS_FRAMES, int(k) - 1); ok, im = cap.read(); cap.release()
    if not ok: continue
    for s, c in COL.items():                                  # label coordinates x scale = position in the full video
        for cx, cy, w, h in rows:
            cv2.rectangle(im, (int((cx-w/2)*s), int((cy-h/2)*s)), (int((cx+w/2)*s), int((cy+h/2)*s)), c, 3)
    cv2.putText(im, 'scale: 1=red 1.5=orange 2=green 3=cyan 4=magenta', (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
    print(t.stem, f'video {im.shape[1]}x{im.shape[0]}')
    cv2_imshow(cv2.resize(im, (960, int(960 * im.shape[0] / im.shape[1]))))
    shown += 1
    if shown == 4: break

In [ ]:
# VERIFY THE REAL INPUT of prepare_data.py: saved frames (960x540) + labels, normalised exactly like the pipeline does
import cv2, random, json
from pathlib import Path
from google.colab.patches import cv2_imshow

B = Path('/content/data/brackish')
rep = json.load(open('/content/data/brackish_extract_report.json'))
print('extraction report:', rep)
assert rep['saved_size'] == [960, 540], f"frames are {rep['saved_size']}, expected [960, 540] - set LABEL_SCALE = 2 in 4a and re-run it"

files = [t for t in B.glob('*.txt') if t.name.lower() not in ('train.txt', 'valid.txt', 'test.txt', 'classes.txt')
         and (B / f'{t.stem}.jpg').exists()]
random.Random(3).shuffle(files)
outside = total = 0
for t in files[:2000]:                                     # numeric check: no box may fall outside its frame
    H, W = 540, 960                                        # saved frame size (asserted above)
    for line in t.read_text().splitlines():
        p = line.split()
        if len(p) >= 5 and int(float(p[0])) in (1, 2):
            cx, cy, w, h = map(float, p[1:5]); total += 1
            if cx - w/2 < -2 or cy - h/2 < -2 or cx + w/2 > W + 2 or cy + h/2 > H + 2: outside += 1
print(f'boxes outside the frame: {outside} / {total}  (should be ~0)')

shown = 0
for t in files:
    rows = []
    for line in t.read_text().splitlines():
        p = line.split()
        if len(p) >= 5 and int(float(p[0])) in (1, 2):
            v = list(map(float, p[1:5]))
            if v[2] >= 60: rows.append(v)                   # clearly visible fish only
    if not rows: continue
    im = cv2.imread(str(B / f'{t.stem}.jpg')); H, W = im.shape[:2]
    for cx, cy, w, h in rows:
        if max(cx, cy, w, h) <= 1.5: cx, cy, w, h = cx*W, cy*H, w*W, h*H      # normalised labels
        cv2.rectangle(im, (int(cx-w/2), int(cy-h/2)), (int(cx+w/2), int(cy+h/2)), (0, 255, 0), 2)
    print(t.stem, f'frame {W}x{H}'); cv2_imshow(im)
    shown += 1
    if shown == 4: break


### 4b. Build the detector dataset (ONE cell, run after 4a)
`PSEUDO_MASKS = True` -> GrabCut masks for box-only data (needed for the length module and for cross-dataset **compositing**). Takes ~20-40 min on CPU. Set `False` only for a quick pipeline test. **Run only after 4a-verify looks right.**

In [ ]:
import os, pandas as pd
from IPython.display import display
%cd /content/aquavision
PSEUDO_MASKS = True      # final data: True (also switches compositing on in section 7)
BRACKISH, DEEPFISH = '/content/data/brackish', '/content/data/DeepFish'
OZFISH = '/content/data/ozfish'          # optional: only used if you unzip OZFish there (PDF Table 1 lists it)
for d in (BRACKISH, DEEPFISH):
    assert os.path.isdir(d) and any(os.scandir(d)), f'{d} is missing/empty - fix section 2 first'
ps = '--pseudo_masks' if PSEUDO_MASKS else ''
oz = f'--ozfish {OZFISH}' if os.path.isdir(OZFISH) else ''
!python prepare_data.py detector --brackish $BRACKISH --deepfish $DEEPFISH $oz $ps --out /content/work/det

m = pd.read_csv('/content/work/det/manifest.csv')
display(m.groupby(['dataset', 'split']).size().unstack(fill_value=0))
assert {'brackish', 'deepfish'} <= set(m.dataset), (
    'DeepFish produced 0 images - check the "DeepFish extensions" printed in section 2 '
    '(loader needs a masks/ folder or YOLO .txt files next to the .jpg files)')

## 5. Check the labels (look at the images!)

In [ ]:
!python prepare_data.py viz --det /content/work/det --n 8
import glob, cv2
from google.colab.patches import cv2_imshow
for f in sorted(glob.glob('viz/*.jpg'))[:8]:
    cv2_imshow(cv2.resize(cv2.imread(f), (800, 450)))

## 6. Augmentation preview (every transform on one real image)
Save `aug_preview.jpg` for the report.

In [ ]:
img = [f for f in sorted(glob.glob('/content/work/det/images/train/*.jpg'))][0]
!python underwater_aug.py "$img" --out /content/aug_preview.jpg
cv2_imshow(cv2.imread('/content/aug_preview.jpg'))

## 7. Cached underwater augmentation (train split only)
`__aug` photometric+erasing+domain colour transfer, `__ovl` overlay copies, `__comp` cross-dataset fish compositing (only if masks exist). Labels are copied (`__comp`: new polygon appended).

In [ ]:
%cd /content/aquavision
import glob
try:
    PSEUDO_MASKS                 # set in section 4 (True = GrabCut masks -> compositing enabled)
except NameError:
    PSEUDO_MASKS = False
    print('[warn] PSEUDO_MASKS not defined (session restarted?) -> compositing OFF')

# Section 7 is self-contained: it (re)writes augment_offline.py (resumable + parallel version)
# and then runs it. If the run is stopped or Colab disconnects, just run this cell AGAIN
# (finished copies are skipped). Do NOT add --clean when resuming.
AUG_CODE = r'''#!/usr/bin/env python
"""
augment_offline.py -- cache underwater-augmented copies of the TRAIN split (report Sec. 6.1)
===========================================================================================
Three kinds of cached copies (val/test are NEVER touched):

  __aug{k}   tier (ii) photometric + tier (iii) random erasing + cross-dataset LAB colour
             transfer ("domain mixing").  Labels are copied unchanged.
  __ovl0     same as above PLUS a synthetic rock/coral overlay (occlusion).  Labels unchanged.
  __comp0    cross-dataset COMPOSITING: a fish cut out (polygon mask) of an image from ANOTHER
             dataset is pasted into this image.  New polygon is appended to the labels.
             Needs real/pseudo masks (polygons with >4 vertices) -> run prepare_data with
             --pseudo_masks, otherwise this step is skipped with a message.

Tier (i) geometric aug + mosaic/mixup stay ONLINE in Ultralytics (see train_aug_config.py).

Report: "overlays ... disabled in the final epochs" -> two image lists are written:
    lists/train_full.txt  (everything)          -> data_full.yaml   (phase 1)
    lists/train_late.txt  (no __ovl / __comp)   -> data_late.yaml   (last ~15 epochs)

RESUMABLE + PARALLEL
    * every copy has a fixed name (<stem>__aug0 ...) and is written atomically (image first,
      label last), so a copy counts as done only when both files exist.  If Colab stops or you
      press Stop, just run the SAME command again: finished copies are skipped.
    * work is spread over --workers CPU processes (0 = all cores).
    * each source image gets its own seed (from --seed + file name), so results do not depend
      on the number of workers or on how many times the run was resumed.
    * --max_side N downsizes the SOURCE before augmenting (labels are normalised, so they stay
      valid) -> much faster and smaller cache.  Training is at 640 px, so 960 loses nothing.

    python augment_offline.py --det /content/work/det --copies 2 --domain_mix \
           --overlay_frac 0.1 --composite_frac 0.1 --max_side 960 --workers 0
"""
import argparse
import multiprocessing as mp
import os
import random
import re
import shutil
import time
import zlib
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

import underwater_aug as uw
from underwater_aug import Sequential, UWSyntheticOverlay, build_underwater_transforms, compute_lab_stats

TAGS = ("__aug", "__ovl", "__comp")
G = {}          # config shared with forked worker processes


def is_copy(p: Path) -> bool:
    return any(t in p.stem for t in TAGS)


def clean(det: Path):
    n = 0
    for p in list((det / "images" / "train").glob("*.jpg")):
        if is_copy(p):
            p.unlink()
            (det / "labels" / "train" / f"{p.stem}.txt").unlink(missing_ok=True)
            n += 1
    shutil.rmtree(det / "lists", ignore_errors=True)
    print(f"cleaned {n} old cached copies")


def changed(pipe, im, tries=4):
    out = pipe(im.copy())
    for _ in range(tries):
        if not np.array_equal(out, im):
            break
        out = pipe(im.copy())
    return out


# ------------------------------------------------------------------ compositing
def read_polys(lab: Path):
    polys = []
    for line in lab.read_text().splitlines():
        v = line.split()
        if len(v) >= 7:
            polys.append(np.array(list(map(float, v[1:])), np.float32).reshape(-1, 2))
    return polys


def build_donors(det: Path, imgs, per_ds=1500, seed=0):
    """Donor fish = polygons with >4 vertices (real or GrabCut masks, not plain rectangles)."""
    donors = defaultdict(list)
    for p in imgs:
        ds = p.stem.split("__")[0]
        lab = det / "labels" / "train" / f"{p.stem}.txt"
        for poly in read_polys(lab):
            if len(poly) <= 4:
                continue
            (x0, y0), (x1, y1) = poly.min(0), poly.max(0)
            if 0.002 <= (x1 - x0) * (y1 - y0) <= 0.4:
                donors[ds].append((p, poly))
    rs = random.Random(seed)
    for ds in donors:
        rs.shuffle(donors[ds])
        donors[ds] = donors[ds][:per_ds]
    return donors


def paste_fish(target, donor_img, poly, rng):
    """Cut `poly` out of donor_img, scale/flip/colour-match, blend into target.
    Returns (image, new normalised polygon) or None."""
    H, W = target.shape[:2]
    h, w = donor_img.shape[:2]
    pts = poly * np.array([w, h], np.float32)
    x0, y0 = np.floor(pts.min(0)).astype(int) - 2
    x1, y1 = np.ceil(pts.max(0)).astype(int) + 2
    x0, y0, x1, y1 = max(0, x0), max(0, y0), min(w, x1), min(h, y1)
    cw, ch = x1 - x0, y1 - y0
    if cw < 8 or ch < 8:
        return None
    crop = donor_img[y0:y1, x0:x1].copy()
    pc = pts - np.array([x0, y0], np.float32)
    mask = np.zeros((ch, cw), np.uint8)
    cv2.fillPoly(mask, [pc.astype(np.int32)], 255)
    if rng.random() < 0.5:                                      # horizontal flip
        crop, mask = crop[:, ::-1].copy(), mask[:, ::-1].copy()
        pc[:, 0] = cw - pc[:, 0]
    s = rng.uniform(0.6, 1.4) * W / w                           # relative to image width
    s = min(s, 0.6 * W / cw, 0.6 * H / ch)
    nw, nh = max(8, int(cw * s)), max(8, int(ch * s))
    if nw >= W or nh >= H:
        return None
    crop = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_LINEAR)
    # colour-match the cut-out to the target scene (Reinhard in LAB, partial)
    lt = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype(np.float32)
    lc = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
    inside = mask > 127
    if inside.sum() > 10:
        mc, sc = lc[inside].mean(0), lc[inside].std(0) + 1e-6
        mt, st = lt.reshape(-1, 3).mean(0), lt.reshape(-1, 3).std(0) + 1e-6
        a = rng.uniform(0.4, 0.8)
        lc = a * ((lc - mc) / sc * st + mt) + (1 - a) * lc
        crop = cv2.cvtColor(np.clip(lc, 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR)
    ox, oy = int(rng.integers(0, W - nw + 1)), int(rng.integers(0, H - nh + 1))
    alpha = cv2.GaussianBlur(mask, (5, 5), 0).astype(np.float32)[..., None] / 255.0
    out = target.copy()
    reg = out[oy:oy + nh, ox:ox + nw].astype(np.float32)
    out[oy:oy + nh, ox:ox + nw] = np.clip(crop * alpha + reg * (1 - alpha), 0, 255).astype(np.uint8)
    newp = pc * np.array([nw / cw, nh / ch], np.float32) + np.array([ox, oy], np.float32)
    return out, np.clip(newp / np.array([W, H], np.float32), 0, 1)


# ------------------------------------------------------------------ one source image
def _done(img_dir: Path, lab_dir: Path, name: str) -> bool:
    return (lab_dir / f"{name}.txt").exists() and (img_dir / f"{name}.jpg").exists()


def _save(img_dir: Path, lab_dir: Path, name: str, img, text: str):
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    if not ok:
        return False
    tmp = img_dir / f"{name}.part"                     # not *.jpg -> never picked up half-written
    tmp.write_bytes(buf.tobytes())
    os.replace(tmp, img_dir / f"{name}.jpg")
    (lab_dir / f"{name}.txt").write_text(text)         # label LAST = "this copy is complete"
    return True


def process(task):
    i, ps = task
    a, det = G["a"], G["det"]
    cv2.setNumThreads(1)
    p = Path(ps)
    ds = p.stem.split("__")[0]
    img_dir, lab_dir = det / "images" / "train", det / "labels" / "train"

    want = [("aug", k) for k in range(a.copies)]
    if i in G["ovl_set"]:
        want.append(("ovl", 0))
    if i in G["comp_set"]:
        want.append(("comp", 0))
    todo = [(t, k) for t, k in want if not _done(img_dir, lab_dir, f"{p.stem}__{t}{k}")]
    res = {"aug": 0, "ovl": 0, "comp": 0}
    if not todo:
        return res

    seed = zlib.crc32(f"{a.seed}:{p.stem}".encode()) & 0x7FFFFFFF      # per-image, worker-independent
    random.seed(seed)
    np.random.seed(seed)
    uw.set_seed(seed)
    rng = np.random.default_rng(seed)

    im = cv2.imread(str(p))
    if im is None:
        return res
    if a.max_side and max(im.shape[:2]) > a.max_side:
        f = a.max_side / max(im.shape[:2])
        im = cv2.resize(im, (int(im.shape[1] * f), int(im.shape[0] * f)), interpolation=cv2.INTER_AREA)
    lab_txt = (lab_dir / f"{p.stem}.txt").read_text()

    stats = G["stats_by_ds"]
    ref = ([s for k, v in stats.items() if k != ds for s in v] or None) if stats else None
    pipe = Sequential(build_underwater_transforms(ref_stats=ref, strength=a.strength, overlay=False))

    for tag, k in todo:
        name = f"{p.stem}__{tag}{k}"
        if tag == "aug":
            if _save(img_dir, lab_dir, name, changed(pipe, im), lab_txt):
                res["aug"] += 1
        elif tag == "ovl":
            opipe = Sequential([UWSyntheticOverlay(p=1.0)] + build_underwater_transforms(
                ref_stats=ref, strength=a.strength, overlay=False))
            if _save(img_dir, lab_dir, name, changed(opipe, im), lab_txt):
                res["ovl"] += 1
        else:                                                   # comp
            donors = G["donors"]
            other = [d for d in donors if d != ds and donors[d]]
            if not other:
                continue
            dp, poly = random.choice(donors[random.choice(other)])
            dimg = cv2.imread(str(dp))
            r = paste_fish(im, dimg, poly, rng) if dimg is not None else None
            if r is None:
                continue
            out, newp = r
            line = "0 " + " ".join(f"{v:.6f}" for v in newp.reshape(-1))
            text = lab_txt.rstrip("\n") + ("\n" if lab_txt.strip() else "") + line + "\n"
            if _save(img_dir, lab_dir, name, out, text):
                res["comp"] += 1
    return res


# ------------------------------------------------------------------ main
def write_lists(det: Path, n_new: dict, n_src: int):
    allp = sorted((det / "images" / "train").glob("*.jpg"))
    late = [p for p in allp if "__ovl" not in p.stem and "__comp" not in p.stem]
    (det / "lists").mkdir(exist_ok=True)
    (det / "lists" / "train_full.txt").write_text("\n".join(str(p.resolve()) for p in allp) + "\n")
    (det / "lists" / "train_late.txt").write_text("\n".join(str(p.resolve()) for p in late) + "\n")
    y = (det / "data.yaml").read_text()
    for nm in ("full", "late"):
        (det / f"data_{nm}.yaml").write_text(
            re.sub(r"(?m)^train:.*$", f"train: {(det / 'lists' / f'train_{nm}.txt').resolve()}", y))
    tot = {t: sum(1 for p in allp if f"__{t}" in p.stem) for t in ("aug", "ovl", "comp")}
    print(f"this run created {n_new['aug']} __aug + {n_new['ovl']} __ovl + {n_new['comp']} __comp copies")
    print(f"in train now: {tot['aug']} __aug + {tot['ovl']} __ovl + {tot['comp']} __comp "
          f"(originals: {n_src}) -> train total {len(allp)}, late-phase list {len(late)}")
    print("yaml: data_full.yaml (phase 1) / data_late.yaml (final epochs, no overlays/composites)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det", required=True)
    ap.add_argument("--copies", type=int, default=2, help="__aug copies per train image")
    ap.add_argument("--strength", type=float, default=1.5, help="scales all probabilities")
    ap.add_argument("--domain_mix", action="store_true", help="cross-dataset LAB colour transfer")
    ap.add_argument("--overlay_frac", type=float, default=0.1, help="fraction of train images that get an __ovl copy")
    ap.add_argument("--composite_frac", type=float, default=0.0, help="fraction that get a cross-dataset __comp copy")
    ap.add_argument("--only_labelled", action="store_true", help="skip background frames as sources")
    ap.add_argument("--clean", action="store_true", help="delete existing cached copies first (NOT needed to resume)")
    ap.add_argument("--max_side", type=int, default=0, help="downsize sources to this long side before augmenting (0 = keep)")
    ap.add_argument("--workers", type=int, default=0, help="worker processes (0 = all CPU cores)")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    cv2.setNumThreads(1)          # OpenCV's own thread pool + fork() can hang; we parallelise per image instead

    det = Path(a.det)
    if a.clean:
        clean(det)
    for t in (det / "images" / "train").glob("*.part"):          # leftovers of an interrupted run
        t.unlink()
    imgs = [p for p in sorted((det / "images" / "train").glob("*.jpg")) if not is_copy(p)]
    if a.only_labelled:
        imgs = [p for p in imgs if (det / "labels" / "train" / f"{p.stem}.txt").read_text().strip()]

    # ---- domain statistics (needs >= 2 datasets to be meaningful)
    by_ds = defaultdict(list)
    for p in imgs:
        by_ds[p.stem.split("__")[0]].append(p)
    stats_by_ds = {}
    if a.domain_mix:
        if len(by_ds) < 2:
            print("[warn] --domain_mix needs >= 2 datasets in train; colour transfer skipped")
        else:
            stats_by_ds = {k: compute_lab_stats(v) for k, v in by_ds.items()}
            print("domain pools:", {k: len(v) for k, v in stats_by_ds.items()})

    rs = random.Random(a.seed)                                   # same picks on every (re)run
    ovl_set = set(rs.sample(range(len(imgs)), int(a.overlay_frac * len(imgs)))) if a.overlay_frac > 0 else set()
    comp_set = set(rs.sample(range(len(imgs)), int(a.composite_frac * len(imgs)))) if a.composite_frac > 0 else set()

    donors = {}
    if comp_set:
        donors = build_donors(det, imgs, seed=a.seed)
        print("composite donor pools (fish with real/pseudo masks):", {k: len(v) for k, v in donors.items()})
        if len([k for k, v in donors.items() if v]) < 2:
            print("[warn] fewer than 2 datasets have mask polygons -> compositing skipped "
                  "(re-run prepare_data.py with --pseudo_masks)")
            comp_set = set()

    G.update(a=a, det=det, stats_by_ds=stats_by_ds, donors=donors, ovl_set=ovl_set, comp_set=comp_set)

    workers = a.workers or os.cpu_count() or 1
    tasks = list(enumerate(map(str, imgs)))
    n = {"aug": 0, "ovl": 0, "comp": 0}
    t0 = time.time()
    print(f"{len(tasks)} source images, {workers} worker(s), copies={a.copies}, max_side={a.max_side or 'orig'}")

    def tally(done, r):
        for k in n:
            n[k] += r[k]
        if done % 250 == 0 or done == len(tasks):
            print(f"  [{done}/{len(tasks)}] {time.time() - t0:.0f}s  new copies so far: "
                  f"{n['aug']} aug / {n['ovl']} ovl / {n['comp']} comp", flush=True)

    try:
        if workers == 1:
            for done, t in enumerate(tasks, 1):
                tally(done, process(t))
        else:
            with mp.get_context("fork").Pool(workers) as pool:
                for done, r in enumerate(pool.imap_unordered(process, tasks, chunksize=8), 1):
                    tally(done, r)
    except KeyboardInterrupt:
        print("\nInterrupted - nothing is lost. Run the same command again to continue "
              "(lists/yaml are written only when the run finishes).")
        raise SystemExit(1)

    write_lists(det, n, len(imgs))


if __name__ == "__main__":
    main()
'''
open('/content/aquavision/augment_offline.py', 'w').write(AUG_CODE)

comp = 0.1 if PSEUDO_MASKS else 0.0
!python augment_offline.py --det /content/work/det --copies 2 --domain_mix \
    --overlay_frac 0.1 --composite_frac $comp --max_side 960 --workers 0
print('train images now:', len(glob.glob('/content/work/det/images/train/*.jpg')))

In [ ]:
import random, cv2, numpy as np
from pathlib import Path
from google.colab.patches import cv2_imshow

DET = Path('/content/work/det')
tr_i, tr_l = DET/'images'/'train', DET/'labels'/'train'
imgs = sorted(tr_i.glob('*.jpg'))

rs = random.Random(1)
pick = []
for t in ('__comp', '__ovl', '__aug', '__aug'):
    c = [p for p in imgs if t in p.stem]
    if c: pick.append(rs.choice(c))
    else: print(f'[none found] {t}')

for p in pick:
    im = cv2.imread(str(p)); H, W = im.shape[:2]
    for l in (tr_l/f'{p.stem}.txt').read_text().splitlines():
        v = np.array(l.split()[1:], np.float32).reshape(-1, 2) * [W, H]
        cv2.polylines(im, [v.astype(np.int32)], True, (0, 255, 0), 2)
    print(p.stem[-45:]); cv2_imshow(cv2.resize(im, (800, int(800*H/W))))

## 7b. Held-out clean-vs-turbid split (report Sec. 6.1: "held-out clean-vs-turbid split")
Turbidity is measured per image (dark-channel haze, contrast, saturation, sharpness) and each dataset is split at the median of its own train scores. Creates `data_test_clean.yaml`, `data_test_turbid.yaml`, `data_trainclean.yaml` (train on clean only -> test on turbid) and `turbidity.csv`. Originals only; run **after** augmentation (copies are ignored).

In [ ]:
%cd /content/aquavision
!python turbid_split.py --det /content/work/det


## 7c. Per-category ablation sets (report Sec. 6.1: "per-category ablation")
One fixed subset of train sources; one copy per image per category (colour cast, turbidity, CLAHE, erasing, haze, motion blur, vignette, overlay, domain transfer, compositing), plus `baseline`, `control_dup` and `all` arms. Each arm = a yaml. Train them with `ablation_run.py` on a GPU (hand-off), then compare clean vs turbid.

In [ ]:
%cd /content/aquavision
!rm -rf /content/work/ablation   # never keep arms from an older (mis-aligned) run
!python ablation_prep.py --det /content/work/det --out /content/work/ablation --frac 0.15 --max_per_cat 1500 --max_side 960
import pandas as pd
display(pd.read_csv('/content/work/ablation/ablation_index.csv'))


## 8. Species data (Stage B): Fish4Knowledge - rare classes get special attention
First a dry run (counts + flags only), then the real build. Read the table: `VERY-LOW`, `heavy-oversample`, `below-target`, `test<5:F1-noisy` mark the rare classes. Only **train** is balanced/augmented; val/test stay real and untouched.

If many classes are `below-target`, raise `--max_aug_ratio` (e.g. 15) or lower `--target` and re-run the dry run.

**Settings for the classifier augmentation brief (baseline run 29 Oct 2026):** do **not** use the defaults - the median class has ~127 train crops, so `--ceil_mult 3` caps *every* class at 381 and throws away ~15,000 real crops of the head classes. Use `--target 300 --max_per_class 2500 --ceil_mult 10 --max_aug_ratio 12 --extra_aug 0.2 --seed 42` (keep `--min_eval 2`, `--seed 42`, so the **test split stays identical to the baseline**). The dry run prints how many real crops the cap throws away.


In [ ]:
# DRY RUN with the brief's settings (writes nothing). Check: "real TRAIN crops thrown away by the cap", below-target classes.
!python f4k_prepare.py --f4k /content/data/fish4knowledge --dry_run \
    --target 300 --max_per_class 2500 --ceil_mult 10 --max_aug_ratio 12 --extra_aug 0.2 --min_eval 2 --seed 42


In [ ]:
# --ref_det = detector frames (Brackish + DeepFish) used for cross-dataset colour transfer -> run AFTER section 4b/7
# --preview_ids = weak mid-size classes (fish_08 / 13 / 14) are added to rare_class_preview.jpg -> LOOK at it
!python f4k_prepare.py --f4k /content/data/fish4knowledge --out /content/work/cls --ref_det /content/work/det \
    --target 300 --max_per_class 2500 --ceil_mult 10 --max_aug_ratio 12 --extra_aug 0.2 --min_eval 2 --seed 42 \
    --preview_ids 8,13,14

# OPTIONAL control job (brief: "does the cap help or hurt?"). NOTE: --max_per_class alone does not uncap, --ceil_mult must go up too.
# !python f4k_prepare.py --f4k /content/data/fish4knowledge --out /content/work/cls_nocap --ref_det /content/work/det \
#     --target 300 --max_per_class 100000 --ceil_mult 1000 --max_aug_ratio 12 --extra_aug 0.2 --min_eval 2 --seed 42

from IPython.display import Image, display
display(Image('/content/work/cls/class_balance.png')); display(Image('/content/work/cls/rare_class_preview.jpg'))


## 8a. Robustness check for the classifier (brief P4): synthetic turbid copy of the F4K **test** split
Writes `/content/work/cls_eval/test_clean` and `test_turbid` (same crops, one untouched, one degraded) + `turbid_preview.jpg`. Evaluate the same weights on both with `eval_cls.py`; the augmentation gain should show up as a **smaller clean -> turbid drop**. Test crops only; nothing here is used for training.

In [ ]:
%%writefile cls_turbid_test.py
#!/usr/bin/env python
"""
cls_turbid_test.py -- robustness check for the classifier (brief P4, report Sec. 6.1 clean-vs-turbid)
=====================================================================================================
All F4K crops come from ONE camera setup, so 0.987 only says "good inside F4K". This makes a
synthetic TURBID copy of the F4K TEST split (blue-green cast + turbidity + haze + vignette, p=1) next
to an untouched CLEAN copy of the same crops. Evaluate the same weights on both:
    eval_cls.py --split_dir /content/work/cls_eval/test_clean  --tag clean
    eval_cls.py --split_dir /content/work/cls_eval/test_turbid --tag turbid
The gain from augmentation should show up as a SMALLER clean->turbid drop. Test crops only - never used for training.
(Brackish/DeepFish carry no species labels, so a real cross-dataset accuracy cannot be measured.)
"""
import argparse, random, shutil
from pathlib import Path
import cv2, numpy as np
import underwater_aug as uw
from underwater_aug import Sequential, UWColorCast, UWHaze, UWTurbidity, UWVignetteIllumination


def pipe():
    return Sequential([
        UWColorCast(red_atten=(0.35, 0.7), hue_pull=(0.15, 0.35), p=1.0),
        UWTurbidity(sat_scale=(0.35, 0.65), speckle_sigma=(0.04, 0.09), p=1.0),
        UWHaze(density=(0.3, 0.6), p=1.0),
        UWVignetteIllumination(p=0.5),
    ])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cls", default="/content/work/cls")
    ap.add_argument("--out", default="/content/work/cls_eval")
    ap.add_argument("--seed", type=int, default=123)
    a = ap.parse_args()
    uw.set_seed(a.seed)
    out = Path(a.out)
    shutil.rmtree(out, ignore_errors=True)
    p, n, pairs = pipe(), 0, []
    for d in sorted((Path(a.cls) / "test").iterdir()):
        (out / "test_clean" / d.name).mkdir(parents=True, exist_ok=True)
        (out / "test_turbid" / d.name).mkdir(parents=True, exist_ok=True)
        for f in sorted(d.iterdir()):
            im = cv2.imread(str(f))
            if im is None:
                continue
            t = p(im.copy())
            cv2.imwrite(str(out / "test_clean" / d.name / (f.stem + ".png")), im)
            cv2.imwrite(str(out / "test_turbid" / d.name / (f.stem + ".png")), t)
            pairs.append((im, t))
            n += 1
    rs = random.Random(0)
    tiles = []
    for im, t in rs.sample(pairs, min(8, len(pairs))):
        r = lambda x: cv2.resize(x, (128, 128))
        tiles.append(np.hstack([r(im), r(t)]))
    if tiles:
        cv2.imwrite(str(out / "turbid_preview.jpg"), np.vstack([np.hstack(tiles[i:i + 2]) for i in range(0, len(tiles) - 1, 2)]))
    print(f"{n} test crops -> {out}/test_clean and {out}/test_turbid  (look at turbid_preview.jpg: left clean, right turbid)")


if __name__ == "__main__":
    main()


In [ ]:
%cd /content/aquavision
!python cls_turbid_test.py --cls /content/work/cls --out /content/work/cls_eval
from IPython.display import Image, display
display(Image('/content/work/cls_eval/turbid_preview.jpg'))   # left = clean, right = turbid: fish must stay recognisable


## 8a-eval. Classifier evaluation script (GPU hand-off, brief section 4)
`eval_cls.py` gives top-1, macro-F1 **with bootstrap CI**, head/mid/tail macro-F1, macro-F1 over the **clip-split classes only**, errors predicted as the majority class, and the per-class table. Run it on `/content/work/cls/test` (clean) and `/content/work/cls_eval/test_turbid`. The same script must be used for the baseline and every augmented variant.

In [ ]:
%%writefile eval_cls.py
#!/usr/bin/env python
"""
eval_cls.py -- fair-comparison evaluation of the species classifier (brief section 4)
======================================================================================
Reports: top-1, MACRO-F1 with stratified-bootstrap 95% CI, macro-F1 for head (fish_01-05) / mid (100-999 real
train crops) / tail (<100) groups, macro-F1 over the CLIP-SPLIT classes only (split_mode == group; the per-image
split classes may share clips between train and test), how many errors are predicted as the majority class
(brief P1), and the per-class table (P/R/F1/support). Group definitions use the REAL pre-balancing train
counts from split_counts.json, so they are identical for every data variant.
GPU hand-off (needs ultralytics):
    !python eval_cls.py --weights runs/cls_aug/weights/best.pt --split_dir /content/work/cls/test --tag clean
    !python eval_cls.py --weights runs/cls_aug/weights/best.pt --split_dir /content/work/cls_eval/test_turbid --tag turbid
"""
import argparse, csv, json, re
from pathlib import Path
import numpy as np

IMG = {".png", ".jpg", ".jpeg", ".bmp"}
cid = lambda name: int(re.search(r"\d+", name).group())


def groups_from_dir(cls_dir):
    cls_dir = Path(cls_dir)
    sc = json.loads((cls_dir / "split_counts.json").read_text())
    mode = {}
    with open(cls_dir / "class_report.csv") as f:
        for r in csv.DictReader(f):
            mode[r["class"]] = r["split_mode"]
    g = {}
    for c, v in sc.items():
        n = v["train"]
        g[c] = "head" if cid(c) <= 5 else ("mid" if n >= 100 else "tail")
    return g, mode, {c: v["train"] for c, v in sc.items()}


def macro(f1, sup, subset):
    idx = [i for i in subset if sup[i] > 0]
    return float(np.mean([f1[i] for i in idx])) if idx else float("nan")


def metrics(y, p, names, grp=None, mode=None, train_n=None, n_boot=1000, seed=0):
    from sklearn.metrics import precision_recall_fscore_support
    y, p = np.asarray(y), np.asarray(p)
    L = list(range(len(names)))
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=L, zero_division=0)
    out = dict(n=int(len(y)), top1=float((y == p).mean()), macro_f1=macro(f1, sup, L))
    present = [i for i in L if sup[i] > 0]
    by_c = {c: np.where(y == c)[0] for c in present}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):                                   # stratified: every class stays present
        idx = np.concatenate([rng.choice(v, len(v), replace=True) for v in by_c.values()])
        _, _, f, s = precision_recall_fscore_support(y[idx], p[idx], labels=L, zero_division=0)
        boots.append(macro(f, s, L))
    out["macro_f1_ci95"] = [float(x) for x in np.percentile(boots, [2.5, 97.5])]
    if grp:
        for gname in ("head", "mid", "tail"):
            out[f"macro_f1_{gname}"] = macro(f1, sup, [i for i in L if grp.get(names[i]) == gname])
    if mode:
        out["macro_f1_clip_split_classes"] = macro(f1, sup, [i for i in L if mode.get(names[i]) == "group"])
        out["macro_f1_image_split_classes"] = macro(f1, sup, [i for i in L if mode.get(names[i]) != "group"])
    if train_n:
        maj = max(L, key=lambda i: train_n.get(names[i], 0))
        err = p != y
        out["errors"] = int(err.sum())
        out["errors_predicted_as_majority"] = int(((p == maj) & err).sum())
        out["majority_class"] = names[maj]
    out["per_class"] = [dict(cls=names[i], precision=float(pr[i]), recall=float(rc[i]), f1=float(f1[i]),
                             support=int(sup[i])) for i in L]
    return out


def predict(weights, split_dir, imgsz=224, batch=64):
    from ultralytics import YOLO
    m = YOLO(weights)
    names = [m.names[i] for i in sorted(m.names)]
    files, y = [], []
    for d in sorted(Path(split_dir).iterdir()):
        if d.is_dir() and d.name in names:
            for f in sorted(d.iterdir()):
                if f.suffix.lower() in IMG:
                    files.append(str(f))
                    y.append(names.index(d.name))
    p = []
    for i in range(0, len(files), batch):
        p += [int(r.probs.top1) for r in m.predict(files[i:i + batch], imgsz=imgsz, verbose=False)]
    return y, p, names


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--split_dir", required=True)
    ap.add_argument("--cls_dir", default="/content/work/cls", help="has split_counts.json + class_report.csv")
    ap.add_argument("--imgsz", type=int, default=224)
    ap.add_argument("--tag", default="test")
    ap.add_argument("--out", default=".")
    a = ap.parse_args()
    y, p, names = predict(a.weights, a.split_dir, a.imgsz)
    grp, mode, train_n = groups_from_dir(a.cls_dir)
    r = metrics(y, p, names, grp, mode, train_n)
    lo, hi = r["macro_f1_ci95"]
    print(f"[{a.tag}] n={r['n']} top-1 {r['top1']:.4f} | MACRO-F1 {r['macro_f1']:.4f} (95% CI {lo:.3f}-{hi:.3f}) "
          f"| head {r['macro_f1_head']:.3f} mid {r['macro_f1_mid']:.3f} tail {r['macro_f1_tail']:.3f} "
          f"| clip-split classes {r['macro_f1_clip_split_classes']:.3f} | image-split {r['macro_f1_image_split_classes']:.3f}")
    print(f"errors {r['errors']}, of which predicted as majority class ({r['majority_class']}): {r['errors_predicted_as_majority']}")
    print("(a change < ~0.05 in macro-F1 is inside the noise of this small test set)")
    print(f"{'class':34s} {'P':>6s} {'R':>6s} {'F1':>6s} {'n':>5s}")
    for c in sorted(r["per_class"], key=lambda c: c["f1"]):
        print(f"{c['cls']:34s} {c['precision']:6.3f} {c['recall']:6.3f} {c['f1']:6.3f} {c['support']:5d}"
              + ("  <- n<5: anecdotal" if 0 < c["support"] < 5 else ""))
    out = Path(a.out)
    (out / f"eval_cls_{a.tag}.json").write_text(json.dumps(r, indent=1))
    with open(out / f"eval_cls_{a.tag}_per_class.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(r["per_class"][0]))
        w.writeheader()
        w.writerows(r["per_class"])


if __name__ == "__main__":
    main()


## 8b. Proof: is every dataset augmented, and where is it stored?
Detector copies live in `/content/work/det/images/train` (`__aug`, `__ovl`, `__comp` suffixes). Fish4Knowledge synthetic crops live in `/content/work/cls/train/<species>/aug*.png`. val/test must show **0 copies**.

In [ ]:
import glob, os, collections, pandas as pd
from IPython.display import display
DET, CLS = '/content/work/det', '/content/work/cls'

rows = []
for split in ('train', 'val', 'test'):
    c = collections.Counter()
    for p in glob.glob(f'{DET}/images/{split}/*.jpg'):
        n = os.path.basename(p)
        kind = next((t for t in ('aug', 'ovl', 'comp') if f'__{t}' in n), 'original')
        c[(n.split('__')[0], kind)] += 1
    rows += [(split, ds, k, v) for (ds, k), v in c.items()]
det_tab = (pd.DataFrame(rows, columns=['split', 'dataset', 'kind', 'n'])
           .pivot_table(index=['dataset', 'split'], columns='kind', values='n', fill_value=0))
display(det_tab)
assert {'brackish', 'deepfish'} <= set(det_tab.index.get_level_values(0)), 'a detector dataset is missing'
for ds in ('brackish', 'deepfish'):
    assert det_tab.loc[(ds, 'train'), 'aug'] > 0, f'{ds}: no augmented train copies'
det_tab.to_csv('/content/aug_counts_detector.csv')
for k in ('aug', 'ovl', 'comp'):
    if k in det_tab.columns:
        bad = det_tab.xs('val', level='split')[k].sum() + det_tab.xs('test', level='split')[k].sum()
        assert bad == 0, f'augmented copies leaked into val/test: {k}'
print('OK: val/test contain only original images')
if 'comp' not in det_tab.columns:
    print('[note] no __comp copies: compositing needs PSEUDO_MASKS=True in 4b (and >= 2 datasets with mask polygons)')

f = []
for d in sorted(glob.glob(f'{CLS}/train/*')):
    names = os.listdir(d)
    syn = sum(n.startswith('aug') for n in names)
    f.append((os.path.basename(d), len(names) - syn, syn,
              len(os.listdir(d.replace('/train/', '/val/'))), len(os.listdir(d.replace('/train/', '/test/')))))
f4k_tab = pd.DataFrame(f, columns=['class', 'train_real', 'train_synth', 'val', 'test'])
display(f4k_tab)
f4k_tab.to_csv('/content/aug_counts_f4k.csv', index=False)
for s in ('val', 'test'):
    assert not glob.glob(f'{CLS}/{s}/*/aug*'), f'synthetic crops found in {s}'
print('OK: Fish4Knowledge val/test are real crops only')


In [ ]:
import glob, os, collections, pandas as pd
from IPython.display import display
DET, CLS = '/content/work/det', '/content/work/cls'

rows = []
for split in ('train', 'val', 'test'):
    c = collections.Counter()
    for p in glob.glob(f'{DET}/images/{split}/*.jpg'):
        n = os.path.basename(p)
        kind = next((t for t in ('aug', 'ovl', 'comp') if f'__{t}' in n), 'original')
        c[(n.split('__')[0], kind)] += 1
    rows += [(split, ds, k, v) for (ds, k), v in c.items()]
det_tab = (pd.DataFrame(rows, columns=['split', 'dataset', 'kind', 'n'])
           .pivot_table(index=['dataset', 'split'], columns='kind', values='n', fill_value=0))
display(det_tab)
assert {'brackish', 'deepfish'} <= set(det_tab.index.get_level_values(0)), 'a detector dataset is missing'
for ds in ('brackish', 'deepfish'):
    assert det_tab.loc[(ds, 'train'), 'aug'] > 0, f'{ds}: no augmented train copies'
det_tab.to_csv('/content/aug_counts_detector.csv')
for k in ('aug', 'ovl', 'comp'):
    if k in det_tab.columns:
        bad = det_tab.xs('val', level='split')[k].sum() + det_tab.xs('test', level='split')[k].sum()
        assert bad == 0, f'augmented copies leaked into val/test: {k}'
print('OK: val/test contain only original images')
if 'comp' not in det_tab.columns:
    print('[note] no __comp copies: compositing needs PSEUDO_MASKS=True in 4b (and >= 2 datasets with mask polygons)')

f = []
for d in sorted(glob.glob(f'{CLS}/train/*')):
    names = os.listdir(d)
    syn = sum(n.startswith('aug') for n in names)
    f.append((os.path.basename(d), len(names) - syn, syn,
              len(os.listdir(d.replace('/train/', '/val/'))), len(os.listdir(d.replace('/train/', '/test/')))))
f4k_tab = pd.DataFrame(f, columns=['class', 'train_real', 'train_synth', 'val', 'test'])
display(f4k_tab)
f4k_tab.to_csv('/content/aug_counts_f4k.csv', index=False)
for s in ('val', 'test'):
    assert not glob.glob(f'{CLS}/{s}/*/aug*'), f'synthetic crops found in {s}'
print('OK: Fish4Knowledge val/test are real crops only')

In [ ]:
import glob, os, cv2, numpy as np, pandas as pd
from IPython.display import display
CLS = '/content/work/cls'

def centre_hue_sat(im):
    h, w = im.shape[:2]
    c = im[h // 4: 3 * h // 4, w // 4: 3 * w // 4]          # beech ka aadha hissa = zyada tar fish
    hsv = cv2.cvtColor(c, cv2.COLOR_BGR2HSV).astype(np.float32)
    wgt = hsv[..., 1] * hsv[..., 2] / 255.0 + 1e-6           # rangeen aur bright pixels ka zyada wazan
    ang = np.deg2rad(hsv[..., 0] * 2)
    hue = np.rad2deg(np.arctan2((np.sin(ang) * wgt).sum(), (np.cos(ang) * wgt).sum())) % 360
    return hue, float((hsv[..., 1] * wgt).sum() / wgt.sum())

rows = []
for d in sorted(glob.glob(f'{CLS}/train/*')):
    files = os.listdir(d)
    src = {os.path.splitext(f)[0]: f for f in files if not f.startswith('aug')}
    dh, ds = [], []
    for f in [x for x in files if x.startswith('aug')][:200]:      # har class ke 200 copies kaafi hain
        stem = f[:-4].split('__', 1)[1]
        if stem not in src: continue
        a, b = cv2.imread(f'{d}/{f}'), cv2.imread(f'{d}/{src[stem]}')
        if a is None or b is None or a.shape != b.shape: continue
        (h1, s1), (h0, s0) = centre_hue_sat(a), centre_hue_sat(b)
        dh.append(abs((h1 - h0 + 180) % 360 - 180)); ds.append(s1 - s0)
    if dh:
        rows.append((os.path.basename(d), len(dh), round(float(np.median(dh)), 1),
                     round(float(np.percentile(dh, 90)), 1), round(float(np.mean(ds)), 1)))
drift = pd.DataFrame(rows, columns=['class', 'n_pairs', 'median_hue_shift_deg', 'p90_hue_shift_deg', 'mean_sat_change'])
display(drift)
drift.to_csv('/content/aug_colour_drift_f4k.csv', index=False)
bad = drift[drift.median_hue_shift_deg > 10]
print('OK: median hue shift <= 10 deg in every class' if bad.empty
      else f'[check] colour drift > 10 deg in: {list(bad["class"])} -> show_class(<id>) se dekho ya cast aur kam karo')

In [ ]:
   for c in (4, 5, 10, 16): show_class(c)

In [ ]:
import glob, os, re, cv2, numpy as np
from google.colab.patches import cv2_imshow

CLS = '/content/work/cls'

def tile(path, sz=128):
    im = cv2.imread(path)
    if im is None: return np.zeros((sz, sz, 3), np.uint8)
    h, w = im.shape[:2]; f = sz / max(h, w)
    im = cv2.resize(im, (max(1, int(w*f)), max(1, int(h*f))), interpolation=cv2.INTER_CUBIC)
    c = np.zeros((sz, sz, 3), np.uint8); c[:im.shape[0], :im.shape[1]] = im
    return c

def class_dir(cid, split='train'):
    for d in sorted(glob.glob(f'{CLS}/{split}/*')):
        if int(re.search(r'\d+', os.path.basename(d)).group()) == cid: return d

def show_class(cid, n_real=5, n_aug=7, seed=0):
    d = class_dir(cid)
    if not d:
        print(f'Class ID {cid} not found.')
        return
    files = sorted(os.listdir(d))
    real = [f for f in files if not f.startswith('aug')]
    aug  = [f for f in files if f.startswith('aug')]
    rs = np.random.RandomState(seed)
    real = list(rs.choice(real, min(n_real, len(real)), replace=False))
    aug  = list(rs.choice(aug,  min(n_aug,  len(aug)),  replace=False)) if aug else []
    n = max(len(real), len(aug), 1)
    pad = lambda row: row + [np.zeros((128,128,3), np.uint8)] * (n - len(row))
    grid = np.vstack([np.hstack(pad([tile(f'{d}/{f}') for f in real])),
                      np.hstack(pad([tile(f'{d}/{f}') for f in aug]))])
    cv2.putText(grid, f'{os.path.basename(d)} | top: real, bottom: augmented', (4, 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)
    out = f'/content/f4k_preview_{cid:02d}.jpg'; cv2.imwrite(out, grid)
    cv2_imshow(grid); print('saved', out)

for c in (4, 5, 10, 16):
    show_class(c)

In [ ]:
import sys, glob, os, re, cv2, numpy as np, pandas as pd
sys.path.insert(0, '/content/aquavision')
import underwater_aug as uw, f4k_prepare as fp
from underwater_aug import Sequential
from IPython.display import display

CLS = '/content/work/cls'
ref = fp.load_ref_stats('/content/work/det')
colour_only = Sequential(fp.build_cls_aug(1.0, ref).transforms[2:])   # same pipeline WITHOUT FlipRotate + ScaleCropJitter

def hue(im):
    h, w = im.shape[:2]; c = im[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
    hsv = cv2.cvtColor(c, cv2.COLOR_BGR2HSV).astype(np.float32); wgt = hsv[..., 1] * hsv[..., 2] / 255.0 + 1e-6
    a = np.deg2rad(hsv[..., 0] * 2); return np.rad2deg(np.arctan2((np.sin(a) * wgt).sum(), (np.cos(a) * wgt).sum())) % 360

uw.set_seed(0); rows = []
for cid in (4, 5, 10, 16, 20):
    d = [x for x in sorted(glob.glob(f'{CLS}/train/*')) if int(re.search(r'\d+', os.path.basename(x)).group()) == cid][0]
    real = [f for f in sorted(os.listdir(d)) if not f.startswith('aug')][:100]
    sh = []
    for f in real:
        im = cv2.imread(f'{d}/{f}')
        if im is None: continue
        sh.append(abs((hue(colour_only(im.copy())) - hue(im) + 180) % 360 - 180))
    rows.append((os.path.basename(d), len(sh), round(float(np.median(sh)), 1), round(float(np.percentile(sh, 90)), 1)))
display(pd.DataFrame(rows, columns=['class', 'n', 'COLOUR-ONLY median shift', 'COLOUR-ONLY p90']))

## 8c. PDF-alignment checklist + storage map
Every row must say PASS. A FAIL names the PDF item that is not yet satisfied.

In [ ]:
import os, glob, json, pandas as pd
from pathlib import Path
DET, CLS, ABL = Path('/content/work/det'), Path('/content/work/cls'), Path('/content/work/ablation')
res = []
def chk(name, ok, detail=''): res.append((name, 'PASS' if ok else 'FAIL', detail))

m = pd.read_csv(DET/'manifest.csv')
_er = Path('/content/data/brackish_extract_report.json')
if _er.exists():
    _r = json.loads(_er.read_text())
    chk('Brackish: every label file has its video frame (label k -> frame k-1)', _r['labels_without_frame'] == 0, str(_r))
    chk('Brackish: frames saved at the label resolution (labels 960x540 -> scale 2)', _r.get('saved_size') == [960, 540] and _r.get('label_scale') == 2, f"scale={_r.get('label_scale')} saved={_r.get('saved_size')}")
else:
    chk('Brackish: extraction report present', False, 'run section 4a')
chk('raw_counts.json saved (true raw counts before empty-frame cap)', (DET/'raw_counts.json').exists())
chk('Sec2/6 ph1: Brackish + DeepFish in ONE single-class detector set', {'brackish','deepfish'} <= set(m.dataset), str(sorted(set(m.dataset))))
for ds, g in m.groupby('dataset'):                       # Sec 6.2: 70/20/10 per dataset
    r = g.split.value_counts(normalize=True).reindex(['train','val','test']).fillna(0).round(2).to_dict()
    chk(f'Sec6.2: 70/20/10 split ({ds})', abs(r['train']-.7) < .08 and abs(r['val']-.2) < .08, str(r))
cnt = {t: len(glob.glob(f'{DET}/images/train/*__{t}*.jpg')) for t in ('aug','ovl','comp')}
chk('Sec6.1(ii): photometric copies cached (__aug)', cnt['aug'] > 0, str(cnt))
chk('Sec6.1(iii): synthetic overlays (__ovl)', cnt['ovl'] > 0)
chk('Sec6.1(iii): cross-dataset compositing (__comp)', cnt['comp'] > 0, 'needs PSEUDO_MASKS=True + both datasets')
leak = [p for s in ('val','test') for p in glob.glob(f'{DET}/images/{s}/*__aug*') + glob.glob(f'{DET}/images/{s}/*__ovl*') + glob.glob(f'{DET}/images/{s}/*__comp*')]
chk('No augmented copy in detector val/test', not leak)
late = (DET/'lists'/'train_late.txt').read_text() if (DET/'lists'/'train_late.txt').exists() else 'x__ovl'
chk('Sec6.1: overlays/composites off in final epochs (data_late.yaml)', '__ovl' not in late and '__comp' not in late)
chk('Sec6.1: held-out clean-vs-turbid split', (DET/'data_test_turbid.yaml').exists() and (DET/'data_test_clean.yaml').exists())
idx = pd.read_csv(ABL/'ablation_index.csv') if (ABL/'ablation_index.csv').exists() else pd.DataFrame(columns=['arm'])
chk('Sec6.1: per-category ablation incl. domain + composite arms', {'abl_domain','abl_composite'} <= set(idx.arm), f'{len(idx)} arms')
ncls = len(os.listdir(CLS/'train')) if (CLS/'train').exists() else 0
chk('Sec2: 23-class Fish4Knowledge classifier set', ncls == 23, f'{ncls} classes')
aug_f4k = {c: sum(f.startswith('aug') for f in os.listdir(CLS/'train'/c)) for c in os.listdir(CLS/'train')} if ncls else {}
chk('Sec6.1: EVERY F4K class has augmented copies (--extra_aug)', ncls > 0 and all(v > 0 for v in aug_f4k.values()), f'min={min(aug_f4k.values()) if aug_f4k else 0}')
chk('No synthetic crops in F4K val/test', not glob.glob(f'{CLS}/val/*/aug*') and not glob.glob(f'{CLS}/test/*/aug*'))
_dc = json.loads((CLS/'data_card.json').read_text()) if (CLS/'data_card.json').exists() else {}
chk('F4K: dominant-class cap does not discard most real crops (ceiling >= 2000; brief: avoid the default 381 cap)', _dc.get('ceiling', 0) >= 2000, f"ceiling={_dc.get('ceiling')} target={_dc.get('target')}")
chk('F4K: cross-dataset colour transfer from detector frames (--ref_det)', bool(_dc.get('ref_det')) and len(glob.glob(f'{DET}/images/train/*.jpg')) > 0, str(_dc.get('ref_det')))
chk('F4K: split identical to the baseline (seed 42, min_eval 2)', _dc.get('seed') == 42 and _dc.get('min_eval') == 2, f"seed={_dc.get('seed')} min_eval={_dc.get('min_eval')}")
_rep = pd.read_csv(CLS/'class_report.csv') if (CLS/'class_report.csv').exists() else pd.DataFrame(columns=['class', 'split_mode'])
_img = [''.join(ch for ch in c if ch.isdigit()) for c, m_ in zip(_rep['class'], _rep['split_mode']) if m_ != 'group']
chk('F4K: per-image-split classes identified (report macro-F1 also on the clip-split classes only)', len(_rep) > 0, f'image/train-only split: {_img}')
chk('F4K: synthetic TURBID copy of the test split (classifier robustness, 6.1 clean-vs-turbid)', os.path.exists('/content/work/cls_eval/test_turbid'), 'run section 8a')
chk('Saved to Drive (section 10)', os.path.exists('/content/drive/MyDrive/aquavision/work.zip'), 'run section 10')
out = pd.DataFrame(res, columns=['PDF requirement', 'status', 'detail']); pd.set_option('display.max_colwidth', 90)
display(out)

print('\n=== WHERE EVERYTHING IS STORED (Colab disk is temporary; section 10 copies it to Drive) ===')
for k, v in {
 'Brackish raw (frames+labels)':'/content/data/brackish', 'DeepFish raw':'/content/data/DeepFish',
 'Fish4Knowledge raw':'/content/data/fish4knowledge', 'Detector set (Brackish+DeepFish) + cached aug':'/content/work/det/{images,labels}/{train,val,test}',
 'Detector aug naming':'<dataset>__<stem>__aug0/aug1 | __ovl0 | __comp0 (train only)', 'Detector yamls / lists':'/content/work/det/data_full.yaml, data_late.yaml, lists/',
 'F4K classifier set + aug':'/content/work/cls/{train,val,test}/<species>/  (aug files: aug#####__<stem>.png)',
 'Ablation arms':'/content/work/ablation', 'Permanent copy on Drive':'/content/drive/MyDrive/aquavision/work.zip + report_assets/'}.items():
    print(f'{k:48s} {v}')

In [ ]:
!grep -n "red_atten=(0.8" /content/aquavision/f4k_prepare.py
!ls -l --time-style=full-iso /content/aquavision/f4k_prepare.py /content/work/cls/data_card.json

In [ ]:
import glob, os, re, cv2, numpy as np
from google.colab.patches import cv2_imshow

CLS = '/content/work/cls'

def tile(path, sz=128):
    im = cv2.imread(path)
    if im is None: return np.zeros((sz, sz, 3), np.uint8)
    h, w = im.shape[:2]; f = sz / max(h, w)
    im = cv2.resize(im, (max(1, int(w*f)), max(1, int(h*f))), interpolation=cv2.INTER_CUBIC)
    c = np.zeros((sz, sz, 3), np.uint8); c[:im.shape[0], :im.shape[1]] = im
    return c

def class_dir(cid, split='train'):
    for d in sorted(glob.glob(f'{CLS}/{split}/*')):
        if int(re.search(r'\d+', os.path.basename(d)).group()) == cid: return d

def show_class(cid, n_real=5, n_aug=7, seed=0):
    d = class_dir(cid); files = sorted(os.listdir(d))
    real = [f for f in files if not f.startswith('aug')]
    aug  = [f for f in files if f.startswith('aug')]
    rs = np.random.RandomState(seed)
    real = list(rs.choice(real, min(n_real, len(real)), replace=False))
    aug  = list(rs.choice(aug,  min(n_aug,  len(aug)),  replace=False)) if aug else []
    n = max(len(real), len(aug), 1)
    pad = lambda row: row + [np.zeros((128,128,3), np.uint8)] * (n - len(row))
    grid = np.vstack([np.hstack(pad([tile(f'{d}/{f}') for f in real])),
                      np.hstack(pad([tile(f'{d}/{f}') for f in aug]))])
    cv2.putText(grid, f'{os.path.basename(d)} | top: real, bottom: augmented', (4, 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)
    out = f'/content/f4k_preview_{cid:02d}.jpg'; cv2.imwrite(out, grid)
    cv2_imshow(grid); print('saved', out)

for cid in (8, 13, 14, 19, 21):      # weak mid classes + very rare classes
    show_class(cid)

In [ ]:
# RESTORE (only for a FRESH Colab session when you just want to reload the saved data).
# Keep False while building: unzipping the old work.zip would OVERWRITE the freshly built data.
RESTORE_FROM_DRIVE = False
import subprocess
from google.colab import drive
drive.mount('/content/drive')
if RESTORE_FROM_DRIVE:
    subprocess.run('mkdir -p /content/work && unzip -q -o /content/drive/MyDrive/aquavision/work.zip -d /content/work', shell=True, check=True)
    print(subprocess.run('ls /content/work /content/work/det | head -20', shell=True, capture_output=True, text=True).stdout)
else:
    print('RESTORE_FROM_DRIVE = False -> nothing restored (set True only in a fresh session).')


In [ ]:
# ===== NUMBERS FOR THE REPORT / VIVA: before vs after augmentation (all computed from disk) =====
import glob, json, os, shutil, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

DET, CLS = Path('/content/work/det'), Path('/content/work/cls')
RAW = {'brackish': Path('/content/data/brackish'), 'deepfish': Path('/content/data/DeepFish')}
IMG = {'.jpg', '.jpeg', '.png'}

def n_raw_images(ds):
    """images in the raw dataset folder (masks folders and BrackishMOT are not used by prepare_data.py)"""
    if not RAW[ds].exists():          # raw folder not on this runtime (e.g. restored from work.zip only)
        return None
    return sum(1 for p in RAW[ds].rglob('*')
               if p.suffix.lower() in IMG
               and not any(x.lower() in ('masks', 'mask') or 'brackishmot' in x.lower() for x in p.parts))

m = pd.read_csv(DET / 'manifest.csv')
m['labelled'] = m.n_instances > 0
_rc = DET / 'raw_counts.json'
RAWC = json.load(open(_rc)) if _rc.exists() else {}       # true raw counts saved by prepare_data.py (before the cap)

# ---------------------------------------------------------------- A. detector (Brackish + DeepFish)
rows = []
for ds in ('brackish', 'deepfish'):
    g = m[m.dataset == ds]
    raw = RAWC[ds]['raw_images'] if ds in RAWC else n_raw_images(ds)
    lab = int(g.labelled.sum())
    if raw is None:
        raw = len(g)
        print(f'[note] {ds}: raw folder missing -> "raw images" shows the count AFTER the empty-frame cap')
    kept_empty = len(g) - lab
    cnt = {t: len(glob.glob(f'{DET}/images/train/{ds}__*__{t}*.jpg')) for t in ('aug', 'ovl', 'comp')}
    tr = int((g.split == 'train').sum())
    rows.append({
        'dataset': ds, 'raw images': raw, 'fish instances': int(g.n_instances.sum()),
        'labelled images': lab, 'empty frames (raw)': raw - lab, 'empty frames kept': kept_empty,
        'images after cap': len(g), 'train (orig)': tr,
        'val': int((g.split == 'val').sum()), 'test': int((g.split == 'test').sum()),
        'train __aug': cnt['aug'], 'train __ovl': cnt['ovl'], 'train __comp': cnt['comp'],
        'TRAIN AFTER': tr + sum(cnt.values())})
det_tab = pd.DataFrame(rows).set_index('dataset')
det_tab.loc['TOTAL'] = det_tab.sum()
det_tab['growth x'] = (det_tab['TRAIN AFTER'] / det_tab['train (orig)']).round(2)
print('=== A. DETECTOR DATA (Brackish + DeepFish): before -> after augmentation ===')
display(det_tab.astype(object).T)

lab_kind = m[m.labelled].groupby(['dataset', 'label_kind']).size().unstack(fill_value=0)
lab_kind['% with a pseudo/real mask'] = (100 * (lab_kind.get('pseudo', 0) + lab_kind.get('box+pseudo', 0))
                                         / lab_kind.sum(axis=1)).round(0)
print('Label quality of the labelled images (box = plain rectangle, pseudo = GrabCut mask):')
display(lab_kind)

# ---------------------------------------------------------------- B. Fish4Knowledge
rep = pd.read_csv(CLS / 'class_report.csv')
sc = json.load(open(CLS / 'split_counts.json'))
rep['train before'] = rep['class'].map(lambda c: sc[c]['train'])
f4k = rep[['class', 'raw', 'train before', 'train_real', 'train_synth', 'train_final', 'val', 'test']].copy()
f4k.columns = ['class', 'raw crops', 'train before', 'train real kept', 'train synthetic', 'TRAIN AFTER', 'val', 'test']
tot = f4k.drop(columns='class').sum()
print('\n=== B. FISH4KNOWLEDGE (23 species): before -> after balancing + augmentation ===')
display(f4k)
print('TOTAL ->', {k: int(v) for k, v in tot.items()})
print(f"imbalance ratio (largest : smallest train class)  before = {f4k['train before'].max()} : {f4k['train before'].min()}"
      f"  ({f4k['train before'].max() / f4k['train before'].min():.0f}x)   after = {f4k['TRAIN AFTER'].max()} : {f4k['TRAIN AFTER'].min()}"
      f"  ({f4k['TRAIN AFTER'].max() / f4k['TRAIN AFTER'].min():.1f}x)")
print(f"real train crops discarded by the per-class cap: {int(tot['train before'] - tot['train real kept'])}")

# ---------------------------------------------------------------- C. headline summary
T = det_tab.loc['TOTAL']
print('\n=== C. HEADLINE NUMBERS ===')
print(f"Detector : {int(T['raw images'])} raw images -> {int(T['images after cap'])} after empty-frame cap "
      f"-> train {int(T['train (orig)'])} / val {int(T['val'])} / test {int(T['test'])}")
print(f"           TRAIN {int(T['train (orig)'])} -> {int(T['TRAIN AFTER'])}  ({T['growth x']}x)  "
      f"[+{int(T['train __aug'])} aug, +{int(T['train __ovl'])} overlay, +{int(T['train __comp'])} composite]")
print(f"           whole detector folder now: {int(T['TRAIN AFTER'] + T['val'] + T['test'])} images (val/test untouched)")
print(f"F4K      : {int(tot['raw crops'])} crops -> train {int(tot['train before'])} -> {int(tot['TRAIN AFTER'])} "
      f"({int(tot['train real kept'])} real + {int(tot['train synthetic'])} synthetic) | val {int(tot['val'])} | test {int(tot['test'])}")
ABL = Path('/content/work/ablation/ablation_index.csv')
if ABL.exists():
    ai = pd.read_csv(ABL)
    print(f"Ablation : {len(ai)} arms, baseline {int(ai[ai.arm == 'abl_baseline'].train_images.iloc[0])} imgs, "
          f"all-categories {int(ai[ai.arm == 'abl_all'].train_images.iloc[0])} imgs")
tm = DET / 'turbidity.csv'
if tm.exists():
    tt = pd.read_csv(tm)
    print('Clean vs turbid (test split):')
    display(tt[tt.split == 'test'].groupby(['dataset', 'label']).size().unstack(fill_value=0))

# ---------------------------------------------------------------- D. Brackish label/frame sanity check
Y = RAW['brackish'] / 'annotations' / 'annotations_YOLO'
if Y.exists():
    labs = [t for t in Y.glob('*.txt') if t.name.lower() != 'classes.txt']
    n_vid = len({t.stem.rsplit('-', 1)[0] for t in labs if '-' in t.stem})
    n_fr = int(det_tab.loc['brackish', 'raw images'])
    print(f'\nBrackish sanity check: {len(labs)} label files, {n_fr} frames extracted, {len(labs) - n_fr} labels without a frame ({n_vid} videos)')
    if len(labs) - n_fr == n_vid:
        print('  [WARNING] missing labels == number of videos -> the LAST label of every video has no frame.\n'
              '  Check LABEL_BASE in section 4a (label k -> frame k-1) and re-run from 4a.')

# ---------------------------------------------------------------- E. charts + save for the report
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
d = det_tab.drop(index='TOTAL')
bottom = d['train (orig)'] * 0
for col, c in (('train (orig)', '#4c78a8'), ('train __aug', '#f58518'), ('train __ovl', '#54a24b'), ('train __comp', '#e45756')):
    ax[0].bar(d.index, d[col], bottom=bottom, label=col, color=c); bottom = bottom + d[col]
ax[0].set_title('Detector TRAIN images: original + cached copies'); ax[0].set_ylabel('images'); ax[0].legend()
x = range(len(f4k)); w = 0.4
ax[1].bar([i - w / 2 for i in x], f4k['train before'], w, label='before (real)', color='#4c78a8')
ax[1].bar([i + w / 2 for i in x], f4k['TRAIN AFTER'], w, label='after (real + synthetic)', color='#f58518')
ax[1].set_yscale('log'); ax[1].set_xticks(list(x)); ax[1].set_xticklabels([c[:2] for c in f4k['class']], fontsize=8)
ax[1].set_xlabel('species id'); ax[1].set_ylabel('train crops (log)'); ax[1].set_title('Fish4Knowledge TRAIN: before vs after'); ax[1].legend()
plt.tight_layout(); plt.savefig('/content/numbers_before_after.png', dpi=150); plt.show()

det_tab.to_csv('/content/numbers_detector.csv'); f4k.to_csv('/content/numbers_f4k.csv', index=False)
ra = '/content/drive/MyDrive/aquavision/report_assets'
if os.path.isdir(ra):
    for f in ('numbers_detector.csv', 'numbers_f4k.csv', 'numbers_before_after.png'):
        shutil.copy2(f'/content/{f}', ra)
    print('saved numbers_detector.csv, numbers_f4k.csv, numbers_before_after.png -> Drive report_assets')

In [ ]:
!ls /content/work /content/data /content/aquavision 2>&1 | head -30

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/aquavision/work.zip 'det/*' -d /content/work
!ls /content/work/det | head -20

## 9. Dataset statistics (for the report)

In [ ]:
import pandas as pd
from IPython.display import display, Image
det = '/content/work/det'
m = pd.read_csv(f'{det}/manifest.csv')
tab = m.groupby(['dataset', 'split']).agg(images=('file', 'count'), instances=('n_instances', 'sum'),
                                          labelled_images=('n_instances', lambda s: int((s > 0).sum()))).reset_index()
display(tab); tab.to_csv(f'{det}/report_detector_split_table.csv', index=False)
print(m.groupby(['dataset', 'label_kind']).size())
cnt = {t: len(glob.glob(f'{det}/images/train/*__{t}*.jpg')) for t in ('aug', 'ovl', 'comp')}
print('cached train copies:', cnt, '| train total:', len(glob.glob(f'{det}/images/train/*.jpg')))
print('val:', len(glob.glob(f'{det}/images/val/*.jpg')), '| test:', len(glob.glob(f'{det}/images/test/*.jpg')))
print('\n=== Fish4Knowledge classifier ===')
display(pd.read_csv('/content/work/cls/class_report.csv'))
display(Image('/content/work/cls/class_balance.png'))
display(Image('/content/work/cls/rare_class_preview.jpg'))

In [ ]:
# ===== 9b. BEFORE vs AFTER (all numbers computed from disk) =====
import glob, json, os, random, shutil
import cv2, numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

DET, CLS = Path('/content/work/det'), Path('/content/work/cls')
RA = Path('/content/drive/MyDrive/aquavision/report_assets')
TAGS = ('aug', 'ovl', 'comp')
m = pd.read_csv(DET / 'manifest.csv')
RAWC = json.load(open(DET / 'raw_counts.json')) if (DET / 'raw_counts.json').exists() else {}
rep = pd.read_csv(CLS / 'class_report.csv')
sc = json.load(open(CLS / 'split_counts.json'))

def kind_of(name):
    return next((t for t in TAGS if f'__{t}' in name), 'original')

# ---------- index every detector train image once: dataset, kind, label lines
rows = []
for p in (DET / 'images' / 'train').glob('*.jpg'):
    lab = DET / 'labels' / 'train' / f'{p.stem}.txt'
    n = sum(1 for l in lab.read_text().splitlines() if l.strip()) if lab.exists() else 0
    rows.append((p.name.split('__')[0], kind_of(p.stem), n))
idx = pd.DataFrame(rows, columns=['dataset', 'kind', 'instances'])

# ================================================================ A. GRAND SUMMARY
summ = []
for ds in ('brackish', 'deepfish'):
    g = m[m.dataset == ds]
    raw = RAWC[ds]['raw_images'] if ds in RAWC else len(g)
    tr0 = int((g.split == 'train').sum())
    add = {t: int(((idx.dataset == ds) & (idx.kind == t)).sum()) for t in TAGS}
    summ.append({'dataset': ds + ' (detector)', 'raw images': raw, 'dropped before split (empty-frame cap)': raw - len(g),
                 'train BEFORE': tr0, '+ added: aug': add['aug'], '+ added: overlay': add['ovl'], '+ added: composite': add['comp'],
                 '- removed from train': 0, 'train AFTER': tr0 + sum(add.values()),
                 'val (untouched)': int((g.split == 'val').sum()), 'test (untouched)': int((g.split == 'test').sum())})
ft = rep.copy()
ft['train before'] = ft['class'].map(lambda c: sc[c]['train'])
tb, kept, syn = int(ft['train before'].sum()), int(ft['train_real'].sum()), int(ft['train_synth'].sum())
summ.append({'dataset': 'fish4knowledge (classifier)', 'raw images': int(ft['raw'].sum()), 'dropped before split (empty-frame cap)': 0,
             'train BEFORE': tb, '+ added: aug': syn, '+ added: overlay': 0, '+ added: composite': 0,
             '- removed from train': tb - kept, 'train AFTER': kept + syn,
             'val (untouched)': int(ft['val'].sum()), 'test (untouched)': int(ft['test'].sum())})
S_ = pd.DataFrame(summ).set_index('dataset')
S_.loc['TOTAL (3 datasets)'] = S_.sum()
S_['added (total)'] = S_['+ added: aug'] + S_['+ added: overlay'] + S_['+ added: composite']
S_['train growth x'] = (S_['train AFTER'] / S_['train BEFORE']).round(2)
print('=== A. IMAGES: BEFORE -> ADDED / REMOVED -> AFTER (train split; val/test are never changed) ===')
display(S_.astype(object).T)
T = S_.loc['TOTAL (3 datasets)']
print(f"TOTAL: train {int(T['train BEFORE'])} -> {int(T['train AFTER'])}   (+{int(T['added (total)'])} added, -{int(T['- removed from train'])} real crops removed by the class cap)")
det_T = S_.loc[['brackish (detector)', 'deepfish (detector)']]
print(f"DETECTOR: train {int(det_T['train BEFORE'].sum())} -> {int(det_T['train AFTER'].sum())}  (+{int(det_T['added (total)'].sum())} added images)")
f_ = S_.loc['fish4knowledge (classifier)']
print(f"CLASSIFIER: train {int(f_['train BEFORE'])} -> {int(f_['train AFTER'])}  (+{int(f_['added (total)'])} synthetic, -{int(f_['- removed from train'])} real removed from big classes)")

# ================================================================ B. FISH LABELS
orig_i = idx[idx.kind == 'original'].groupby('dataset').instances.sum()
all_i = idx.groupby('dataset').instances.sum()
B_ = pd.DataFrame({'fish labels BEFORE (train originals)': orig_i, 'fish labels AFTER (train + copies)': all_i})
B_['added labels'] = B_.iloc[:, 1] - B_.iloc[:, 0]
B_['growth x'] = (B_.iloc[:, 1] / B_.iloc[:, 0]).round(2)
print('\n=== B. FISH LABELS (instances) in the train split ===')
display(B_)
print('(aug and overlay copies keep the original labels; each composite copy adds 1 pasted fish)')

# ================================================================ C. LABEL QUALITY
lab_ = m[m.n_instances > 0].groupby(['dataset', 'label_kind']).size().unstack(fill_value=0)
masks = (lab_.get('pseudo', 0) + lab_.get('box+pseudo', 0))
C_ = pd.DataFrame({'labelled images': lab_.sum(axis=1), 'with mask BEFORE': 0, 'with mask AFTER (GrabCut)': masks})
C_['% with mask BEFORE'] = 0.0
C_['% with mask AFTER'] = (100 * C_['with mask AFTER (GrabCut)'] / C_['labelled images']).round(1)
print('\n=== C. LABEL QUALITY: plain rectangles (before) -> segmentation masks (after) ===')
display(C_)

# ================================================================ D. IMAGE PROPERTIES (paired)
def props(img):
    f = 320.0 / max(img.shape[:2])
    if f < 1: img = cv2.resize(img, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY); hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV); lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    dark = cv2.erode(img.min(axis=2), np.ones((9, 9), np.uint8)).mean()
    return {'brightness': g.mean(), 'contrast': g.std(), 'saturation': hsv[..., 1].mean(), 'haze (dark channel)': dark,
            'L': lab[..., 0].mean(), 'a': lab[..., 1].mean(), 'b': lab[..., 2].mean()}

rng = random.Random(0)
P = []                                   # rows: dataset, which (original/augmented), properties
for ds in ('brackish', 'deepfish'):
    stems = [p.stem for p in (DET / 'images' / 'train').glob(f'{ds}__*__aug0.jpg')]
    rng.shuffle(stems)
    n = 0
    for s in stems:
        o = DET / 'images' / 'train' / (s.replace('__aug0', '') + '.jpg'); a = DET / 'images' / 'train' / f'{s}.jpg'
        io, ia = cv2.imread(str(o)), cv2.imread(str(a))
        if io is None or ia is None: continue
        P.append({'dataset': ds, 'which': 'BEFORE (original)', **props(io)}); P.append({'dataset': ds, 'which': 'AFTER (augmented copy)', **props(ia)})
        n += 1
        if n == 150: break
P = pd.DataFrame(P)
cols = ['brightness', 'contrast', 'saturation', 'haze (dark channel)']
D_ = P.groupby(['dataset', 'which'])[cols].mean().round(1)
print('\n=== D. IMAGE PROPERTIES: original vs its augmented copy (mean of 150 pairs per dataset) ===')
display(D_)
ch = (P[P.which.str.startswith('AFTER')].groupby('dataset')[cols].mean() / P[P.which.str.startswith('BEFORE')].groupby('dataset')[cols].mean() - 1) * 100
print('change after augmentation (%):'); display(ch.round(1))
print('(haze up / saturation down / contrast down = murkier water, i.e. the turbidity the model must learn to handle)')

# ================================================================ E. DOMAIN GAP
lab_cols = ['L', 'a', 'b']
mu_b = P[P.which.str.startswith('BEFORE')].groupby('dataset')[lab_cols].mean()
mu_a = P[P.which.str.startswith('AFTER')].groupby('dataset')[lab_cols].mean()
gap_b = float(np.linalg.norm(mu_b.loc['brackish'] - mu_b.loc['deepfish']))
gap_a = float(np.linalg.norm(mu_a.loc['brackish'] - mu_a.loc['deepfish']))
E_ = pd.DataFrame({'mean colour (LAB) BEFORE': mu_b.round(1).astype(str).agg(' / '.join, axis=1),
                   'mean colour (LAB) AFTER': mu_a.round(1).astype(str).agg(' / '.join, axis=1)})
print('\n=== E. DOMAIN GAP between Brackish and DeepFish (distance of mean LAB colour) ===')
display(E_)
print(f'gap BEFORE = {gap_b:.1f}   AFTER = {gap_a:.1f}   ({100 * (gap_a / gap_b - 1):+.0f}%)   [smaller = the two datasets look more alike = cross-dataset mixing works]')

# ================================================================ F. FISH4KNOWLEDGE real vs synthetic (rare classes)
F = []
for c in rep[rep['raw'] < 150]['class']:
    fs = sorted((CLS / 'train' / c).glob('*.png')) + sorted((CLS / 'train' / c).glob('*.jpg'))
    real = [f for f in fs if not f.name.startswith('aug')][:15]; synth = [f for f in fs if f.name.startswith('aug')][:15]
    for lbl, lst in (('real', real), ('synthetic', synth)):
        for f in lst:
            im = cv2.imread(str(f))
            if im is not None: F.append({'which': lbl, **props(im)})
F = pd.DataFrame(F)
if len(F):
    print('\n=== F. FISH4KNOWLEDGE rare classes (raw < 150 crops): real vs synthetic crops (mean of sampled crops) ===')
    display(F.groupby('which')[cols].mean().round(1))
imb_b, imb_a = ft['train before'].max() / ft['train before'].min(), ft['train_final'].max() / ft['train_final'].min()
print(f"class imbalance (largest : smallest train class): BEFORE {ft['train before'].max()} : {ft['train before'].min()} ({imb_b:.0f}x)   ->   AFTER {ft['train_final'].max()} : {ft['train_final'].min()} ({imb_a:.1f}x)")

# ================================================================ G. FIGURES
fig, ax = plt.subplots(figsize=(8, 4.5))
d = S_.drop(index='TOTAL (3 datasets)'); xs = np.arange(len(d)); w = 0.38
ax.bar(xs - w/2, d['train BEFORE'], w, label='before', color='#4c78a8'); ax.bar(xs + w/2, d['train AFTER'], w, label='after', color='#f58518')
for i, (b0, a0, ad) in enumerate(zip(d['train BEFORE'], d['train AFTER'], d['added (total)'])):
    ax.text(i + w/2, a0, f'+{int(ad)}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels([i.replace(' (', '\n(') for i in d.index]); ax.set_ylabel('train images'); ax.legend()
ax.set_title('Train images: before vs after (numbers = images added)'); plt.tight_layout(); plt.savefig('/content/before_after_summary.png', dpi=150); plt.show()

def draw(path):
    im = cv2.imread(str(path))
    if im is None: return np.zeros((200, 300, 3), np.uint8)
    H, W = im.shape[:2]; lab = DET / 'labels' / 'train' / f'{Path(path).stem}.txt'
    if lab.exists():
        for l in lab.read_text().splitlines():
            v = l.split()
            if len(v) >= 7:
                pts = (np.array(list(map(float, v[1:])), np.float32).reshape(-1, 2) * [W, H]).astype(np.int32)
                cv2.polylines(im, [pts], True, (0, 255, 0), max(2, W // 400))
    f = 640.0 / W; im = cv2.resize(im, (640, int(H * f)))
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

fig, axs = plt.subplots(2, 4, figsize=(18, 7))
for r, ds in enumerate(('brackish', 'deepfish')):
    a0 = sorted((DET / 'images' / 'train').glob(f'{ds}__*__aug0.jpg'))
    labelled = [p for p in a0 if (DET / 'labels' / 'train' / f'{p.stem}.txt').read_text().strip()]
    pick = random.Random(7).choice(labelled or a0)
    orig = DET / 'images' / 'train' / (pick.stem.replace('__aug0', '') + '.jpg')
    ov = sorted((DET / 'images' / 'train').glob(f'{ds}__*__ovl0.jpg')); cp = sorted((DET / 'images' / 'train').glob(f'{ds}__*__comp0.jpg'))
    items = [(orig, 'BEFORE: original'), (pick, 'AFTER: augmented copy'),
             (random.Random(3).choice(ov) if ov else None, 'AFTER: + overlay / erasing'),
             (random.Random(3).choice(cp) if cp else None, 'AFTER: + pasted fish from the other dataset')]
    for c, (p, t) in enumerate(items):
        axs[r, c].imshow(draw(p) if p else np.zeros((200, 300, 3), np.uint8)); axs[r, c].set_title(f'{ds}: {t}', fontsize=10); axs[r, c].axis('off')
plt.tight_layout(); plt.savefig('/content/before_after_examples.png', dpi=130); plt.show()

# ================================================================ SAVE
S_.to_csv('/content/before_after_summary.csv'); B_.to_csv('/content/before_after_labels.csv'); C_.to_csv('/content/before_after_label_quality.csv')
D_.to_csv('/content/before_after_properties.csv')
files = ['before_after_summary.csv', 'before_after_labels.csv', 'before_after_label_quality.csv', 'before_after_properties.csv',
         'before_after_summary.png', 'before_after_examples.png']
if RA.is_dir():
    for f in files: shutil.copy2(f'/content/{f}', RA)
    print('saved to Drive report_assets:', ', '.join(files))
else:
    print('[note] Drive report_assets not found - files are in /content/ (run section 10 or copy them yourself)')

## 9c. Originals-only train list (`data_orig.yaml`, from the training-fix notebook)
Train on originals only (no `__aug`/`__ovl`/`__comp` copies) for the baseline-style comparison run.

In [ ]:
import re
from pathlib import Path
DET = Path('/content/work/det')
imgs = sorted((DET/'images'/'train').glob('*.jpg'))
(DET/'lists').mkdir(exist_ok=True)
is_copy = lambda s: re.search(r'__(aug|ovl|comp)\d+$', s)
lst = DET/'lists'/'train_orig.txt'
keep = [p for p in imgs if not is_copy(p.stem)]
lst.write_text('\n'.join(str(p.resolve()) for p in keep) + '\n')
y = re.sub(r'(?m)^train:.*$', f'train: {lst.resolve()}', (DET/'data_full.yaml').read_text())
(DET/'data_orig.yaml').write_text(y)
print('data_orig.yaml:', len(keep), 'train images')   # expect 12489

## 10. Save to Drive (hand-off to the training teammates)
Do not interrupt the zip. JPEGs are already compressed, so the zip is stored (fast).

In [ ]:
!mkdir -p /content/drive/MyDrive/aquavision/report_assets
!cd /content/work && zip -qr0 /content/drive/MyDrive/aquavision/work.zip det cls ablation
!cp /content/aquavision/*.py /content/drive/MyDrive/aquavision/
!cp /content/aug_preview.jpg /content/work/cls/class_balance.png /content/work/cls/rare_class_preview.jpg /content/work/cls/class_report.csv /content/work/det/report_detector_split_table.csv /content/aug_counts_detector.csv /content/aug_counts_f4k.csv /content/drive/MyDrive/aquavision/report_assets/
!ls -lh /content/drive/MyDrive/aquavision
!cp /content/work/det/turbidity.csv /content/work/ablation/ablation_index.csv /content/drive/MyDrive/aquavision/report_assets/ 2>/dev/null || true
!cp -r /content/work/cls_eval /content/drive/MyDrive/aquavision/ 2>/dev/null || true   # turbid/clean F4K test copies for the classifier evaluation


## 11. Hand-off notes for training
- Detector data = **Brackish + DeepFish** (OZFish not used). Phase 1: `data_full.yaml` (mosaic + mixup on). Phase 2 (last ~15 epochs): `data_late.yaml` + `mosaic=0, mixup=0` (no overlays/composites). Reference code: `train_aug_config.py`.
  Training-config fix (from the segmenter baseline, best epoch 20/100): `train_detector` now uses `patience=25` (phase 1), `batch=16`, and a phase-2 warm-up fix (`optimizer="SGD"`, `lr0=0.001`, `warmup_epochs=0`). `data_orig.yaml` (originals only, `lists/train_orig.txt`) is built in section 9c. Not in this notebook: `prepare_data.py --size_strata 2`, `test_metrics.json`, recall-by-fish-size (use the updated segmenter scripts); `--bg_ratio` defaults to 1.0 here (brief suggests 0.3-0.5, a separate data step).
- Classifier: `/content/work/cls` (train/val/test). `train/` is class-balanced, val/test untouched (identical to the baseline split: seed 42, min_eval 2). `class_weights.json` is optional (train is already balanced offline - do not double-count).
  Build with `--target 300 --max_per_class 2500 --ceil_mult 10 --max_aug_ratio 12 --extra_aug 0.2 --ref_det /content/work/det --seed 42` (section 8). Train with `train_aug_config.train_classifier` (same model/schedule as the baseline + `CLS_ONLINE` flags). Evaluate with `eval_cls.py` on `cls/test` and `cls_eval/test_turbid`; report top-1, macro-F1 + CI, head/mid/tail, the clip-split-only macro-F1 and per-class table.
  Control job: `--max_per_class 100000 --ceil_mult 1000` (both are needed to remove the cap).
  Limits to state: fish compositing is not applied to crops; very rare classes (11-30 real crops) gain little from copies; a macro-F1 change < ~0.05 is noise; F4K -> Brackish/DeepFish accuracy cannot be measured (no species labels) - only the synthetic turbid test.
- Ablation (GPU): `python ablation_run.py --det /content/work/det --abl /content/work/ablation --epochs 20` -> `ablation_results.csv` (clean vs turbid; mAP, box IoU, mask-boundary error). Held-out robustness run: train with `data_trainclean.yaml`, evaluate `split='test'` (turbid).
- Report rare-class caveat: classes flagged `VERY-LOW` / `test<5` have noisy macro-F1; show per-class results.

## 12. Edits to make in the report PDF
1. Sec. 2: data-prep bullet -> DeepFish and Brackish.
2. Sec. 3 (Detection): 'joint OZFish+Brackish training' -> DeepFish+Brackish.
3. Sec. 5 Table 1: delete the OZFish row; Brackish becomes the main detector data.
4. Sec. 5 Data paragraph: drop OZFish from the 'boxes/masks from...' sentence.
5. References: remove [9] and renumber.
6. Sec. 6.1: add one sentence - Fish4Knowledge uses a milder, classifier-safe version of the same tiers.
7. Sec. 6.1: the clean-vs-turbid split and per-category ablation are now prepared here (`turbid_split.py`, `ablation_prep.py`); report the results from `ablation_run.py` (GPU) - bounding-box IoU and mask-boundary error are computed there.
8. Sec. 6.1: say Fish4Knowledge crops get the same three tiers (geometric, photometric, small overlay + cross-dataset colour transfer) except fish compositing.
9. If you want OZFish back (PDF Table 1 lists it as primary detector data): unzip it to `/content/data/ozfish`; section 4b picks it up automatically.
- **v4:** Brackish frames are now aligned (label k -> frame k-1). Any detector data or weights built from v3 should be rebuilt/retrained.


In [ ]:
!unzip -q -o /content/drive/MyDrive/aquavision/work.zip 'ablation/*' -d /content/work
!ls /content/work

In [ ]:
!df -h /content/drive | tail -1
!cd /content/work && zip -qr0 /content/drive/MyDrive/aquavision/work_new.zip det cls ablation cls_eval
!ls -la /content/drive/MyDrive/aquavision/*.zip

In [ ]:
!cd /content/work && zip -qr0 /content/drive/MyDrive/aquavision/work_new.zip det cls ablation cls_eval
!ls -la /content/drive/MyDrive/aquavision/*.zip

In [ ]:
!mv /content/drive/MyDrive/aquavision/work.zip /content/drive/MyDrive/aquavision/work_old_backup.zip
!mv /content/drive/MyDrive/aquavision/work_new.zip /content/drive/MyDrive/aquavision/work.zip

## Appendix - maintenance / session-repair cells (from the training-fix notebook)
Not part of the normal top-to-bottom run. Use only when needed: A1 restores `work.zip` (it OVERWRITES freshly built data), A2 and A4 DELETE files, A3 is the alternative local-zip route (it does not include `cls_eval`; section 10 does).

### A1 - restore det from Drive + sanity check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/aquavision/ | head -30

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls -la /content/drive/MyDrive/aquavision/work.zip
!mkdir -p /content/work && unzip -q -o /content/drive/MyDrive/aquavision/work.zip -d /content/work
!ls /content/work /content/work/det | head -20

In [ ]:
PSEUDO_MASKS = True

import os, glob
assert os.path.isdir('/content/work/det/images/train'), \
    'det folder is gone (session reset) - do NOT continue; use RESTORE_FROM_DRIVE = True in a fresh session'
for ds in ('brackish', 'deepfish'):
    print(ds, {t: len(glob.glob(f'/content/work/det/images/train/{ds}__*__{t}*.jpg'))
               for t in ('aug', 'ovl', 'comp')})

In [ ]:
   for ds in ('brackish', 'deepfish'):
       print(ds, {t: len(glob.glob(f'/content/work/det/images/train/{ds}__*__{t}*.jpg'))
                  for t in ('aug', 'ovl', 'comp')})

In [ ]:
!ls /content/work/det

### A2 - remove __ovl/__comp copies (they get redone by section 7) + has-fish check

In [ ]:
import glob, os
n = 0
for t in ('__ovl', '__comp'):
    for p in glob.glob(f'/content/work/det/images/train/*{t}*.jpg'):
        os.remove(p)
        lab = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lab):
            os.remove(lab)
        n += 1
print('removed', n)

In [ ]:
import pandas as pd
m = pd.read_csv('/content/work/det/manifest.csv')
m['has_fish'] = m.n_instances > 0
print(m.groupby(['dataset', 'split']).has_fish.agg(total='size', with_fish='sum'))

### A3 - alternative save route via local zip (low Drive space)

In [ ]:
!df -h /content | tail -1
!cd /content/work && zip -qr0 /content/work_new.zip det cls ablation
!ls -lh /content/work_new.zip
!cp /content/work_new.zip /content/drive/MyDrive/aquavision/work_new.zip
!ls -lh /content/work_new.zip /content/drive/MyDrive/aquavision/work_new.zip

In [ ]:
!unzip -l /content/drive/MyDrive/aquavision/work_new.zip | tail -1
!mv -f /content/drive/MyDrive/aquavision/work_new.zip /content/drive/MyDrive/aquavision/work.zip
!ls -lh /content/drive/MyDrive/aquavision/work.zip

### A4 - clean orphan files vs manifest (set APPLY = False first to only look)

In [ ]:
import pandas as pd, re, sys
from pathlib import Path
sys.path.insert(0, '/content/aquavision')
from prepare_data import group_key

APPLY = True        # first run: only look. Set True and re-run to delete.

DET = Path('/content/work/det')
m = pd.read_csv(DET/'manifest.csv')
keep = {sp: set(m[m.split == sp].file.str[:-4]) for sp in ('train', 'val', 'test')}
base = lambda s: re.sub(r'__(aug|ovl|comp)\d+$', '', s)
is_copy = lambda s: base(s) != s
clip = lambda stems: {group_key(s.split('__', 1)[1]) for s in stems}

bad, disk, all_ok = [], {}, True
for sp in ('train', 'val', 'test'):
    files = list((DET/'images'/sp).glob('*.jpg'))
    orig = {p.stem for p in files if not is_copy(p.stem)}
    disk[sp] = orig
    orphan = [p for p in files if base(p.stem) not in keep[sp]]
    all_ok &= not (keep[sp] - orig)
    print(f'{sp:5s} disk {len(orig):6d} | manifest {len(keep[sp]):6d} | extra {len(orig-keep[sp]):5d} '
          f'| missing {len(keep[sp]-orig):4d} | files to delete {len(orphan)}')
    bad += orphan
for name, S in (('on disk', disk), ('manifest', keep)):
    c = {sp: clip(S[sp]) for sp in S}
    print(f'clip overlap ({name}): train&val {len(c["train"]&c["val"])}, '
          f'train&test {len(c["train"]&c["test"])}, val&test {len(c["val"]&c["test"])}')

if APPLY:
    assert all_ok, 'manifest files are missing on disk - do NOT delete, send me the output'
    n = 0
    for p in bad + [q for q in (DET/'images'/'train').glob('*.jpg')
                    if '__ovl' in q.stem or '__comp' in q.stem]:
        p.unlink(missing_ok=True)
        (DET/'labels'/p.parent.name/f'{p.stem}.txt').unlink(missing_ok=True)
        n += 1
    print('deleted', n, '(extras + all __ovl/__comp, which are redone next)')

### A5 - reload det if the session reset + before/after proof plots + summary tables

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE = "/content/drive/MyDrive/aquavision"
DET = "/content/work/det"
if not os.path.isdir(f"{DET}/images/train"):
    !mkdir -p /content/work
    !unzip -q -o "{DRIVE}/work.zip" "det/*" -d /content/work
print({s: len(glob.glob(f"{DET}/images/{s}/*.jpg")) for s in ("train", "val", "test")})

In [ ]:
import os, re, glob, pandas as pd, matplotlib.pyplot as plt
cp = re.compile(r"__(aug|ovl|comp)\d+$")
rows = []
for sp in ("train", "val", "test"):
    for p in glob.glob(f"{DET}/images/{sp}/*.jpg"):
        stem = os.path.basename(p)[:-4]
        m = cp.search(stem)
        lab = f"{DET}/labels/{sp}/{stem}.txt"
        n = sum(1 for l in open(lab).read().splitlines() if l.strip()) if os.path.exists(lab) else 0
        rows.append((sp, stem.split("__")[0], m.group(1) if m else "original", n))
df = pd.DataFrame(rows, columns=["split", "dataset", "kind", "labels"])
df["empty"] = df.labels == 0

# 1) images per dataset / split / kind
t1 = df.pivot_table(index=["dataset", "kind"], columns="split", values="labels", aggfunc="size", fill_value=0)
t1["total"] = t1.sum(axis=1)
print("1) IMAGES per dataset / kind / split"); display(t1)

# 2) BEFORE vs AFTER augmentation (train only; val/test are never augmented)
tr = df[df.split == "train"]
before = tr[tr.kind == "original"].groupby("dataset").agg(images=("labels", "size"), fish_labels=("labels", "sum"), empty_frames=("empty", "sum"))
after = tr.groupby("dataset").agg(images=("labels", "size"), fish_labels=("labels", "sum"), empty_frames=("empty", "sum"))
ba = before.join(after, lsuffix="_before", rsuffix="_after")
ba.loc["TOTAL"] = ba.sum()
ba["image_growth_x"] = (ba.images_after / ba.images_before).round(2)
ba["label_growth_x"] = (ba.fish_labels_after / ba.fish_labels_before).round(2)
print("2) TRAIN SET: BEFORE (originals) vs AFTER (originals + augmented copies)"); display(ba)

# 3) what the augmented copies are made of
t3 = tr[tr.kind != "original"].groupby(["dataset", "kind"]).size().unstack(fill_value=0)
t3["total_copies"] = t3.sum(axis=1)
print("3) AUGMENTED COPIES by type"); display(t3)

# 4) bar chart: train images and fish labels before vs after
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
b = ba.drop("TOTAL"); x = range(len(b))
ax[0].bar([i - 0.2 for i in x], b.images_before, 0.4, label="before"); ax[0].bar([i + 0.2 for i in x], b.images_after, 0.4, label="after")
ax[0].set_xticks(list(x)); ax[0].set_xticklabels(b.index); ax[0].set_title("Train images"); ax[0].legend()
ax[1].bar([i - 0.2 for i in x], b.fish_labels_before, 0.4, label="before"); ax[1].bar([i + 0.2 for i in x], b.fish_labels_after, 0.4, label="after")
ax[1].set_xticks(list(x)); ax[1].set_xticklabels(b.index); ax[1].set_title("Fish labels"); ax[1].legend()
plt.tight_layout(); plt.savefig("/content/before_after_counts.png", dpi=150); plt.show()
ba.to_csv("/content/before_after_train.csv"); t1.to_csv("/content/images_by_split.csv")
print("saved /content/before_after_counts.png, before_after_train.csv, images_by_split.csv")

In [ ]:
import os, glob, random, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def draw(ax, img_path, title):
    im = Image.open(img_path).convert("RGB"); W, H = im.size
    ax.imshow(im); ax.set_title(title, fontsize=9); ax.axis("off")
    lab = img_path.replace("/images/", "/labels/")[:-4] + ".txt"
    if os.path.exists(lab):
        for l in open(lab).read().splitlines():
            v = l.split()
            if len(v) >= 7:
                xy = np.array(v[1:], float).reshape(-1, 2) * [W, H]
                ax.plot(*np.vstack([xy, xy[:1]]).T, lw=1.5, color="yellow")

random.seed(1); shown = 0
for ds in ("brackish", "deepfish"):
    origs = [p for p in glob.glob(f"{DET}/images/train/{ds}__*.jpg")
             if not cp.search(os.path.basename(p)[:-4])
             and os.path.exists(p.replace("/images/", "/labels/")[:-4] + ".txt")
             and os.path.getsize(p.replace("/images/", "/labels/")[:-4] + ".txt") > 0]
    random.shuffle(origs)
    for p in origs[:3]:
        stem = p[:-4]
        copies = sorted(glob.glob(stem + "__aug*.jpg"))[:1] + sorted(glob.glob(stem + "__ovl*.jpg"))[:1] + sorted(glob.glob(stem + "__comp*.jpg"))[:1]
        fig, axs = plt.subplots(1, 1 + len(copies), figsize=(4.2 * (1 + len(copies)), 3.4))
        axs = np.atleast_1d(axs)
        draw(axs[0], p, f"BEFORE: original ({ds})")
        for a, c in zip(axs[1:], copies):
            draw(a, c, "AFTER: " + re.search(r"__(aug|ovl|comp)\d+", c).group(1) + " copy")
        plt.tight_layout(); plt.show(); shown += 1
print("shown", shown, "examples (yellow outline = the fish label)")

In [ ]:
import os
import re
import glob
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Define the regex compiled pattern for identifying copies
cp = re.compile(r"__(aug|ovl|comp)\d+$")

def draw(ax, img_path, title):
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    lab = img_path.replace("/images/", "/labels/")[:-4] + ".txt"
    if os.path.exists(lab):
        for l in open(lab).read().splitlines():
            v = l.split()
            if len(v) >= 7:
                xy = np.array(v[1:], float).reshape(-1, 2) * [W, H]
                ax.plot(*np.vstack([xy, xy[:1]]).T, lw=1.5, color="yellow")

DET = '/content/work/det'
random.seed(1)
shown = 0
for ds in ("brackish", "deepfish"):
    origs = [p for p in glob.glob(f"{DET}/images/train/{ds}__*.jpg")
             if not cp.search(os.path.basename(p)[:-4])
             and os.path.exists(p.replace("/images/", "/labels/")[:-4] + ".txt")
             and os.path.getsize(p.replace("/images/", "/labels/")[:-4] + ".txt") > 0]
    random.shuffle(origs)
    for p in origs[:3]:
        stem = p[:-4]
        copies = sorted(glob.glob(stem + "__aug*.jpg"))[:1] + sorted(glob.glob(stem + "__ovl*.jpg"))[:1] + sorted(glob.glob(stem + "__comp*.jpg"))[:1]
        fig, axs = plt.subplots(1, 1 + len(copies), figsize=(4.2 * (1 + len(copies)), 3.4))
        axs = np.atleast_1d(axs)
        draw(axs[0], p, f"BEFORE: original ({ds})")
        for a, c in zip(axs[1:], copies):
            draw(a, c, "AFTER: " + re.search(r"__(aug|ovl|comp)\d+", c).group(1) + " copy")
        plt.tight_layout()
        plt.show()
        shown += 1
print("shown", shown, "examples (yellow outline = the fish label)")

In [ ]:
import os
import glob
import json
import pandas as pd
from IPython.display import display, HTML

DET = '/content/work/det'
CLS = '/content/work/cls'

# Helper to display formatted markdown tables via HTML
def print_md_table(title, headers, rows):
    html = f'<h3>{title}</h3><table border="1" style="border-collapse: collapse; text-align: left;">'
    html += '<tr>' + ''.join(f'<th style="padding: 8px; background-color: #f2f2f2;">{h}</th>' for h in headers) + '</tr>'
    for row in rows:
        html += '<tr>' + ''.join(f'<td style="padding: 8px;">{val}</td>' for val in row) + '</tr>'
    html += '</table><br>'
    display(HTML(html))

# 1. DETECTOR SUMMARY
tr_imgs = glob.glob(f'{DET}/images/train/*.jpg')
val_imgs = glob.glob(f'{DET}/images/val/*.jpg')
test_imgs = glob.glob(f'{DET}/images/test/*.jpg')

# Group training by dataset and kind
detector_counts = {}
for path in tr_imgs:
    filename = os.path.basename(path)
    ds = filename.split('__')[0]
    kind = 'original'
    for t in ('aug', 'ovl', 'comp'):
        if f'__{t}' in filename:
            kind = t
            break
    detector_counts.setdefault(ds, {'original': 0, 'aug': 0, 'ovl': 0, 'comp': 0})[kind] += 1

det_rows = []
for ds, counts in detector_counts.items():
    orig = counts['original']
    aug = counts['aug']
    ovl = counts['ovl']
    comp = counts['comp']
    total_train = orig + aug + ovl + comp
    val_cnt = sum(1 for p in val_imgs if os.path.basename(p).startswith(ds))
    test_cnt = sum(1 for p in test_imgs if os.path.basename(p).startswith(ds))
    growth = f'{total_train / orig:.2f}x' if orig > 0 else 'N/A'
    det_rows.append([ds, orig, aug, ovl, comp, total_train, val_cnt, test_cnt, growth])

print_md_table(
    "Detector Datasets: Offline Augmentation Summary (Train Split)",
    ["Dataset", "Train (Original)", "__aug (Photometric)", "__ovl (Occlusion)", "__comp (Compositing)", "Train (Final)", "Val (Untouched)", "Test (Untouched)", "Growth"],
    det_rows
)

# 2. CLASSIFIER SUMMARY (Fish4Knowledge)
class_report_path = f'{CLS}/class_report.csv'
if os.path.exists(class_report_path):
    try:
        df_cls = pd.read_csv(class_report_path)
        cls_rows = []
        for _, r in df_cls.iterrows():
            cls_rows.append([
                r['class'],
                r['raw'],
                r['train_real'],
                r['train_synth'],
                r['train_final'],
                r['val'],
                r['test'],
                r['split_mode']
            ])
        print_md_table(
            "Fish4Knowledge (23-Class Classifier): Balancing and Augmentation Summary",
            ["Species Class", "Raw Crops", "Train (Real Kept)", "Train (Synthetic)", "Train (Final)", "Val", "Test", "Split Mode"],
            cls_rows
        )
    except Exception as e:
        print(f"Could not parse class_report.csv: {e}")
else:
    print("Fish4Knowledge class report not found. Please verify the classification data split has been generated.")


In [ ]:
import os
import glob
from IPython.display import display, HTML

DET = '/content/work/det'

def print_html_table(title, headers, rows):
    html = f'<h3>{title}</h3><table border="1" style="border-collapse: collapse; text-align: left; width: 100%;">'
    html += '<tr>' + ''.join(f'<th style="padding: 8px; background-color: #f2f2f2;">{h}</th>' for h in headers) + '</tr>'
    for row in rows:
        html += '<tr>' + ''.join(f'<td style="padding: 8px;">{val}</td>' for val in row) + '</tr>'
    html += '</table><br>'
    display(HTML(html))

# Group training images by dataset and kind for Segmentation
tr_imgs = glob.glob(f'{DET}/images/train/*.jpg')
val_imgs = glob.glob(f'{DET}/images/val/*.jpg')
test_imgs = glob.glob(f'{DET}/images/test/*.jpg')

segmentation_counts = {}
for path in tr_imgs:
    filename = os.path.basename(path)
    ds = filename.split('__')[0]
    kind = 'original'
    for t in ('aug', 'ovl', 'comp'):
        if f'__{t}' in filename:
            kind = t
            break
    segmentation_counts.setdefault(ds, {'original': 0, 'aug': 0, 'ovl': 0, 'comp': 0})[kind] += 1

seg_rows = []
for ds, counts in sorted(segmentation_counts.items()):
    orig = counts['original']
    aug = counts['aug']
    ovl = counts['ovl']
    comp = counts['comp']
    total_train = orig + aug + ovl + comp
    val_cnt = sum(1 for p in val_imgs if os.path.basename(p).startswith(ds))
    test_cnt = sum(1 for p in test_imgs if os.path.basename(p).startswith(ds))
    growth = f'{total_train / orig:.2f}x' if orig > 0 else 'N/A'
    seg_rows.append([ds, str(orig), str(aug), str(ovl), str(comp), str(total_train), str(val_cnt), str(test_cnt), growth])

print_html_table(
    "Segmentation Datasets (Stage A): Offline Augmentation & Split Summary (Train Split)",
    ["Dataset", "Train (Original)", "__aug (Photometric)", "__ovl (Occlusion)", "__comp (Compositing)", "Train (Final)", "Val (Untouched)", "Test (Untouched)", "Growth"],
    seg_rows
)